In [51]:
# TOTALS #1


import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML

ROOT_DIR = Path(r"C:\Users\djbar\popalgo")
years = [2022, 2023, 2024, 2025, 2026]

# Define the months used in the analysis and give them a simple sort order
month_labels = ["DEC", "JAN", "FEB"]
month_to_num = {"DEC": 1, "JAN": 2, "FEB": 3}

# Load each season file and stack everything into one dataframe
dfs = []
for y in years:
    p = ROOT_DIR / f"season_results_{y}.csv"
    if not p.exists():
        print(f"Missing file: {p}")
        continue
    temp = pd.read_csv(p)
    temp["year"] = y
    dfs.append(temp)

if not dfs:
    raise FileNotFoundError("No season_results_YYYY.csv files were found.")

full_df = pd.concat(dfs, ignore_index=True)

# Clean the month field. If MONTH is missing, build it from an available date column.
if "MONTH" in full_df.columns:
    full_df["MONTH"] = full_df["MONTH"].astype(str).str.upper().str.strip()
    full_df["MONTH"] = full_df["MONTH"].replace({
        "FEB/MAR": "FEB",
        "MARCH": "FEB",
        "MAR": "FEB"
    })
else:
    date_col = next((c for c in ["slate_date", "game_date", "date"] if c in full_df.columns), None)
    if date_col is None:
        raise KeyError("Could not find MONTH column or a usable date column.")
    dt = pd.to_datetime(full_df[date_col], errors="coerce")
    m = dt.dt.month
    full_df["MONTH"] = np.select(
        [m == 12, m == 1, m.isin([2, 3])],
        ["DEC", "JAN", "FEB"],
        default="OTHER"
    )

full_df = full_df[full_df["MONTH"].isin(month_labels)].copy()
full_df["MONTH_NUM"] = full_df["MONTH"].map(month_to_num)

# Clean the conference group columns and map them to readable labels
if not {"home_conf_grp", "away_conf_grp"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_conf_grp and away_conf_grp")

full_df["home_conf_grp"] = full_df["home_conf_grp"].astype(str).str.upper().str.strip()
full_df["away_conf_grp"] = full_df["away_conf_grp"].astype(str).str.upper().str.strip()

conf_label_map = {
    "P6": "Power 5",
    "HM2": "High Mid-Majors",
    "MM": "Mid-Majors",
    "LM": "Low Majors",
}

full_df["home_conf_label"] = full_df["home_conf_grp"].map(conf_label_map)
full_df["away_conf_label"] = full_df["away_conf_grp"].map(conf_label_map)

valid_conf_options = ["Power 5", "High Mid-Majors", "Mid-Majors", "Low Majors"]

full_df = full_df[
    full_df["home_conf_label"].isin(valid_conf_options) &
    full_df["away_conf_label"].isin(valid_conf_options)
].copy()

# Clean the pace group columns so only usable pace tags remain
if not {"home_pace_group", "away_pace_group"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_pace_group and away_pace_group")

full_df["home_pace_group"] = full_df["home_pace_group"].astype(str).str.upper().str.strip()
full_df["away_pace_group"] = full_df["away_pace_group"].astype(str).str.upper().str.strip()

bad_pace_vals = {"", "NAN", "NONE", "NULL", "OTHER"}
full_df = full_df[
    ~full_df["home_pace_group"].isin(bad_pace_vals) &
    ~full_df["away_pace_group"].isin(bad_pace_vals)
].copy()

pace_options = sorted(
    set(full_df["home_pace_group"].dropna().unique()).union(
        set(full_df["away_pace_group"].dropna().unique())
    )
)

# Make sure the totals fields needed for the chart are present
required_cols = ["totals_conf", "total_line_mvt", "total_play", "total_result"]
missing = [c for c in required_cols if c not in full_df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

full_df["totals_conf"] = pd.to_numeric(full_df["totals_conf"], errors="coerce")
full_df["total_line_mvt"] = pd.to_numeric(full_df["total_line_mvt"], errors="coerce")
full_df["total_result"] = pd.to_numeric(full_df["total_result"], errors="coerce")
full_df["total_play"] = full_df["total_play"].astype(str).str.upper().str.strip()

# Standardize pace values so the matchup buckets are consistent
def normalize_pace(val):
    v = str(val).strip().upper()
    if "FAST" in v:
        return "FAST"
    if "MID" in v:
        return "MID"
    if "SLOW" in v:
        return "SLOW"
    return np.nan

full_df["home_pace_clean"] = full_df["home_pace_group"].map(normalize_pace)
full_df["away_pace_clean"] = full_df["away_pace_group"].map(normalize_pace)

full_df = full_df[
    full_df["home_pace_clean"].notna() &
    full_df["away_pace_clean"].notna()
].copy()

# Create pace matchup groupings for filtering and x-axis views
def make_pace_matchup(a, b):
    s = {a, b}

    if a == "FAST" and b == "FAST":
        return "Fast vs Fast"
    if a == "MID" and b == "MID":
        return "Mid vs Mid"
    if a == "SLOW" and b == "SLOW":
        return "Slow vs Slow"
    if s == {"FAST", "SLOW"}:
        return "Fast vs Slow"
    if a in {"MID", "SLOW"} and b in {"MID", "SLOW"}:
        return "No Fast"
    if a in {"FAST", "MID"} and b in {"FAST", "MID"}:
        return "No Slow"
    return "Other"

full_df["pace_matchup"] = full_df.apply(
    lambda r: make_pace_matchup(r["home_pace_clean"], r["away_pace_clean"]),
    axis=1
)

pace_matchup_order = [
    "Fast vs Fast",
    "Mid vs Mid",
    "Slow vs Slow",
    "Fast vs Slow",
    "No Fast",
    "No Slow",
]

# Create conference matchup buckets for filtering and x-axis views
def make_conf_matchup(a, b):
    top2 = {"Power 5", "High Mid-Majors"}
    bottom2 = {"Mid-Majors", "Low Majors"}
    hmm_mm = {"High Mid-Majors", "Mid-Majors"}
    mm_lm = {"Mid-Majors", "Low Majors"}

    if a == "Power 5" and b == "Power 5":
        return "P5 vs P5"
    if a == "Low Majors" and b == "Low Majors":
        return "LM vs LM"
    if a in hmm_mm and b in hmm_mm:
        return "HMM & MM vs HMM & MM"
    if a in mm_lm and b in mm_lm:
        return "MM & LM vs MM & LM"
    if (a in top2 and b in bottom2) or (a in bottom2 and b in top2):
        return "Top 2 vs Bottom 2"
    if a in top2 and b in top2:
        return "Top 2 vs Top 2"
    if a in bottom2 and b in bottom2:
        return "Bottom 2 vs Bottom 2"
    if a != "Power 5" and b != "Power 5":
        return "No P5"
    return "Other"

full_df["conf_group_matchup"] = full_df.apply(
    lambda r: make_conf_matchup(r["home_conf_label"], r["away_conf_label"]),
    axis=1
)

conf_group_matchup_order = [
    "P5 vs P5",
    "HMM & MM vs HMM & MM",
    "MM & LM vs MM & LM",
    "LM vs LM",
    "Top 2 vs Bottom 2",
    "Top 2 vs Top 2",
    "Bottom 2 vs Bottom 2",
    "No P5",
]

# Small helper functions for the widget layout and summaries
def make_checkbox_group(options, selected=None, box_width="220px"):
    if selected is None:
        selected = options
    boxes = []
    for opt in options:
        cb = widgets.Checkbox(
            value=(opt in selected),
            description=opt,
            indent=False,
            layout=widgets.Layout(width=box_width)
        )
        boxes.append(cb)
    return boxes, widgets.VBox(boxes)

def get_checked_values(boxes):
    return [b.description for b in boxes if b.value]

def card(title, body, width="250px"):
    return widgets.VBox(
        [widgets.HTML(f"<b>{title}</b>"), body],
        layout=widgets.Layout(
            width=width,
            border="1px solid #d9d9d9",
            padding="10px",
            margin="4px"
        )
    )

# Turn any filtered subset into a quick performance summary
def summarize_subset(subset, label):
    wins = int((subset["total_result"] == 1).sum())
    losses = int((subset["total_result"] == -1).sum())
    n = len(subset)

    if n == 0:
        win_pct = np.nan
        net_units = np.nan
        avg_line_mvt = np.nan
    else:
        win_pct = wins / n
        net_units = wins - 1.1 * losses
        avg_line_mvt = subset["total_line_mvt"].mean()

    return {
        "X": label,
        "Wins": wins,
        "Losses": losses,
        "Win %": win_pct,
        "Net Units": net_units,
        "Avg Total Line Mvt": avg_line_mvt,
        "Sample": n
    }

# Build the grouped results based on the chosen x-axis view
def build_grouped_results(df, x_axis_mode, min_conf):
    rows = []

    if x_axis_mode == "Confidence Threshold":
        thresholds = list(range(0, 7))
        for t in thresholds:
            if min_conf > 0 and t < min_conf:
                subset = df.iloc[0:0].copy()
            else:
                subset = df if t == 0 else df[df["totals_conf"] >= t]
            rows.append(summarize_subset(subset, f"{t}" if t == 0 else f"{t}+"))

    elif x_axis_mode == "Year":
        order = [y for y in years if y in set(df["year"].dropna().unique())]
        for yr in order:
            rows.append(summarize_subset(df[df["year"] == yr], str(int(yr))))

    elif x_axis_mode == "Month":
        for mon in ["DEC", "JAN", "FEB"]:
            rows.append(summarize_subset(df[df["MONTH"] == mon], mon))

    elif x_axis_mode == "Pace Matchup":
        for grp in pace_matchup_order:
            rows.append(summarize_subset(df[df["pace_matchup"] == grp], grp))

    elif x_axis_mode == "Conference Group Matchup":
        for grp in conf_group_matchup_order:
            rows.append(summarize_subset(df[df["conf_group_matchup"] == grp], grp))

    elif x_axis_mode == "Bet Type":
        rows.append(summarize_subset(df[df["total_play"] == "OVER"], "OVER"))
        rows.append(summarize_subset(df[df["total_play"] == "UNDER"], "UNDER"))

    else:
        raise ValueError(f"Unsupported x_axis_mode: {x_axis_mode}")

    return pd.DataFrame(rows)

# Put the run button back in its default state after any input changes
def reset_run_button(change=None):
    submit_btn.button_style = "primary"
    submit_btn.description = "Run Report"
    submit_btn.icon = "bar-chart"

# Main function that applies the filters, builds the chart, and shows the table
def run_plot(
    x_axis_mode,
    start_year,
    end_year,
    start_month,
    end_month,
    conf_team_a,
    conf_team_b,
    pace_team_a,
    pace_team_b,
    bet_type,
    min_conf,
    show_line_movement
):
    if start_year > end_year:
        print("Start season must be <= end season.")
        return False

    if month_to_num[start_month] > month_to_num[end_month]:
        print("Start month must be <= end month.")
        return False

    if len(conf_team_a) == 0 or len(conf_team_b) == 0:
        print("Pick at least one conference group for Team A and Team B.")
        return False

    if len(pace_team_a) == 0 or len(pace_team_b) == 0:
        print("Pick at least one pace group for Team A and Team B.")
        return False

    df = full_df[
        (full_df["year"] >= start_year) &
        (full_df["year"] <= end_year) &
        (full_df["MONTH_NUM"] >= month_to_num[start_month]) &
        (full_df["MONTH_NUM"] <= month_to_num[end_month])
    ].copy()

    conf_mask = (
        (df["home_conf_label"].isin(conf_team_a) & df["away_conf_label"].isin(conf_team_b)) |
        (df["home_conf_label"].isin(conf_team_b) & df["away_conf_label"].isin(conf_team_a))
    )
    df = df[conf_mask].copy()

    pace_mask = (
        (df["home_pace_group"].isin(pace_team_a) & df["away_pace_group"].isin(pace_team_b)) |
        (df["home_pace_group"].isin(pace_team_b) & df["away_pace_group"].isin(pace_team_a))
    )
    df = df[pace_mask].copy()

    df = df[
        df["total_play"].notna() &
        df["total_result"].isin([1, -1]) &
        df["totals_conf"].notna()
    ].copy()

    if bet_type != "":
        df = df[df["total_play"] == bet_type].copy()

    if x_axis_mode != "Confidence Threshold" and min_conf > 0:
        df = df[df["totals_conf"] >= min_conf].copy()

    if df.empty or df["totals_conf"].notna().sum() == 0:
        print("No rows for this filter.")
        return False

    cutoff = df["totals_conf"].quantile(1)
    df = df[df["totals_conf"] <= cutoff].copy()

    if df.empty:
        print("No rows remain after outlier filter.")
        return False

    results_df = build_grouped_results(df, x_axis_mode, min_conf)
    results_df_nonzero = results_df[results_df["Sample"] > 0].copy()

    if results_df_nonzero.empty:
        print("No valid results after filtering.")
        return False

    if x_axis_mode == "Confidence Threshold" and min_conf > 0:
        totals_source_df = df[df["totals_conf"] >= min_conf].copy()
    else:
        totals_source_df = df.copy()

    totals_row = summarize_subset(totals_source_df, "TOTAL")

    best_row = results_df_nonzero.sort_values(
        ["Net Units", "Win %", "Sample"],
        ascending=[False, False, False]
    ).iloc[0]

    labels = results_df["X"].tolist()
    win_pct_vals = results_df["Win %"].to_numpy(dtype=float)
    sample_vals = results_df["Sample"].to_numpy(dtype=float)
    net_unit_vals = results_df["Net Units"].to_numpy(dtype=float)
    line_mvt_vals = results_df["Avg Total Line Mvt"].to_numpy(dtype=float)

    x = np.arange(len(labels))
    win_mask = ~np.isnan(win_pct_vals)
    mvt_mask = ~np.isnan(line_mvt_vals)

    fig_width = max(14, len(labels) * 2.0)
    fig_height = 10.4
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("#FCFCFC")
    ax2 = None

    bar_colors = ["#74C69D"] * len(labels)
    best_idx = results_df.index[results_df["X"] == best_row["X"]][0]
    bar_colors[best_idx] = "#FFD700"

    plot_vals = np.nan_to_num(win_pct_vals, nan=0.0)
    bars = ax.bar(
        x,
        plot_vals,
        edgecolor="#3A3A3A",
        linewidth=1.1,
        alpha=0.94,
        color=bar_colors,
        zorder=3,
        width=0.68
    )

    bars[best_idx].set_linewidth(2.4)
    bars[best_idx].set_edgecolor("#9C7B00")

    for i, wp in enumerate(win_pct_vals):
        if pd.isna(wp):
            bars[i].set_alpha(0.25)
            bars[i].set_edgecolor("#BDBDBD")

    ax.axhline(
        0.5238,
        linestyle="--",
        linewidth=2,
        color="black",
        alpha=0.75,
        zorder=2
    )

    win_trend_handle = None
    win_r = np.nan
    if win_mask.sum() >= 2:
        win_slope, win_intercept = np.polyfit(x[win_mask], win_pct_vals[win_mask], 1)
        win_trend = win_intercept + win_slope * x[win_mask]
        win_r = np.corrcoef(x[win_mask], win_pct_vals[win_mask])[0, 1]
        win_trend_handle, = ax.plot(
            x[win_mask],
            win_trend,
            linestyle=":",
            linewidth=3.2,
            color="orange",
            alpha=0.90,
            zorder=4
        )

    line_handle = None
    mvt_r = np.nan
    if show_line_movement:
        ax2 = ax.twinx()
        if mvt_mask.sum() >= 1:
            line_handle, = ax2.plot(
                x[mvt_mask],
                line_mvt_vals[mvt_mask],
                marker="o",
                linewidth=2.5,
                linestyle="--",
                color="royalblue",
                alpha=0.35,
                zorder=1
            )
        if mvt_mask.sum() >= 2:
            mvt_r = np.corrcoef(x[mvt_mask], line_mvt_vals[mvt_mask])[0, 1]

    valid_wp = win_pct_vals[~np.isnan(win_pct_vals)]
    if len(valid_wp) > 0:
        wp_min = min(valid_wp.min(), 0.5238)
        wp_max = max(valid_wp.max(), 0.5238)
        wp_pad = max(0.01, (wp_max - wp_min) * 0.25)
        y1_low = max(0.0, wp_min - wp_pad)
        y1_high = min(1.0, wp_max + wp_pad)

        if (y1_high - y1_low) < 0.08:
            mid = (y1_low + y1_high) / 2
            y1_low = max(0.0, mid - 0.04)
            y1_high = min(1.0, mid + 0.04)

        ax.set_ylim(y1_low, y1_high)

        tick_step = 0.01 if (y1_high - y1_low) <= 0.12 else 0.02
        yticks = np.arange(round(y1_low, 2), round(y1_high + tick_step, 2), tick_step)
        ax.set_yticks(yticks)
    else:
        ax.set_ylim(0.40, 0.55)

    if show_line_movement and ax2 is not None:
        valid_mvt = line_mvt_vals[~np.isnan(line_mvt_vals)]
        if len(valid_mvt) > 0:
            mvt_min = valid_mvt.min()
            mvt_max = valid_mvt.max()
            pad = max(0.25, abs(mvt_min) * 0.20) if mvt_min == mvt_max else max(0.25, (mvt_max - mvt_min) * 0.30)
            ax2.set_ylim(mvt_min - pad, mvt_max + pad)

    rotation = 0 if x_axis_mode in ["Confidence Threshold", "Year", "Month", "Bet Type"] else 25

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        fontsize=12,
        fontweight="bold",
        rotation=rotation,
        ha="right" if rotation else "center"
    )

    ax.tick_params(axis="y", labelsize=13)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")

    if show_line_movement and ax2 is not None:
        ax2.tick_params(axis="y", labelsize=13)
        for lbl in ax2.get_yticklabels():
            lbl.set_fontweight("bold")

    ax.set_ylabel("Winning %", fontsize=17, fontweight="bold", labelpad=12)
    if show_line_movement and ax2 is not None:
        ax2.set_ylabel("Avg Total Line Movement", fontsize=16, fontweight="bold", labelpad=12)

    ax.set_xlabel(x_axis_mode, fontsize=17, fontweight="bold", labelpad=12)

    bet_type_text = "ALL" if bet_type == "" else bet_type
    min_conf_text = "ALL (0)" if min_conf == 0 else f"{min_conf}+"

    ax.set_title(
        f"Total Winning % by {x_axis_mode}\n"
        f"Seasons: {start_year}–{end_year} | Months: {start_month} to {end_month} | "
        f"Bet Type: {bet_type_text} | Min Included Confidence: {min_conf_text}",
        fontsize=15,
        fontweight="bold",
        pad=26
    )

    ax.grid(axis="y", linestyle=":", linewidth=0.9, alpha=0.70)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if show_line_movement and ax2 is not None:
        ax2.spines["top"].set_visible(False)
    ax.spines["left"].set_color("#999999")
    ax.spines["bottom"].set_color("#999999")
    ax.margins(x=0.05)

    annotation_lines = [
        f"Win % r = {win_r:.3f}" if pd.notna(win_r) and win_mask.sum() >= 2 else "Win % r = N/A"
    ]
    if show_line_movement:
        annotation_lines.append(
            f"Line Mvt r = {mvt_r:.3f}" if pd.notna(mvt_r) and mvt_mask.sum() >= 2 else "Line Mvt r = N/A"
        )

    ax.text(
        0.02,
        0.98,
        "\n".join(annotation_lines),
        transform=ax.transAxes,
        fontsize=12.5,
        fontweight="bold",
        va="top",
        ha="left",
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="#D7D7D7",
            alpha=0.96
        )
    )

    summary_text = (
        f"TOTAL SAMPLE: {int(totals_row['Sample'])}   |   "
        f"RECORD: {int(totals_row['Wins'])}-{int(totals_row['Losses'])}   |   "
        f"WIN %: {totals_row['Win %']:.1%}   |   "
        f"NET UNITS: {totals_row['Net Units']:.1f}u"
    )
    if pd.notna(totals_row["Avg Total Line Mvt"]):
        summary_text += f"   |   AVG MVT: {totals_row['Avg Total Line Mvt']:.2f}"

    fig.text(
        0.5,
        0.945,
        summary_text,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        bbox=dict(facecolor="#F6F6F6", edgecolor="#D0D0D0", boxstyle="round,pad=0.40")
    )

    if int(totals_row["Sample"]) < 100:
        fig.text(
            0.5,
            0.912,
            f"Warning: small sample after filters (n={int(totals_row['Sample'])})",
            ha="center",
            va="center",
            fontsize=10.5,
            fontweight="bold",
            color="#B45309"
        )

    y1_low, y1_high = ax.get_ylim()
    y_range = y1_high - y1_low
    top_offset = y_range * 0.015
    bottom_1 = y1_low + y_range * 0.05
    bottom_2 = y1_low + y_range * 0.10
    bottom_3 = y1_low + y_range * 0.15

    for i, (bar, wp, n, nu, lm) in enumerate(zip(bars, win_pct_vals, sample_vals, net_unit_vals, line_mvt_vals)):
        x_text = bar.get_x() + bar.get_width() / 2

        if pd.notna(wp):
            ax.text(
                x_text,
                wp + top_offset,
                f"{wp:.1%}",
                ha="center",
                va="bottom",
                fontsize=11,
                fontweight="bold"
            )
        else:
            ax.text(
                x_text,
                top_offset + y1_low + y_range * 0.005,
                "—",
                ha="center",
                va="bottom",
                fontsize=12,
                fontweight="bold",
                color="#888888"
            )

        if show_line_movement and pd.notna(lm):
            ax.text(
                x_text,
                bottom_1,
                f"mvt={lm:.2f}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )

        ax.text(
            x_text,
            bottom_2 if show_line_movement else bottom_1,
            f"n={int(n)}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

        if pd.notna(nu):
            label_text = f"{nu:.1f}u"
            if i == best_idx:
                label_text = f"★ {nu:.1f}u"
            ax.text(
                x_text,
                bottom_3 if show_line_movement else bottom_2,
                label_text,
                ha="center",
                va="bottom",
                fontsize=11,
                fontweight="bold"
            )

    handles = []
    labels_legend = []

    if win_trend_handle is not None:
        handles.append(win_trend_handle)
        labels_legend.append("Win % Trend")

    if show_line_movement and line_handle is not None:
        handles.append(line_handle)
        labels_legend.append("Avg Total Line Mvt")

    if handles:
        ax.legend(handles, labels_legend, loc="upper right", fontsize=12, frameon=True)

    footer_text = (
        f"BEST NET UNITS: {best_row['X']}   |   "
        f"Record: {int(best_row['Wins'])}-{int(best_row['Losses'])}   |   "
        f"Win %: {best_row['Win %']:.1%}   |   "
        f"Net Units: {best_row['Net Units']:.1f}u   |   "
        f"Sample: {int(best_row['Sample'])}"
    )

    fig.text(
        0.5,
        0.028,
        footer_text,
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        bbox=dict(
            facecolor="#EEF6FF",
            edgecolor="#94BCE5",
            boxstyle="round,pad=0.70",
            linewidth=1.8
        )
    )

    plt.tight_layout(rect=[0.03, 0.10, 0.97, 0.90])
    plt.show()

    # Build the summary table shown underneath the chart
    show_df = results_df.copy()
    show_df = pd.concat([show_df, pd.DataFrame([totals_row])], ignore_index=True)

    show_df["Win %"] = show_df["Win %"].map(lambda v: "" if pd.isna(v) else f"{v:.1%}")
    show_df["Net Units"] = show_df["Net Units"].map(lambda v: "" if pd.isna(v) else f"{v:.1f}")
    show_df["Avg Total Line Mvt"] = show_df["Avg Total Line Mvt"].map(lambda v: "" if pd.isna(v) else f"{v:.2f}")

    show_df = show_df[
        ["X", "Wins", "Losses", "Win %", "Net Units", "Avg Total Line Mvt", "Sample"]
    ].copy()

    show_df.columns = [
        "Group",
        "Wins",
        "Losses",
        "Win %",
        "Net Units",
        "Avg Line Mvt",
        "Sample"
    ]

    best_group = best_row["X"]

    def zebra_rows(df):
        styles = pd.DataFrame("", index=df.index, columns=df.columns)
        for i in range(len(df)):
            if df.iloc[i]["Group"] != "TOTAL":
                bg = "#FAFAFA" if i % 2 == 0 else "#FFFFFF"
                styles.iloc[i, :] = f"background-color:{bg};"
        return styles

    def color_units(val):
        try:
            v = float(val)
        except:
            return ""

        if v > 0:
            return "color:#0B7D3E; font-weight:700;"
        if v < 0:
            return "color:#B42318;"
        return ""

    def highlight_rows(row):
        styles = [""] * len(row)

        if row["Group"] == best_group:
            styles = ["background-color:#EAF4FF; font-weight:700;"] * len(row)

        if row["Group"] == "TOTAL":
            styles = ["background-color:#E8F5E9; font-weight:800; border-top:4px solid #333;"] * len(row)

        return styles

    styled_table = (
        show_df.style
        .hide(axis="index")
        .apply(zebra_rows, axis=None)
        .apply(highlight_rows, axis=1)
        .map(color_units, subset=["Net Units"])
        .set_properties(**{
            "text-align": "center",
            "font-size": "17px",
            "padding": "14px 18px",
            "border": "none"
        })
        .set_table_styles([
            {
                "selector": "table",
                "props": [
                    ("margin-left", "auto"),
                    ("margin-right", "auto"),
                    ("margin-top", "50px"),
                    ("margin-bottom", "20px"),
                    ("min-width", "1150px"),
                    ("border-collapse", "separate"),
                    ("border-spacing", "0"),
                    ("font-family", "Arial, Helvetica, sans-serif"),
                    ("background-color", "white"),
                    ("border", "1px solid #DADADA"),
                    ("border-radius", "10px"),
                    ("overflow", "hidden"),
                    ("box-shadow", "0 4px 14px rgba(0,0,0,0.08)")
                ]
            },
            {
                "selector": "thead th",
                "props": [
                    ("background-color", "#1F1F1F"),
                    ("color", "white"),
                    ("font-size", "17px"),
                    ("font-weight", "700"),
                    ("text-align", "center"),
                    ("padding", "14px 18px"),
                    ("border-bottom", "2px solid #111")
                ]
            },
            {
                "selector": "tbody td",
                "props": [
                    ("text-align", "center"),
                    ("padding", "14px 18px"),
                    ("font-size", "17px")
                ]
            },
            {
                "selector": "tbody td.col0",
                "props": [
                    ("font-weight", "700"),
                    ("text-align", "left"),
                    ("padding-left", "20px")
                ]
            }
        ])
    )

    display(HTML("<div style='height:40px'></div>"))
    display(styled_table)

    return True

# Set up the dashboard widgets
style = {"description_width": "145px"}

wide_dd = widgets.Layout(width="320px")
med_dd = widgets.Layout(width="260px")
month_dd = widgets.Layout(width="220px")

x_axis_dd = widgets.Dropdown(
    options=[
        "Confidence Threshold",
        "Year",
        "Month",
        "Pace Matchup",
        "Conference Group Matchup",
        "Bet Type",
    ],
    value="Confidence Threshold",
    description="X Axis",
    style=style,
    layout=wide_dd
)

bet_type_dd = widgets.Dropdown(
    options=[("", ""), ("OVER", "OVER"), ("UNDER", "UNDER")],
    value="",
    description="Bet Type",
    style=style,
    layout=med_dd
)

conf_min_dd = widgets.Dropdown(
    options=[
        ("ALL (0)", 0),
        ("1", 1),
        ("2", 2),
        ("3", 3),
        ("4", 4),
        ("5", 5),
        ("6", 6),
    ],
    value=0,
    description="Min Included Conf",
    style=style,
    layout=wide_dd
)

start_year_dd = widgets.Dropdown(
    options=years,
    value=years[0],
    description="Start Season",
    style=style,
    layout=med_dd
)

end_year_dd = widgets.Dropdown(
    options=years,
    value=years[-1],
    description="End Season",
    style=style,
    layout=med_dd
)

start_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[0],
    description="Start Month",
    style=style,
    layout=month_dd
)

end_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[-1],
    description="End Month",
    style=style,
    layout=month_dd
)

show_line_mvt_cb = widgets.Checkbox(
    value=True,
    description="Show Line Movement",
    indent=False,
    layout=widgets.Layout(width="220px")
)

submit_btn = widgets.Button(
    description="Run Report",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="170px", height="40px")
)

conf_a_boxes, conf_a_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
conf_b_boxes, conf_b_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
pace_a_boxes, pace_a_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)
pace_b_boxes, pace_b_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)

output = widgets.Output()

# Reset the run button whenever an input changes
all_reset_widgets = [
    x_axis_dd,
    bet_type_dd,
    conf_min_dd,
    start_year_dd,
    end_year_dd,
    start_month_dd,
    end_month_dd,
    show_line_mvt_cb,
]

for w in all_reset_widgets:
    w.observe(reset_run_button, names="value")

for box in conf_a_boxes + conf_b_boxes + pace_a_boxes + pace_b_boxes:
    box.observe(reset_run_button, names="value")

# Run the report when the button is clicked
def on_submit_clicked(_):
    submit_btn.button_style = ""
    submit_btn.description = "Running..."
    submit_btn.icon = "spinner"

    with output:
        output.clear_output(wait=True)

        try:
            ok = run_plot(
                x_axis_mode=x_axis_dd.value,
                start_year=start_year_dd.value,
                end_year=end_year_dd.value,
                start_month=start_month_dd.value,
                end_month=end_month_dd.value,
                conf_team_a=get_checked_values(conf_a_boxes),
                conf_team_b=get_checked_values(conf_b_boxes),
                pace_team_a=get_checked_values(pace_a_boxes),
                pace_team_b=get_checked_values(pace_b_boxes),
                bet_type=bet_type_dd.value,
                min_conf=conf_min_dd.value,
                show_line_movement=show_line_mvt_cb.value
            )

            if ok:
                submit_btn.button_style = "success"
                submit_btn.description = "Report Ready"
                submit_btn.icon = "check"
            else:
                submit_btn.button_style = "warning"
                submit_btn.description = "No Results"
                submit_btn.icon = "info-circle"

        except Exception:
            import traceback
            print("RUN REPORT FAILED\n")
            traceback.print_exc()

            submit_btn.button_style = "danger"
            submit_btn.description = "Error"
            submit_btn.icon = "warning"

submit_btn.on_click(on_submit_clicked)

# Arrange the dashboard layout
controls_row_1 = widgets.HBox(
    [x_axis_dd, bet_type_dd, conf_min_dd, start_year_dd, end_year_dd, start_month_dd, end_month_dd],
    layout=widgets.Layout(
        gap="12px",
        margin="0 0 10px 0",
        flex_flow="row wrap",
        justify_content="flex-start",
        align_items="center"
    )
)

controls_row_2 = widgets.HBox(
    [show_line_mvt_cb],
    layout=widgets.Layout(
        gap="12px",
        margin="0 0 10px 0",
        align_items="center"
    )
)

controls_row_3 = widgets.HBox(
    [submit_btn],
    layout=widgets.Layout(
        justify_content="center",
        margin="6px 0 12px 0"
    )
)

selection_row = widgets.HBox(
    [
        card("Conference Group — Team A", conf_a_ui, "260px"),
        card("Conference Group — Team B", conf_b_ui, "260px"),
        card("Pace Group — Team A", pace_a_ui, "220px"),
        card("Pace Group — Team B", pace_b_ui, "220px"),
    ],
    layout=widgets.Layout(
        justify_content="center",
        align_items="flex-start",
        gap="8px",
        flex_flow="row wrap"
    )
)

ui = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 10px 0;'>Totals Confidence Filter Dashboard</h3>"),
    controls_row_1,
    controls_row_2,
    controls_row_3,
    selection_row,
    output
])

display(ui)

In [52]:
#2 - SPREADS #1

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML

ROOT_DIR = Path(r"C:\Users\djbar\popalgo")
years = [2022, 2023, 2024, 2025, 2026]

# Months we want to keep, plus a simple numeric order for filtering
month_labels = ["DEC", "JAN", "FEB"]
month_to_num = {"DEC": 1, "JAN": 2, "FEB": 3}

# Load each season file and stack everything into one dataframe
dfs = []
for y in years:
    p = ROOT_DIR / f"season_results_{y}.csv"
    if not p.exists():
        print(f"Missing file: {p}")
        continue
    temp = pd.read_csv(p)
    temp["year"] = y
    dfs.append(temp)

if not dfs:
    raise FileNotFoundError("No season_results_YYYY.csv files were found.")

full_df = pd.concat(dfs, ignore_index=True)

# Standardize the month labels
# If MONTH is missing, try building it from a date column instead
if "MONTH" in full_df.columns:
    full_df["MONTH"] = full_df["MONTH"].astype(str).str.upper().str.strip()
    full_df["MONTH"] = full_df["MONTH"].replace({
        "FEB/MAR": "FEB",
        "MARCH": "FEB",
        "MAR": "FEB"
    })
else:
    date_col = next((c for c in ["slate_date", "game_date", "date"] if c in full_df.columns), None)
    if date_col is None:
        raise KeyError("Could not find MONTH column or a usable date column.")

    dt = pd.to_datetime(full_df[date_col], errors="coerce")
    m = dt.dt.month

    full_df["MONTH"] = np.select(
        [m == 12, m == 1, m.isin([2, 3])],
        ["DEC", "JAN", "FEB"],
        default="OTHER"
    )

# Keep only the core months you care about
full_df = full_df[full_df["MONTH"].isin(month_labels)].copy()
full_df["MONTH_NUM"] = full_df["MONTH"].map(month_to_num)

# Clean up conference group fields for both teams
if not {"home_conf_grp", "away_conf_grp"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_conf_grp and away_conf_grp")

full_df["home_conf_grp"] = full_df["home_conf_grp"].astype(str).str.upper().str.strip()
full_df["away_conf_grp"] = full_df["away_conf_grp"].astype(str).str.upper().str.strip()

# Convert raw conference codes into readable labels
conf_label_map = {
    "P6": "Power 5",
    "HM2": "High Mid-Majors",
    "MM": "Mid-Majors",
    "LM": "Low Majors",
}

full_df["home_conf_label"] = full_df["home_conf_grp"].map(conf_label_map)
full_df["away_conf_label"] = full_df["away_conf_grp"].map(conf_label_map)

valid_conf_options = ["Power 5", "High Mid-Majors", "Mid-Majors", "Low Majors"]

# Remove rows with conference labels outside the groups we use
full_df = full_df[
    full_df["home_conf_label"].isin(valid_conf_options) &
    full_df["away_conf_label"].isin(valid_conf_options)
].copy()

# Clean up pace group fields
if not {"home_pace_group", "away_pace_group"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_pace_group and away_pace_group")

full_df["home_pace_group"] = full_df["home_pace_group"].astype(str).str.upper().str.strip()
full_df["away_pace_group"] = full_df["away_pace_group"].astype(str).str.upper().str.strip()

bad_pace_vals = {"", "NAN", "NONE", "NULL", "OTHER"}
full_df = full_df[
    ~full_df["home_pace_group"].isin(bad_pace_vals) &
    ~full_df["away_pace_group"].isin(bad_pace_vals)
].copy()

pace_options = sorted(
    set(full_df["home_pace_group"].dropna().unique()).union(
        set(full_df["away_pace_group"].dropna().unique())
    )
)

# Make sure the spread columns we need are present
required_cols = ["spread_conf", "spread_line_mvt", "spread_play", "spread_result"]
missing = [c for c in required_cols if c not in full_df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

full_df["spread_conf"] = pd.to_numeric(full_df["spread_conf"], errors="coerce")
full_df["spread_line_mvt"] = pd.to_numeric(full_df["spread_line_mvt"], errors="coerce")
full_df["spread_result"] = pd.to_numeric(full_df["spread_result"], errors="coerce")
full_df["spread_play"] = full_df["spread_play"].astype(str).str.upper().str.strip()

# Normalize pace labels into the three buckets the dashboard uses
def normalize_pace(val):
    v = str(val).strip().upper()
    if "FAST" in v:
        return "FAST"
    if "MID" in v:
        return "MID"
    if "SLOW" in v:
        return "SLOW"
    return np.nan

full_df["home_pace_clean"] = full_df["home_pace_group"].map(normalize_pace)
full_df["away_pace_clean"] = full_df["away_pace_group"].map(normalize_pace)

full_df = full_df[
    full_df["home_pace_clean"].notna() &
    full_df["away_pace_clean"].notna()
].copy()

# Build pace matchup buckets for grouping later
def make_pace_matchup(a, b):
    s = {a, b}

    if a == "FAST" and b == "FAST":
        return "Fast vs Fast"
    if a == "MID" and b == "MID":
        return "Mid vs Mid"
    if a == "SLOW" and b == "SLOW":
        return "Slow vs Slow"
    if s == {"FAST", "SLOW"}:
        return "Fast vs Slow"
    if a in {"MID", "SLOW"} and b in {"MID", "SLOW"}:
        return "No Fast"
    if a in {"FAST", "MID"} and b in {"FAST", "MID"}:
        return "No Slow"
    return "Other"

full_df["pace_matchup"] = full_df.apply(
    lambda r: make_pace_matchup(r["home_pace_clean"], r["away_pace_clean"]),
    axis=1
)

pace_matchup_order = [
    "Fast vs Fast",
    "Mid vs Mid",
    "Slow vs Slow",
    "Fast vs Slow",
    "No Fast",
    "No Slow",
]

# Build conference matchup buckets for grouping later
def make_conf_matchup(a, b):
    top2 = {"Power 5", "High Mid-Majors"}
    bottom2 = {"Mid-Majors", "Low Majors"}
    hmm_mm = {"High Mid-Majors", "Mid-Majors"}
    mm_lm = {"Mid-Majors", "Low Majors"}

    if a == "Power 5" and b == "Power 5":
        return "P5 vs P5"
    if a == "Low Majors" and b == "Low Majors":
        return "LM vs LM"
    if a in hmm_mm and b in hmm_mm:
        return "HMM & MM vs HMM & MM"
    if a in mm_lm and b in mm_lm:
        return "MM & LM vs MM & LM"
    if (a in top2 and b in bottom2) or (a in bottom2 and b in top2):
        return "Top 2 vs Bottom 2"
    if a in top2 and b in top2:
        return "Top 2 vs Top 2"
    if a in bottom2 and b in bottom2:
        return "Bottom 2 vs Bottom 2"
    if a != "Power 5" and b != "Power 5":
        return "No P5"
    return "Other"

full_df["conf_group_matchup"] = full_df.apply(
    lambda r: make_conf_matchup(r["home_conf_label"], r["away_conf_label"]),
    axis=1
)

conf_group_matchup_order = [
    "P5 vs P5",
    "HMM & MM vs HMM & MM",
    "MM & LM vs MM & LM",
    "LM vs LM",
    "Top 2 vs Bottom 2",
    "Top 2 vs Top 2",
    "Bottom 2 vs Bottom 2",
    "No P5",
]

# Build a vertical list of checkboxes
def make_checkbox_group(options, selected=None, box_width="220px"):
    if selected is None:
        selected = options

    boxes = []
    for opt in options:
        cb = widgets.Checkbox(
            value=(opt in selected),
            description=opt,
            indent=False,
            layout=widgets.Layout(width=box_width)
        )
        boxes.append(cb)

    return boxes, widgets.VBox(boxes)

# Pull the selected values out of a checkbox list
def get_checked_values(boxes):
    return [b.description for b in boxes if b.value]

# Small helper to put a border around widget sections
def card(title, body, width="250px"):
    return widgets.VBox(
        [widgets.HTML(f"<b>{title}</b>"), body],
        layout=widgets.Layout(
            width=width,
            border="1px solid #d9d9d9",
            padding="10px",
            margin="4px"
        )
    )

# Turn a filtered subset into summary stats
def summarize_subset(subset, label):
    wins = int((subset["spread_result"] == 1).sum())
    losses = int((subset["spread_result"] == -1).sum())
    n = len(subset)

    if n == 0:
        win_pct = np.nan
        net_units = np.nan
        avg_line_mvt = np.nan
    else:
        win_pct = wins / n
        net_units = wins - 1.1 * losses
        avg_line_mvt = subset["spread_line_mvt"].mean()

    return {
        "X": label,
        "Wins": wins,
        "Losses": losses,
        "Win %": win_pct,
        "Net Units": net_units,
        "Avg Spread Line Mvt": avg_line_mvt,
        "Sample": n
    }

# Build the grouped results table based on the selected x-axis mode
def build_grouped_results(df, x_axis_mode, min_conf):
    rows = []

    if x_axis_mode == "Confidence Threshold":
        thresholds = list(range(0, 8))
        for t in thresholds:
            if min_conf > 0 and t < min_conf:
                subset = df.iloc[0:0].copy()
            else:
                subset = df if t == 0 else df[df["spread_conf"] >= t]
            rows.append(summarize_subset(subset, f"{t}" if t == 0 else f"{t}+"))

    elif x_axis_mode == "Year":
        order = [y for y in years if y in set(df["year"].dropna().unique())]
        for yr in order:
            rows.append(summarize_subset(df[df["year"] == yr], str(int(yr))))

    elif x_axis_mode == "Month":
        for mon in ["DEC", "JAN", "FEB"]:
            rows.append(summarize_subset(df[df["MONTH"] == mon], mon))

    elif x_axis_mode == "Pace Matchup":
        for grp in pace_matchup_order:
            rows.append(summarize_subset(df[df["pace_matchup"] == grp], grp))

    elif x_axis_mode == "Conference Group Matchup":
        for grp in conf_group_matchup_order:
            rows.append(summarize_subset(df[df["conf_group_matchup"] == grp], grp))

    elif x_axis_mode == "Bet Type":
        rows.append(summarize_subset(df, "ALL"))
        rows.append(summarize_subset(df[df["spread_play"] == "HOME"], "HOME"))
        rows.append(summarize_subset(df[df["spread_play"] == "AWAY"], "AWAY"))

    else:
        raise ValueError(f"Unsupported x_axis_mode: {x_axis_mode}")

    return pd.DataFrame(rows)

# Reset the button style whenever a filter changes
def reset_run_button(change=None):
    submit_btn.button_style = "primary"
    submit_btn.description = "Run Report"
    submit_btn.icon = "bar-chart"

# Placeholder hook kept in case you expand spread-side options later
def update_spread_side_state(change=None):
    is_all = (spread_side_dd.value == "ALL")
    reset_run_button()

# Main plotting function for the dashboard
def run_plot(
    x_axis_mode,
    start_year,
    end_year,
    start_month,
    end_month,
    conf_team_a,
    conf_team_b,
    pace_team_a,
    pace_team_b,
    bet_type,
    min_conf,
    show_line_movement
):
    if start_year > end_year:
        print("Start season must be <= end season.")
        return False

    if month_to_num[start_month] > month_to_num[end_month]:
        print("Start month must be <= end month.")
        return False

    if len(conf_team_a) == 0 or len(conf_team_b) == 0:
        print("Pick at least one conference group for Team A and Team B.")
        return False

    if len(pace_team_a) == 0 or len(pace_team_b) == 0:
        print("Pick at least one pace group for Team A and Team B.")
        return False

    df = full_df[
        (full_df["year"] >= start_year) &
        (full_df["year"] <= end_year) &
        (full_df["MONTH_NUM"] >= month_to_num[start_month]) &
        (full_df["MONTH_NUM"] <= month_to_num[end_month])
    ].copy()

    conf_mask = (
        (df["home_conf_label"].isin(conf_team_a) & df["away_conf_label"].isin(conf_team_b)) |
        (df["home_conf_label"].isin(conf_team_b) & df["away_conf_label"].isin(conf_team_a))
    )
    df = df[conf_mask].copy()

    pace_mask = (
        (df["home_pace_group"].isin(pace_team_a) & df["away_pace_group"].isin(pace_team_b)) |
        (df["home_pace_group"].isin(pace_team_b) & df["away_pace_group"].isin(pace_team_a))
    )
    df = df[pace_mask].copy()

    df = df[
        df["spread_play"].notna() &
        df["spread_result"].isin([1, -1]) &
        df["spread_conf"].notna()
    ].copy()

    if bet_type != "ALL":
        df = df[df["spread_play"] == bet_type].copy()

    if x_axis_mode != "Confidence Threshold" and min_conf > 0:
        df = df[df["spread_conf"] >= min_conf].copy()

    if df.empty or df["spread_conf"].notna().sum() == 0:
        print("No rows for this filter.")
        return False

    # Keep the original outlier trim from your prior version
    cutoff = df["spread_conf"].quantile(0.95)
    df = df[df["spread_conf"] <= cutoff].copy()

    if df.empty:
        print("No rows remain after outlier filter.")
        return False

    results_df = build_grouped_results(df, x_axis_mode, min_conf)
    results_df_nonzero = results_df[results_df["Sample"] > 0].copy()

    if results_df_nonzero.empty:
        print("No valid results after filtering.")
        return False

    if x_axis_mode == "Confidence Threshold" and min_conf > 0:
        totals_source_df = df[df["spread_conf"] >= min_conf].copy()
    else:
        totals_source_df = df.copy()

    totals_row = summarize_subset(totals_source_df, "TOTAL")

    best_row = results_df_nonzero.sort_values(
        ["Net Units", "Win %", "Sample"],
        ascending=[False, False, False]
    ).iloc[0]

    labels = results_df["X"].tolist()
    win_pct_vals = results_df["Win %"].to_numpy(dtype=float)
    sample_vals = results_df["Sample"].to_numpy(dtype=float)
    net_unit_vals = results_df["Net Units"].to_numpy(dtype=float)
    line_mvt_vals = results_df["Avg Spread Line Mvt"].to_numpy(dtype=float)

    x = np.arange(len(labels))
    win_mask = ~np.isnan(win_pct_vals)
    mvt_mask = ~np.isnan(line_mvt_vals)

    fig_width = max(14, len(labels) * 2.0)
    fig_height = 10.4
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("#FCFCFC")
    ax2 = None

    bar_colors = ["lightblue"] * len(labels)
    best_idx = results_df.index[results_df["X"] == best_row["X"]][0]
    bar_colors[best_idx] = "#FFD700"

    plot_vals = np.nan_to_num(win_pct_vals, nan=0.0)
    bars = ax.bar(
        x,
        plot_vals,
        edgecolor="#3A3A3A",
        linewidth=1.1,
        alpha=0.94,
        color=bar_colors,
        zorder=3,
        width=0.68
    )

    bars[best_idx].set_linewidth(2.4)
    bars[best_idx].set_edgecolor("#9C7B00")

    for i, wp in enumerate(win_pct_vals):
        if pd.isna(wp):
            bars[i].set_alpha(0.25)
            bars[i].set_edgecolor("#BDBDBD")

    ax.axhline(
        0.5238,
        linestyle="--",
        linewidth=2,
        color="black",
        alpha=0.75,
        zorder=2
    )

    win_trend_handle = None
    win_r = np.nan
    if win_mask.sum() >= 2:
        win_slope, win_intercept = np.polyfit(x[win_mask], win_pct_vals[win_mask], 1)
        win_trend = win_intercept + win_slope * x[win_mask]
        win_r = np.corrcoef(x[win_mask], win_pct_vals[win_mask])[0, 1]
        win_trend_handle, = ax.plot(
            x[win_mask],
            win_trend,
            linestyle=":",
            linewidth=3.2,
            color="orange",
            alpha=0.90,
            zorder=4
        )

    line_handle = None
    mvt_r = np.nan
    if show_line_movement:
        ax2 = ax.twinx()
        if mvt_mask.sum() >= 1:
            line_handle, = ax2.plot(
                x[mvt_mask],
                line_mvt_vals[mvt_mask],
                marker="o",
                linewidth=2.5,
                linestyle="--",
                color="royalblue",
                alpha=0.35,
                zorder=1
            )
        if mvt_mask.sum() >= 2:
            mvt_r = np.corrcoef(x[mvt_mask], line_mvt_vals[mvt_mask])[0, 1]

    valid_wp = win_pct_vals[~np.isnan(win_pct_vals)]
    if len(valid_wp) > 0:
        wp_min = min(valid_wp.min(), 0.5238)
        wp_max = max(valid_wp.max(), 0.5238)
        wp_pad = max(0.01, (wp_max - wp_min) * 0.25)
        y1_low = max(0.0, wp_min - wp_pad)
        y1_high = min(1.0, wp_max + wp_pad)

        if (y1_high - y1_low) < 0.08:
            mid = (y1_low + y1_high) / 2
            y1_low = max(0.0, mid - 0.04)
            y1_high = min(1.0, mid + 0.04)

        ax.set_ylim(y1_low, y1_high)

        tick_step = 0.01 if (y1_high - y1_low) <= 0.12 else 0.02
        yticks = np.arange(round(y1_low, 2), round(y1_high + tick_step, 2), tick_step)
        ax.set_yticks(yticks)
    else:
        ax.set_ylim(0.40, 0.55)

    if show_line_movement and ax2 is not None:
        valid_mvt = line_mvt_vals[~np.isnan(line_mvt_vals)]
        if len(valid_mvt) > 0:
            mvt_min = valid_mvt.min()
            mvt_max = valid_mvt.max()
            pad = max(0.25, abs(mvt_min) * 0.20) if mvt_min == mvt_max else max(0.25, (mvt_max - mvt_min) * 0.30)
            ax2.set_ylim(mvt_min - pad, mvt_max + pad)

    rotation = 0 if x_axis_mode in ["Confidence Threshold", "Year", "Month", "Bet Type"] else 25

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        fontsize=12,
        fontweight="bold",
        rotation=rotation,
        ha="right" if rotation else "center"
    )

    ax.tick_params(axis="y", labelsize=13)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")

    if show_line_movement and ax2 is not None:
        ax2.tick_params(axis="y", labelsize=13)
        for lbl in ax2.get_yticklabels():
            lbl.set_fontweight("bold")

    ax.set_ylabel("Winning %", fontsize=17, fontweight="bold", labelpad=12)
    if show_line_movement and ax2 is not None:
        ax2.set_ylabel("Avg Spread Line Movement", fontsize=16, fontweight="bold", labelpad=12)

    ax.set_xlabel(x_axis_mode, fontsize=17, fontweight="bold", labelpad=12)

    min_conf_text = "ALL (0)" if min_conf == 0 else f"{min_conf}+"

    ax.set_title(
        f"Spread Winning % by {x_axis_mode}\n"
        f"Seasons: {start_year}–{end_year} | Months: {start_month} to {end_month} | "
        f"Bet Type: {bet_type} | Min Included Confidence: {min_conf_text}",
        fontsize=15,
        fontweight="bold",
        pad=26
    )

    ax.grid(axis="y", linestyle=":", linewidth=0.9, alpha=0.70)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if show_line_movement and ax2 is not None:
        ax2.spines["top"].set_visible(False)
    ax.spines["left"].set_color("#999999")
    ax.spines["bottom"].set_color("#999999")
    ax.margins(x=0.05)

    annotation_lines = [
        f"Win % r = {win_r:.3f}" if pd.notna(win_r) and win_mask.sum() >= 2 else "Win % r = N/A"
    ]
    if show_line_movement:
        annotation_lines.append(
            f"Line Mvt r = {mvt_r:.3f}" if pd.notna(mvt_r) and mvt_mask.sum() >= 2 else "Line Mvt r = N/A"
        )

    ax.text(
        0.02,
        0.98,
        "\n".join(annotation_lines),
        transform=ax.transAxes,
        fontsize=12.5,
        fontweight="bold",
        va="top",
        ha="left",
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="#D7D7D7",
            alpha=0.96
        )
    )

    summary_text = (
        f"TOTAL SAMPLE: {int(totals_row['Sample'])}   |   "
        f"RECORD: {int(totals_row['Wins'])}-{int(totals_row['Losses'])}   |   "
        f"WIN %: {totals_row['Win %']:.1%}   |   "
        f"NET UNITS: {totals_row['Net Units']:.1f}u"
    )
    if pd.notna(totals_row["Avg Spread Line Mvt"]):
        summary_text += f"   |   AVG MVT: {totals_row['Avg Spread Line Mvt']:.2f}"

    fig.text(
        0.5,
        0.945,
        summary_text,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        bbox=dict(facecolor="#F6F6F6", edgecolor="#D0D0D0", boxstyle="round,pad=0.40")
    )

    if int(totals_row["Sample"]) < 100:
        fig.text(
            0.5,
            0.912,
            f"Warning: small sample after filters (n={int(totals_row['Sample'])})",
            ha="center",
            va="center",
            fontsize=10.5,
            fontweight="bold",
            color="#B45309"
        )

    y1_low, y1_high = ax.get_ylim()
    y_range = y1_high - y1_low
    top_offset = y_range * 0.015
    bottom_1 = y1_low + y_range * 0.05
    bottom_2 = y1_low + y_range * 0.10
    bottom_3 = y1_low + y_range * 0.15

    for i, (bar, wp, n, nu, lm) in enumerate(zip(bars, win_pct_vals, sample_vals, net_unit_vals, line_mvt_vals)):
        x_text = bar.get_x() + bar.get_width() / 2

        if pd.notna(wp):
            ax.text(
                x_text,
                wp + top_offset,
                f"{wp:.1%}",
                ha="center",
                va="bottom",
                fontsize=11,
                fontweight="bold"
            )
        else:
            ax.text(
                x_text,
                top_offset + y1_low + y_range * 0.005,
                "—",
                ha="center",
                va="bottom",
                fontsize=12,
                fontweight="bold",
                color="#888888"
            )

        if show_line_movement and pd.notna(lm):
            ax.text(
                x_text,
                bottom_1,
                f"mvt={lm:.2f}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )

        ax.text(
            x_text,
            bottom_2 if show_line_movement else bottom_1,
            f"n={int(n)}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

        if pd.notna(nu):
            label_text = f"{nu:.1f}u"
            if i == best_idx:
                label_text = f"★ {nu:.1f}u"
            ax.text(
                x_text,
                bottom_3 if show_line_movement else bottom_2,
                label_text,
                ha="center",
                va="bottom",
                fontsize=11,
                fontweight="bold"
            )

    handles = []
    labels_legend = []

    if win_trend_handle is not None:
        handles.append(win_trend_handle)
        labels_legend.append("Win % Trend")

    if show_line_movement and line_handle is not None:
        handles.append(line_handle)
        labels_legend.append("Avg Spread Line Mvt")

    if handles:
        ax.legend(handles, labels_legend, loc="upper right", fontsize=12, frameon=True)

    footer_text = (
        f"BEST NET UNITS: {best_row['X']}   |   "
        f"Record: {int(best_row['Wins'])}-{int(best_row['Losses'])}   |   "
        f"Win %: {best_row['Win %']:.1%}   |   "
        f"Net Units: {best_row['Net Units']:.1f}u   |   "
        f"Sample: {int(best_row['Sample'])}"
    )

    fig.text(
        0.5,
        0.028,
        footer_text,
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        bbox=dict(
            facecolor="#EEF6FF",
            edgecolor="#94BCE5",
            boxstyle="round,pad=0.70",
            linewidth=1.8
        )
    )

    plt.tight_layout(rect=[0.03, 0.10, 0.97, 0.90])
    plt.show()

    # Build the summary table shown under the chart
    show_df = results_df.copy()
    show_df = pd.concat([show_df, pd.DataFrame([totals_row])], ignore_index=True)

    show_df["Win %"] = show_df["Win %"].map(lambda v: "" if pd.isna(v) else f"{v:.1%}")
    show_df["Net Units"] = show_df["Net Units"].map(lambda v: "" if pd.isna(v) else f"{v:.1f}")
    show_df["Avg Spread Line Mvt"] = show_df["Avg Spread Line Mvt"].map(lambda v: "" if pd.isna(v) else f"{v:.2f}")

    show_df = show_df[
        ["X", "Wins", "Losses", "Win %", "Net Units", "Avg Spread Line Mvt", "Sample"]
    ].copy()

    show_df.columns = [
        "Group",
        "Wins",
        "Losses",
        "Win %",
        "Net Units",
        "Avg Line Mvt",
        "Sample"
    ]

    best_group = best_row["X"]

    # Add alternating row shading for readability
    def zebra_rows(df):
        styles = pd.DataFrame("", index=df.index, columns=df.columns)
        for i in range(len(df)):
            if df.iloc[i]["Group"] != "TOTAL":
                bg = "#FAFAFA" if i % 2 == 0 else "#FFFFFF"
                styles.iloc[i, :] = f"background-color:{bg};"
        return styles

    # Color positive and negative unit values
    def color_units(val):
        try:
            v = float(val)
        except:
            return ""

        if v > 0:
            return "color:#0B7D3E; font-weight:700;"
        if v < 0:
            return "color:#B42318;"
        return ""

    # Highlight the best-performing row and the total row
    def highlight_rows(row):
        styles = [""] * len(row)

        if row["Group"] == best_group:
            styles = [
                "background-color:#EAF4FF; font-weight:700;"
            ] * len(row)

        if row["Group"] == "TOTAL":
            styles = [
                "background-color:#E8F5E9; font-weight:800; border-top:4px solid #333;"
            ] * len(row)

        return styles

    styled_table = (
        show_df.style
        .hide(axis="index")
        .apply(zebra_rows, axis=None)
        .apply(highlight_rows, axis=1)
        .map(color_units, subset=["Net Units"])
        .set_properties(**{
            "text-align": "center",
            "font-size": "17px",
            "padding": "14px 18px",
            "border": "none"
        })
        .set_table_styles([
            {
                "selector": "table",
                "props": [
                    ("margin-left", "auto"),
                    ("margin-right", "auto"),
                    ("margin-top", "50px"),
                    ("margin-bottom", "20px"),
                    ("min-width", "1150px"),
                    ("border-collapse", "separate"),
                    ("border-spacing", "0"),
                    ("font-family", "Arial, Helvetica, sans-serif"),
                    ("background-color", "white"),
                    ("border", "1px solid #DADADA"),
                    ("border-radius", "10px"),
                    ("overflow", "hidden"),
                    ("box-shadow", "0 4px 14px rgba(0,0,0,0.08)")
                ]
            },
            {
                "selector": "thead th",
                "props": [
                    ("background-color", "#1F1F1F"),
                    ("color", "white"),
                    ("font-size", "17px"),
                    ("font-weight", "700"),
                    ("text-align", "center"),
                    ("padding", "14px 18px"),
                    ("border-bottom", "2px solid #111")
                ]
            },
            {
                "selector": "tbody td",
                "props": [
                    ("text-align", "center"),
                    ("padding", "14px 18px"),
                    ("font-size", "17px")
                ]
            },
            {
                "selector": "tbody td.col0",
                "props": [
                    ("font-weight", "700"),
                    ("text-align", "left"),
                    ("padding-left", "20px")
                ]
            }
        ])
    )

    display(HTML("<div style='height:40px'></div>"))
    display(styled_table)

    return True

# Widget styling shared across dropdowns
style = {"description_width": "145px"}

wide_dd = widgets.Layout(width="320px")
med_dd = widgets.Layout(width="260px")
month_dd = widgets.Layout(width="220px")

# Main x-axis selector for the chart
x_axis_dd = widgets.Dropdown(
    options=[
        "Confidence Threshold",
        "Year",
        "Month",
        "Pace Matchup",
        "Conference Group Matchup",
        "Bet Type",
    ],
    value="Confidence Threshold",
    description="X Axis",
    style=style,
    layout=wide_dd
)

# Optional filter for home/away spread plays
bet_type_dd = widgets.Dropdown(
    options=["ALL", "HOME", "AWAY"],
    value="ALL",
    description="Bet Type",
    style=style,
    layout=med_dd
)

# Minimum confidence included in the report
conf_min_dd = widgets.Dropdown(
    options=[
        ("ALL (0)", 0),
        ("1", 1),
        ("2", 2),
        ("3", 3),
        ("4", 4),
        ("5", 5),
        ("6", 6),
        ("7", 7),
    ],
    value=0,
    description="Min Included Conf",
    style=style,
    layout=wide_dd
)

# Season range filters
start_year_dd = widgets.Dropdown(
    options=years,
    value=years[0],
    description="Start Season",
    style=style,
    layout=med_dd
)

end_year_dd = widgets.Dropdown(
    options=years,
    value=years[-1],
    description="End Season",
    style=style,
    layout=med_dd
)

# Month range filters
start_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[0],
    description="Start Month",
    style=style,
    layout=month_dd
)

end_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[-1],
    description="End Month",
    style=style,
    layout=month_dd
)

# Kept for layout consistency even though it only has ALL right now
spread_side_dd = widgets.Dropdown(
    options=["ALL"],
    value="ALL",
    description="Spread Side",
    style=style,
    layout=med_dd
)

# Toggle the line movement overlay on and off
show_line_mvt_cb = widgets.Checkbox(
    value=True,
    description="Show Line Movement",
    indent=False,
    layout=widgets.Layout(width="220px")
)

# Run button for the dashboard
submit_btn = widgets.Button(
    description="Run Report",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="170px", height="40px")
)

# Build the checkbox panels for conference and pace filters
conf_a_boxes, conf_a_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
conf_b_boxes, conf_b_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
pace_a_boxes, pace_a_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)
pace_b_boxes, pace_b_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)

output = widgets.Output()

# Revert the run button whenever the user changes a control
spread_side_dd.observe(update_spread_side_state, names="value")

all_reset_widgets = [
    x_axis_dd,
    bet_type_dd,
    conf_min_dd,
    start_year_dd,
    end_year_dd,
    start_month_dd,
    end_month_dd,
    spread_side_dd,
    show_line_mvt_cb,
]

for w in all_reset_widgets:
    w.observe(reset_run_button, names="value")

for box in conf_a_boxes + conf_b_boxes + pace_a_boxes + pace_b_boxes:
    box.observe(reset_run_button, names="value")

# Run the report and update the button state based on the result
def on_submit_clicked(_):
    submit_btn.button_style = ""
    submit_btn.description = "Running..."
    submit_btn.icon = "spinner"

    with output:
        output.clear_output(wait=True)

        try:
            ok = run_plot(
                x_axis_mode=x_axis_dd.value,
                start_year=start_year_dd.value,
                end_year=end_year_dd.value,
                start_month=start_month_dd.value,
                end_month=end_month_dd.value,
                conf_team_a=get_checked_values(conf_a_boxes),
                conf_team_b=get_checked_values(conf_b_boxes),
                pace_team_a=get_checked_values(pace_a_boxes),
                pace_team_b=get_checked_values(pace_b_boxes),
                bet_type=bet_type_dd.value,
                min_conf=conf_min_dd.value,
                show_line_movement=show_line_mvt_cb.value
            )

            if ok:
                submit_btn.button_style = "success"
                submit_btn.description = "Report Ready"
                submit_btn.icon = "check"
            else:
                submit_btn.button_style = "warning"
                submit_btn.description = "No Results"
                submit_btn.icon = "info-circle"

        except Exception:
            import traceback
            print("RUN REPORT FAILED\n")
            traceback.print_exc()

            submit_btn.button_style = "danger"
            submit_btn.description = "Error"
            submit_btn.icon = "warning"

submit_btn.on_click(on_submit_clicked)

# Top row of report controls
controls_row_1 = widgets.HBox(
    [x_axis_dd, bet_type_dd, conf_min_dd, start_year_dd, end_year_dd, start_month_dd, end_month_dd],
    layout=widgets.Layout(
        gap="12px",
        margin="0 0 10px 0",
        flex_flow="row wrap",
        justify_content="flex-start",
        align_items="center"
    )
)

# Secondary controls row
controls_row_2 = widgets.HBox(
    [spread_side_dd, show_line_mvt_cb],
    layout=widgets.Layout(
        gap="12px",
        margin="0 0 10px 0",
        align_items="center"
    )
)

# Center the run button
controls_row_3 = widgets.HBox(
    [submit_btn],
    layout=widgets.Layout(
        justify_content="center",
        margin="6px 0 12px 0"
    )
)

# Selection panels for matchup filtering
selection_row = widgets.HBox(
    [
        card("Conference Group — Team A", conf_a_ui, "260px"),
        card("Conference Group — Team B", conf_b_ui, "260px"),
        card("Pace Group — Team A", pace_a_ui, "220px"),
        card("Pace Group — Team B", pace_b_ui, "220px"),
    ],
    layout=widgets.Layout(
        justify_content="center",
        align_items="flex-start",
        gap="8px",
        flex_flow="row wrap"
    )
)

# Final dashboard layout
ui = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 10px 0;'>Spread Confidence Filter Dashboard</h3>"),
    controls_row_1,
    controls_row_2,
    controls_row_3,
    selection_row,
    output
])

display(ui)

In [53]:
# MAE - TOTALS

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

ROOT_DIR = Path(r"C:\Users\djbar\popalgo")
years = [2022, 2023, 2024, 2025, 2026]

month_labels = ["DEC", "JAN", "FEB"]
month_to_num = {"DEC": 1, "JAN": 2, "FEB": 3}

# Load and stack season files
dfs = []
for y in years:
    p = ROOT_DIR / f"season_results_{y}.csv"
    if not p.exists():
        print(f"Missing file: {p}")
        continue
    temp = pd.read_csv(p)
    temp["year"] = y
    dfs.append(temp)

if not dfs:
    raise FileNotFoundError("No season_results_YYYY.csv files were found.")

full_df = pd.concat(dfs, ignore_index=True)

# Standardize month values
if "MONTH" in full_df.columns:
    full_df["MONTH"] = full_df["MONTH"].astype(str).str.upper().str.strip()
    full_df["MONTH"] = full_df["MONTH"].replace({
        "FEB/MAR": "FEB",
        "MARCH": "FEB",
        "MAR": "FEB"
    })
else:
    date_col = next((c for c in ["slate_date", "game_date", "date"] if c in full_df.columns), None)
    if date_col is None:
        raise KeyError("Could not find MONTH column or a usable date column.")
    dt = pd.to_datetime(full_df[date_col], errors="coerce")
    m = dt.dt.month
    full_df["MONTH"] = np.select(
        [m == 12, m == 1, m.isin([2, 3])],
        ["DEC", "JAN", "FEB"],
        default="OTHER"
    )

full_df = full_df[full_df["MONTH"].isin(month_labels)].copy()
full_df["MONTH_NUM"] = full_df["MONTH"].map(month_to_num)

# Conference cleanup
if not {"home_conf_grp", "away_conf_grp"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_conf_grp and away_conf_grp")

full_df["home_conf_grp"] = full_df["home_conf_grp"].astype(str).str.upper().str.strip()
full_df["away_conf_grp"] = full_df["away_conf_grp"].astype(str).str.upper().str.strip()

conf_label_map = {
    "P6": "Power 5",
    "HM2": "High Mid-Majors",
    "MM": "Mid-Majors",
    "LM": "Low Majors",
}

full_df["home_conf_label"] = full_df["home_conf_grp"].map(conf_label_map)
full_df["away_conf_label"] = full_df["away_conf_grp"].map(conf_label_map)

valid_conf_options = ["Power 5", "High Mid-Majors", "Mid-Majors", "Low Majors"]

full_df = full_df[
    full_df["home_conf_label"].isin(valid_conf_options) &
    full_df["away_conf_label"].isin(valid_conf_options)
].copy()

# Pace cleanup
if not {"home_pace_group", "away_pace_group"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_pace_group and away_pace_group")

full_df["home_pace_group"] = full_df["home_pace_group"].astype(str).str.upper().str.strip()
full_df["away_pace_group"] = full_df["away_pace_group"].astype(str).str.upper().str.strip()

bad_pace_vals = {"", "NAN", "NONE", "NULL", "OTHER"}

full_df = full_df[
    ~full_df["home_pace_group"].isin(bad_pace_vals) &
    ~full_df["away_pace_group"].isin(bad_pace_vals)
].copy()

def normalize_pace(val):
    v = str(val).strip().upper()
    if "FAST" in v:
        return "FAST"
    if "MID" in v:
        return "MID"
    if "SLOW" in v:
        return "SLOW"
    return np.nan

full_df["home_pace_clean"] = full_df["home_pace_group"].map(normalize_pace)
full_df["away_pace_clean"] = full_df["away_pace_group"].map(normalize_pace)

full_df = full_df[
    full_df["home_pace_clean"].notna() &
    full_df["away_pace_clean"].notna()
].copy()

pace_options = ["FAST", "MID", "SLOW"]

# Find model total column
model_total_col = next(
    (c for c in ["projected_total", "exp_total", "model_total"] if c in full_df.columns),
    None
)
if model_total_col is None:
    raise KeyError("Could not find a model total column. Tried: projected_total, exp_total, model_total")

required_cols = [
    "total_actual",
    "totals_conf",
    "total_play",
    "total_result",
    model_total_col,
    "open_total",
]
missing = [c for c in required_cols if c not in full_df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

full_df["open_total"] = pd.to_numeric(full_df["open_total"], errors="coerce")
full_df["total_actual"] = pd.to_numeric(full_df["total_actual"], errors="coerce")
full_df["totals_conf"] = pd.to_numeric(full_df["totals_conf"], errors="coerce")
full_df[model_total_col] = pd.to_numeric(full_df[model_total_col], errors="coerce")
full_df["total_play"] = full_df["total_play"].astype(str).str.upper().str.strip()
full_df["total_result"] = pd.to_numeric(full_df["total_result"], errors="coerce")

# Pace matchup groups
pace_matchup_order = [
    "Fast vs Fast",
    "Mid vs Mid",
    "Slow vs Slow",
    "Fast vs Mid",
    "Fast vs Slow",
    "Mid vs Slow",
    "No Fast",
    "No Slow",
]

def get_pace_group_subset(df, group_name):
    home = df["home_pace_clean"]
    away = df["away_pace_clean"]

    if group_name == "Fast vs Fast":
        mask = (home == "FAST") & (away == "FAST")

    elif group_name == "Mid vs Mid":
        mask = (home == "MID") & (away == "MID")

    elif group_name == "Slow vs Slow":
        mask = (home == "SLOW") & (away == "SLOW")

    elif group_name == "Fast vs Mid":
        mask = (
            ((home == "FAST") & (away == "MID")) |
            ((home == "MID") & (away == "FAST"))
        )

    elif group_name == "Fast vs Slow":
        mask = (
            ((home == "FAST") & (away == "SLOW")) |
            ((home == "SLOW") & (away == "FAST"))
        )

    elif group_name == "Mid vs Slow":
        mask = (
            ((home == "MID") & (away == "SLOW")) |
            ((home == "SLOW") & (away == "MID"))
        )

    elif group_name == "No Fast":
        mask = home.isin({"MID", "SLOW"}) & away.isin({"MID", "SLOW"})

    elif group_name == "No Slow":
        mask = home.isin({"FAST", "MID"}) & away.isin({"FAST", "MID"})

    else:
        raise ValueError(f"Unsupported pace group: {group_name}")

    return df[mask].copy()
pace_matchup_order = [
    "Fast vs Fast",
    "Mid vs Mid",
    "Slow vs Slow",
    "Fast vs Slow",
    "No Fast",
    "No Slow",
]

# Conference group order
conf_matchup_order = [
    "Top 2 vs Top 2",
    "Top 2 vs Bottom 2",
    "Bottom 2 vs Bottom 2",
    "P5 vs P5",
    "Mid 2 vs Mid 2",
    "LM vs LM",
    "No P5",
    "No LM",
]

# Widget helpers
def make_checkbox_group(options, selected=None, box_width="220px"):
    if selected is None:
        selected = options
    boxes = []
    for opt in options:
        cb = widgets.Checkbox(
            value=(opt in selected),
            description=opt,
            indent=False,
            layout=widgets.Layout(width=box_width)
        )
        boxes.append(cb)
    return boxes, widgets.VBox(boxes)

def get_checked_values(boxes):
    return [b.description for b in boxes if b.value]

def card(title, body, width="250px"):
    return widgets.VBox(
        [widgets.HTML(f"<b>{title}</b>"), body],
        layout=widgets.Layout(
            width=width,
            border="1px solid #d9d9d9",
            padding="10px",
            margin="4px"
        )
    )

# Summary helpers
def calc_pct_diff(model_mae, book_mae):
    if pd.isna(model_mae) or pd.isna(book_mae) or model_mae == 0:
        return np.nan
    return ((book_mae - model_mae) / model_mae) * 100

def summarize_subset(subset, label):
    n = len(subset)

    model_mae = np.nan if n == 0 else subset["model_abs_error"].mean()
    book_mae = np.nan if n == 0 else subset["book_abs_error"].mean()
    pct_diff = calc_pct_diff(model_mae, book_mae)

    return {
        "X": label,
        "Model MAE": model_mae,
        "Book MAE": book_mae,
        "% Diff": pct_diff,
        "Sample": n
    }

def get_conf_group_subset(df, group_name):
    top2 = {"Power 5", "High Mid-Majors"}
    bottom2 = {"Mid-Majors", "Low Majors"}
    mid2 = {"High Mid-Majors", "Mid-Majors"}

    home = df["home_conf_label"]
    away = df["away_conf_label"]

    if group_name == "Top 2 vs Top 2":
        mask = home.isin(top2) & away.isin(top2)

    elif group_name == "Top 2 vs Bottom 2":
        mask = (
            (home.isin(top2) & away.isin(bottom2)) |
            (home.isin(bottom2) & away.isin(top2))
        )

    elif group_name == "Bottom 2 vs Bottom 2":
        mask = home.isin(bottom2) & away.isin(bottom2)

    elif group_name == "P5 vs P5":
        mask = (home == "Power 5") & (away == "Power 5")

    elif group_name == "Mid 2 vs Mid 2":
        mask = home.isin(mid2) & away.isin(mid2)

    elif group_name == "LM vs LM":
        mask = (home == "Low Majors") & (away == "Low Majors")

    elif group_name == "No P5":
        mask = (home != "Power 5") & (away != "Power 5")

    elif group_name == "No LM":
        mask = (home != "Low Majors") & (away != "Low Majors")

    else:
        raise ValueError(f"Unsupported conference group: {group_name}")

    return df[mask].copy()

def build_grouped_results(df, x_axis_mode):
    rows = []

    if x_axis_mode == "Year":
        order = [y for y in years if y in set(df["year"].dropna().unique())]
        for yr in order:
            rows.append(summarize_subset(df[df["year"] == yr], str(int(yr))))

    elif x_axis_mode == "Month":
        for mon in ["DEC", "JAN", "FEB"]:
            rows.append(summarize_subset(df[df["MONTH"] == mon], mon))

    elif x_axis_mode == "Pace Matchup":
        for grp in pace_matchup_order:
            grp_df = get_pace_group_subset(df, grp)
            rows.append(summarize_subset(grp_df, grp))

    elif x_axis_mode == "Conf. Matchup Group":
        for grp in conf_matchup_order:
            grp_df = get_conf_group_subset(df, grp)
            rows.append(summarize_subset(grp_df, grp))

    elif x_axis_mode == "Bet Type":
        rows.append(summarize_subset(df[df["total_play"] == "OVER"], "OVER"))
        rows.append(summarize_subset(df[df["total_play"] == "UNDER"], "UNDER"))

    else:
        raise ValueError(f"Unsupported x_axis_mode: {x_axis_mode}")

    return pd.DataFrame(rows)

# Main plotting function
def run_plot(
    x_axis_mode,
    start_year,
    end_year,
    start_month,
    end_month,
    conf_team_a,
    conf_team_b,
    pace_team_a,
    pace_team_b,
    bet_type,
    show_model_mae
):
    if start_year > end_year:
        print("Start season must be <= end season.")
        return

    if month_to_num[start_month] > month_to_num[end_month]:
        print("Start month must be <= end month.")
        return

    if len(conf_team_a) == 0 or len(conf_team_b) == 0:
        print("Pick at least one conference group for Team A and Team B.")
        return

    if len(pace_team_a) == 0 or len(pace_team_b) == 0:
        print("Pick at least one pace group for Team A and Team B.")
        return

    df = full_df[
        (full_df["year"] >= start_year) &
        (full_df["year"] <= end_year) &
        (full_df["MONTH_NUM"] >= month_to_num[start_month]) &
        (full_df["MONTH_NUM"] <= month_to_num[end_month])
    ].copy()

    conf_mask = (
        (df["home_conf_label"].isin(conf_team_a) & df["away_conf_label"].isin(conf_team_b)) |
        (df["home_conf_label"].isin(conf_team_b) & df["away_conf_label"].isin(conf_team_a))
    )
    df = df[conf_mask].copy()

    pace_mask = (
        (df["home_pace_clean"].isin(pace_team_a) & df["away_pace_clean"].isin(pace_team_b)) |
        (df["home_pace_clean"].isin(pace_team_b) & df["away_pace_clean"].isin(pace_team_a))
    )
    df = df[pace_mask].copy()

    df = df[
        df["total_play"].isin(["OVER", "UNDER"]) &
        df["total_result"].isin([1, -1]) &
        df["open_total"].notna() &
        df["total_actual"].notna() &
        df[model_total_col].notna() &
        df["totals_conf"].notna()
    ].copy()

    if bet_type != "ALL":
        df = df[df["total_play"] == bet_type].copy()

    if df.empty:
        print("No rows for this filter.")
        return

    df["model_abs_error"] = (df[model_total_col] - df["total_actual"]).abs()
    df["book_abs_error"] = (df["open_total"] - df["total_actual"]).abs()

    results_df = build_grouped_results(df, x_axis_mode)
    results_df = results_df[results_df["Sample"] > 0].copy()

    if results_df.empty:
        print("No valid grouped results.")
        return

    results_df = results_df.sort_values(
        ["Book MAE", "Sample", "X"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    totals_row = summarize_subset(df, "TOTAL")

    valid_best = results_df[results_df["% Diff"].notna()].copy()
    best_diff_label = None
    if not valid_best.empty:
        best_diff_label = valid_best.sort_values(
            ["% Diff", "Sample"],
            ascending=[False, False]
        ).iloc[0]["X"]

    labels = results_df["X"].tolist()
    model_vals = results_df["Model MAE"].to_numpy(dtype=float)
    book_vals = results_df["Book MAE"].to_numpy(dtype=float)
    sample_vals = results_df["Sample"].to_numpy(dtype=float)

    x = np.arange(len(labels))
    width = 0.34

    fig_width = max(12, len(labels) * 1.9)
    fig_height = 9.6
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    if show_model_mae:
        model_bars = ax.bar(
            x - width / 2,
            model_vals,
            width=width,
            color="mediumseagreen",
            linewidth=0,
            alpha=0.95,
            label="Model MAE vs Actual",
            zorder=3
        )

        book_bars = ax.bar(
            x + width / 2,
            book_vals,
            width=width,
            color="orange",
            linewidth=0,
            alpha=0.95,
            label="Book Open MAE vs Actual",
            zorder=3
        )

        valid_y = np.concatenate([
            model_vals[~np.isnan(model_vals)],
            book_vals[~np.isnan(book_vals)]
        ])
    else:
        model_bars = []
        book_bars = ax.bar(
            x,
            book_vals,
            width=0.56,
            color="orange",
            linewidth=0,
            alpha=0.95,
            label="Book Open MAE vs Actual",
            zorder=3
        )
        valid_y = book_vals[~np.isnan(book_vals)]

    if len(valid_y) > 0:
        y_max = valid_y.max()
        y_min = valid_y.min()
        y_floor = max(0, y_min * 0.9)
        y_pad = max(0.4, y_max * 0.15)
        ax.set_ylim(y_floor, y_max + y_pad)
    else:
        ax.set_ylim(0, 10)

    rotation = 0 if x_axis_mode in ["Year", "Month", "Bet Type"] else 28

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        fontsize=12,
        fontweight="bold",
        rotation=rotation,
        ha="right" if rotation else "center"
    )
    ax.tick_params(axis="y", labelsize=13)

    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")

    ax.set_ylabel("MAE vs Actual", fontsize=17, fontweight="bold", labelpad=12)
    ax.set_xlabel(x_axis_mode, fontsize=17, fontweight="bold", labelpad=12)

    title_text = "Book Open Total MAE" if not show_model_mae else "Model vs Book Open Total MAE"
    ax.set_title(
        f"{title_text} by {x_axis_mode}\n"
        f"Seasons: {start_year}–{end_year} | Months: {start_month} to {end_month}",
        fontsize=16,
        fontweight="bold",
        pad=20
    )

    ax.grid(axis="y", linestyle=":", linewidth=0.9, alpha=0.75)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.margins(x=0.04)

    y_low, y_high = ax.get_ylim()
    y_range = y_high - y_low
    top_offset = y_range * 0.015
    bottom_n = y_low + y_range * 0.062

    if show_model_mae:
        for i, (mb, bb, mv, bv, n) in enumerate(zip(model_bars, book_bars, model_vals, book_vals, sample_vals)):
            if pd.notna(mv):
                ax.text(
                    mb.get_x() + mb.get_width() / 2,
                    mv + top_offset,
                    f"{mv:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    fontweight="bold"
                )

            if pd.notna(bv):
                ax.text(
                    bb.get_x() + bb.get_width() / 2,
                    bv + top_offset,
                    f"{bv:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    fontweight="bold"
                )

            ax.text(
                x[i],
                bottom_n,
                f"n={int(n)}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )
    else:
        for i, (bb, bv, n) in enumerate(zip(book_bars, book_vals, sample_vals)):
            if pd.notna(bv):
                ax.text(
                    bb.get_x() + bb.get_width() / 2,
                    bv + top_offset,
                    f"{bv:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    fontweight="bold"
                )

            ax.text(
                x[i],
                bottom_n,
                f"n={int(n)}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )

    footer_text = (
        f"TOTAL UNIQUE SAMPLE: {int(totals_row['Sample'])} | "
        f"Book MAE: {totals_row['Book MAE']:.2f}"
    )

    if show_model_mae:
        footer_text += (
            f" | Model MAE: {totals_row['Model MAE']:.2f}"
            f" | % Diff: {totals_row['% Diff']:+.1f}%"
        )

    fig.text(
        0.5,
        0.018,
        footer_text,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        bbox=dict(facecolor="#F3F7F3", edgecolor="#B7D7B7", boxstyle="round,pad=0.45")
    )

    ax.legend(loc="upper right", fontsize=12, frameon=True)
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()

    show_df = results_df.copy()
    show_df = pd.concat([show_df, pd.DataFrame([totals_row])], ignore_index=True)

    show_df["Model MAE"] = show_df["Model MAE"].map(lambda v: "" if pd.isna(v) else f"{v:.2f}")
    show_df["Book MAE"] = show_df["Book MAE"].map(lambda v: "" if pd.isna(v) else f"{v:.2f}")
    show_df["% Diff"] = show_df["% Diff"].map(lambda v: "" if pd.isna(v) else f"{v:+.1f}%")

    show_df = show_df[["X", "Model MAE", "Book MAE", "% Diff", "Sample"]]

    def highlight_rows(row):
        styles = [""] * len(row)
        if best_diff_label is not None and row["X"] == best_diff_label:
            styles = ["background-color: #FFF3BF; font-weight: bold;"] * len(row)
        if row["X"] == "TOTAL":
            styles = ["background-color: #E8F5E9; font-weight: bold; border-top: 2px solid #666;"] * len(row)
        return styles

    display(show_df.style.hide(axis="index").apply(highlight_rows, axis=1))

# Widget setup
style = {"description_width": "95px"}

x_axis_dd = widgets.Dropdown(
    options=[
        "Year",
        "Month",
        "Pace Matchup",
        "Conf. Matchup Group",
        "Bet Type",
    ],
    value="Year",
    description="X Axis",
    style=style,
    layout=widgets.Layout(width="220px")
)

bet_type_dd = widgets.Dropdown(
    options=["ALL", "OVER", "UNDER"],
    value="ALL",
    description="Bet Type",
    style=style,
    layout=widgets.Layout(width="145px")
)

show_model_cb = widgets.Checkbox(
    value=False,
    description="Include Model MAE",
    indent=False,
    layout=widgets.Layout(width="170px")
)

start_year_dd = widgets.Dropdown(
    options=years,
    value=years[0],
    description="Start Season",
    style=style,
    layout=widgets.Layout(width="165px")
)

end_year_dd = widgets.Dropdown(
    options=years,
    value=years[-1],
    description="End Season",
    style=style,
    layout=widgets.Layout(width="165px")
)

start_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[0],
    description="Start Month",
    style=style,
    layout=widgets.Layout(width="155px")
)

end_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[-1],
    description="End Month",
    style=style,
    layout=widgets.Layout(width="155px")
)

submit_btn = widgets.Button(
    description="Run Report",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="140px")
)

conf_a_boxes, conf_a_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
conf_b_boxes, conf_b_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
pace_a_boxes, pace_a_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)
pace_b_boxes, pace_b_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)

output = widgets.Output()

def on_submit_clicked(_):
    with output:
        output.clear_output(wait=True)
        try:
            run_plot(
                x_axis_mode=x_axis_dd.value,
                start_year=start_year_dd.value,
                end_year=end_year_dd.value,
                start_month=start_month_dd.value,
                end_month=end_month_dd.value,
                conf_team_a=get_checked_values(conf_a_boxes),
                conf_team_b=get_checked_values(conf_b_boxes),
                pace_team_a=get_checked_values(pace_a_boxes),
                pace_team_b=get_checked_values(pace_b_boxes),
                bet_type=bet_type_dd.value,
                show_model_mae=show_model_cb.value
            )
        except Exception:
            import traceback
            print("RUN REPORT FAILED\n")
            traceback.print_exc()

submit_btn.on_click(on_submit_clicked)

controls_row_1 = widgets.HBox(
    [x_axis_dd, bet_type_dd, show_model_cb],
    layout=widgets.Layout(gap="12px", margin="0 0 8px 0")
)

controls_row_2 = widgets.HBox(
    [start_year_dd, end_year_dd, start_month_dd, end_month_dd, submit_btn],
    layout=widgets.Layout(gap="12px", margin="0 0 10px 0")
)

selection_row = widgets.HBox(
    [
        card("Conference Group — Team A", conf_a_ui, "260px"),
        card("Conference Group — Team B", conf_b_ui, "260px"),
        card("Pace Group — Team A", pace_a_ui, "220px"),
        card("Pace Group — Team B", pace_b_ui, "220px"),
    ],
    layout=widgets.Layout(gap="10px", margin="0 0 8px 0")
)

ui = widgets.VBox([
    widgets.HTML("<h3>Book Open Total MAE Dashboard</h3>"),
    controls_row_1,
    controls_row_2,
    selection_row,
    output
])

display(ui)

In [54]:
# MAE - SPREADS

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

ROOT_DIR = Path(r"C:\Users\djbar\popalgo")
years = [2022, 2023, 2024, 2025, 2026]

month_labels = ["DEC", "JAN", "FEB"]
month_to_num = {"DEC": 1, "JAN": 2, "FEB": 3}

# Load and stack season files
dfs = []
for y in years:
    p = ROOT_DIR / f"season_results_{y}.csv"
    if not p.exists():
        print(f"Missing file: {p}")
        continue
    temp = pd.read_csv(p)
    temp["year"] = y
    dfs.append(temp)

if not dfs:
    raise FileNotFoundError("No season_results_YYYY.csv files were found.")

full_df = pd.concat(dfs, ignore_index=True)

# Standardize month values
if "MONTH" in full_df.columns:
    full_df["MONTH"] = full_df["MONTH"].astype(str).str.upper().str.strip()
    full_df["MONTH"] = full_df["MONTH"].replace({
        "FEB/MAR": "FEB",
        "MARCH": "FEB",
        "MAR": "FEB"
    })
else:
    date_col = next((c for c in ["slate_date", "game_date", "date"] if c in full_df.columns), None)
    if date_col is None:
        raise KeyError("Could not find MONTH column or a usable date column.")
    dt = pd.to_datetime(full_df[date_col], errors="coerce")
    m = dt.dt.month
    full_df["MONTH"] = np.select(
        [m == 12, m == 1, m.isin([2, 3])],
        ["DEC", "JAN", "FEB"],
        default="OTHER"
    )

full_df = full_df[full_df["MONTH"].isin(month_labels)].copy()
full_df["MONTH_NUM"] = full_df["MONTH"].map(month_to_num)

# Conference cleanup
if not {"home_conf_grp", "away_conf_grp"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_conf_grp and away_conf_grp")

full_df["home_conf_grp"] = full_df["home_conf_grp"].astype(str).str.upper().str.strip()
full_df["away_conf_grp"] = full_df["away_conf_grp"].astype(str).str.upper().str.strip()

conf_label_map = {
    "P6": "Power 5",
    "HM2": "High Mid-Majors",
    "MM": "Mid-Majors",
    "LM": "Low Majors",
}

full_df["home_conf_label"] = full_df["home_conf_grp"].map(conf_label_map)
full_df["away_conf_label"] = full_df["away_conf_grp"].map(conf_label_map)

valid_conf_options = ["Power 5", "High Mid-Majors", "Mid-Majors", "Low Majors"]

full_df = full_df[
    full_df["home_conf_label"].isin(valid_conf_options) &
    full_df["away_conf_label"].isin(valid_conf_options)
].copy()

# Pace cleanup
if not {"home_pace_group", "away_pace_group"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_pace_group and away_pace_group")

full_df["home_pace_group"] = full_df["home_pace_group"].astype(str).str.upper().str.strip()
full_df["away_pace_group"] = full_df["away_pace_group"].astype(str).str.upper().str.strip()

bad_pace_vals = {"", "NAN", "NONE", "NULL", "OTHER"}

full_df = full_df[
    ~full_df["home_pace_group"].isin(bad_pace_vals) &
    ~full_df["away_pace_group"].isin(bad_pace_vals)
].copy()

def normalize_pace(val):
    v = str(val).strip().upper()
    if "FAST" in v:
        return "FAST"
    if "MID" in v:
        return "MID"
    if "SLOW" in v:
        return "SLOW"
    return np.nan

full_df["home_pace_clean"] = full_df["home_pace_group"].map(normalize_pace)
full_df["away_pace_clean"] = full_df["away_pace_group"].map(normalize_pace)

full_df = full_df[
    full_df["home_pace_clean"].notna() &
    full_df["away_pace_clean"].notna()
].copy()

pace_options = ["FAST", "MID", "SLOW"]

# Find model spread column
model_spread_col = next(
    (c for c in ["proj_spread", "home_proj_spread", "model_spread"] if c in full_df.columns),
    None
)
if model_spread_col is None:
    raise KeyError("Could not find a model spread column. Tried: proj_spread, home_proj_spread, model_spread")

required_cols = [
    "home_actual",
    "away_actual",
    "spread_conf",
    "spread_play",
    "spread_result",
    model_spread_col,
    "open_spread_home",
]
missing = [c for c in required_cols if c not in full_df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

full_df["open_spread_home"] = pd.to_numeric(full_df["open_spread_home"], errors="coerce")
full_df["home_actual"] = pd.to_numeric(full_df["home_actual"], errors="coerce")
full_df["away_actual"] = pd.to_numeric(full_df["away_actual"], errors="coerce")
full_df["spread_conf"] = pd.to_numeric(full_df["spread_conf"], errors="coerce")
full_df[model_spread_col] = pd.to_numeric(full_df[model_spread_col], errors="coerce")
full_df["spread_play"] = full_df["spread_play"].astype(str).str.upper().str.strip()
full_df["spread_result"] = pd.to_numeric(full_df["spread_result"], errors="coerce")

# Pace matchup groups
pace_matchup_order = [
    "Fast vs Fast",
    "Mid vs Mid",
    "Slow vs Slow",
    "Fast vs Mid",
    "Fast vs Slow",
    "Mid vs Slow",
    "No Fast",
    "No Slow",
]

def get_pace_group_subset(df, group_name):
    home = df["home_pace_clean"]
    away = df["away_pace_clean"]

    if group_name == "Fast vs Fast":
        mask = (home == "FAST") & (away == "FAST")

    elif group_name == "Mid vs Mid":
        mask = (home == "MID") & (away == "MID")

    elif group_name == "Slow vs Slow":
        mask = (home == "SLOW") & (away == "SLOW")

    elif group_name == "Fast vs Mid":
        mask = (
            ((home == "FAST") & (away == "MID")) |
            ((home == "MID") & (away == "FAST"))
        )

    elif group_name == "Fast vs Slow":
        mask = (
            ((home == "FAST") & (away == "SLOW")) |
            ((home == "SLOW") & (away == "FAST"))
        )

    elif group_name == "Mid vs Slow":
        mask = (
            ((home == "MID") & (away == "SLOW")) |
            ((home == "SLOW") & (away == "MID"))
        )

    elif group_name == "No Fast":
        mask = home.isin({"MID", "SLOW"}) & away.isin({"MID", "SLOW"})

    elif group_name == "No Slow":
        mask = home.isin({"FAST", "MID"}) & away.isin({"FAST", "MID"})

    else:
        raise ValueError(f"Unsupported pace group: {group_name}")

    return df[mask].copy()

# Conference group order
conf_matchup_order = [
    "Top 2 vs Top 2",
    "Top 2 vs Bottom 2",
    "Bottom 2 vs Bottom 2",
    "P5 vs P5",
    "Mid 2 vs Mid 2",
    "LM vs LM",
    "No P5",
    "No LM",
]

# Widget helpers
def make_checkbox_group(options, selected=None, box_width="220px"):
    if selected is None:
        selected = options
    boxes = []
    for opt in options:
        cb = widgets.Checkbox(
            value=(opt in selected),
            description=opt,
            indent=False,
            layout=widgets.Layout(width=box_width)
        )
        boxes.append(cb)
    return boxes, widgets.VBox(boxes)

def get_checked_values(boxes):
    return [b.description for b in boxes if b.value]

def card(title, body, width="250px"):
    return widgets.VBox(
        [widgets.HTML(f"<b>{title}</b>"), body],
        layout=widgets.Layout(
            width=width,
            border="1px solid #d9d9d9",
            padding="10px",
            margin="4px"
        )
    )

# Summary helpers
def calc_pct_diff(model_mae, book_mae):
    if pd.isna(model_mae) or pd.isna(book_mae) or model_mae == 0:
        return np.nan
    return ((book_mae - model_mae) / model_mae) * 100

def summarize_subset(subset, label):
    n = len(subset)

    model_mae = np.nan if n == 0 else subset["model_abs_error"].mean()
    book_mae = np.nan if n == 0 else subset["book_abs_error"].mean()
    pct_diff = calc_pct_diff(model_mae, book_mae)

    return {
        "X": label,
        "Model MAE": model_mae,
        "Book MAE": book_mae,
        "% Diff": pct_diff,
        "Sample": n
    }

def get_conf_group_subset(df, group_name):
    top2 = {"Power 5", "High Mid-Majors"}
    bottom2 = {"Mid-Majors", "Low Majors"}
    mid2 = {"High Mid-Majors", "Mid-Majors"}

    home = df["home_conf_label"]
    away = df["away_conf_label"]

    if group_name == "Top 2 vs Top 2":
        mask = home.isin(top2) & away.isin(top2)

    elif group_name == "Top 2 vs Bottom 2":
        mask = (
            (home.isin(top2) & away.isin(bottom2)) |
            (home.isin(bottom2) & away.isin(top2))
        )

    elif group_name == "Bottom 2 vs Bottom 2":
        mask = home.isin(bottom2) & away.isin(bottom2)

    elif group_name == "P5 vs P5":
        mask = (home == "Power 5") & (away == "Power 5")

    elif group_name == "Mid 2 vs Mid 2":
        mask = home.isin(mid2) & away.isin(mid2)

    elif group_name == "LM vs LM":
        mask = (home == "Low Majors") & (away == "Low Majors")

    elif group_name == "No P5":
        mask = (home != "Power 5") & (away != "Power 5")

    elif group_name == "No LM":
        mask = (home != "Low Majors") & (away != "Low Majors")

    else:
        raise ValueError(f"Unsupported conference group: {group_name}")

    return df[mask].copy()

def build_grouped_results(df, x_axis_mode):
    rows = []

    if x_axis_mode == "Year":
        order = [y for y in years if y in set(df["year"].dropna().unique())]
        for yr in order:
            rows.append(summarize_subset(df[df["year"] == yr], str(int(yr))))

    elif x_axis_mode == "Month":
        for mon in ["DEC", "JAN", "FEB"]:
            rows.append(summarize_subset(df[df["MONTH"] == mon], mon))

    elif x_axis_mode == "Pace Matchup":
        for grp in pace_matchup_order:
            grp_df = get_pace_group_subset(df, grp)
            rows.append(summarize_subset(grp_df, grp))

    elif x_axis_mode == "Conf. Matchup Group":
        for grp in conf_matchup_order:
            grp_df = get_conf_group_subset(df, grp)
            rows.append(summarize_subset(grp_df, grp))

    elif x_axis_mode == "Bet Type":
        rows.append(summarize_subset(df[df["spread_play"] == "HOME"], "HOME"))
        rows.append(summarize_subset(df[df["spread_play"] == "AWAY"], "AWAY"))

    else:
        raise ValueError(f"Unsupported x_axis_mode: {x_axis_mode}")

    return pd.DataFrame(rows)

# Main plotting function
def run_plot(
    x_axis_mode,
    start_year,
    end_year,
    start_month,
    end_month,
    conf_team_a,
    conf_team_b,
    pace_team_a,
    pace_team_b,
    bet_type,
    show_model_mae
):
    if start_year > end_year:
        print("Start season must be <= end season.")
        return

    if month_to_num[start_month] > month_to_num[end_month]:
        print("Start month must be <= end month.")
        return

    if len(conf_team_a) == 0 or len(conf_team_b) == 0:
        print("Pick at least one conference group for Team A and Team B.")
        return

    if len(pace_team_a) == 0 or len(pace_team_b) == 0:
        print("Pick at least one pace group for Team A and Team B.")
        return

    df = full_df[
        (full_df["year"] >= start_year) &
        (full_df["year"] <= end_year) &
        (full_df["MONTH_NUM"] >= month_to_num[start_month]) &
        (full_df["MONTH_NUM"] <= month_to_num[end_month])
    ].copy()

    conf_mask = (
        (df["home_conf_label"].isin(conf_team_a) & df["away_conf_label"].isin(conf_team_b)) |
        (df["home_conf_label"].isin(conf_team_b) & df["away_conf_label"].isin(conf_team_a))
    )
    df = df[conf_mask].copy()

    pace_mask = (
        (df["home_pace_clean"].isin(pace_team_a) & df["away_pace_clean"].isin(pace_team_b)) |
        (df["home_pace_clean"].isin(pace_team_b) & df["away_pace_clean"].isin(pace_team_a))
    )
    df = df[pace_mask].copy()

    df = df[
        df["spread_play"].isin(["HOME", "AWAY"]) &
        df["spread_result"].isin([1, -1]) &
        df["open_spread_home"].notna() &
        df["home_actual"].notna() &
        df["away_actual"].notna() &
        df[model_spread_col].notna() &
        df["spread_conf"].notna()
    ].copy()

    if bet_type != "ALL":
        df = df[df["spread_play"] == bet_type].copy()

    if df.empty:
        print("No rows for this filter.")
        return

    df["actual_spread_home"] = df["away_actual"] - df["home_actual"]
    df["model_abs_error"] = (df[model_spread_col] - df["actual_spread_home"]).abs()
    df["book_abs_error"] = (df["open_spread_home"] - df["actual_spread_home"]).abs()

    results_df = build_grouped_results(df, x_axis_mode)
    results_df = results_df[results_df["Sample"] > 0].copy()

    if results_df.empty:
        print("No valid grouped results.")
        return

    results_df = results_df.sort_values(
        ["Book MAE", "Sample", "X"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    totals_row = summarize_subset(df, "TOTAL")

    valid_best = results_df[results_df["% Diff"].notna()].copy()
    best_diff_label = None
    if not valid_best.empty:
        best_diff_label = valid_best.sort_values(
            ["% Diff", "Sample"],
            ascending=[False, False]
        ).iloc[0]["X"]

    labels = results_df["X"].tolist()
    model_vals = results_df["Model MAE"].to_numpy(dtype=float)
    book_vals = results_df["Book MAE"].to_numpy(dtype=float)
    sample_vals = results_df["Sample"].to_numpy(dtype=float)

    x = np.arange(len(labels))
    width = 0.34

    fig_width = max(12, len(labels) * 1.9)
    fig_height = 9.6
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    if show_model_mae:
        model_bars = ax.bar(
            x - width / 2,
            model_vals,
            width=width,
            color="mediumseagreen",
            linewidth=0,
            alpha=0.95,
            label="Model MAE vs Actual",
            zorder=3
        )

        book_bars = ax.bar(
            x + width / 2,
            book_vals,
            width=width,
            color="skyblue",
            linewidth=0,
            alpha=0.95,
            label="Book Open MAE vs Actual",
            zorder=3
        )

        valid_y = np.concatenate([
            model_vals[~np.isnan(model_vals)],
            book_vals[~np.isnan(book_vals)]
        ])
    else:
        model_bars = []
        book_bars = ax.bar(
            x,
            book_vals,
            width=0.56,
            color="skyblue",
            linewidth=0,
            alpha=0.95,
            label="Book Open MAE vs Actual",
            zorder=3
        )
        valid_y = book_vals[~np.isnan(book_vals)]

    if len(valid_y) > 0:
        y_max = valid_y.max()
        y_min = valid_y.min()
        y_floor = max(0, y_min * 0.9)
        y_pad = max(0.4, y_max * 0.15)
        ax.set_ylim(y_floor, y_max + y_pad)
    else:
        ax.set_ylim(0, 10)

    rotation = 0 if x_axis_mode in ["Year", "Month", "Bet Type"] else 28

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        fontsize=12,
        fontweight="bold",
        rotation=rotation,
        ha="right" if rotation else "center"
    )
    ax.tick_params(axis="y", labelsize=13)

    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")

    ax.set_ylabel("MAE vs Actual", fontsize=17, fontweight="bold", labelpad=12)
    ax.set_xlabel(x_axis_mode, fontsize=17, fontweight="bold", labelpad=12)

    title_text = "Book Open Spread MAE" if not show_model_mae else "Model vs Book Open Spread MAE"
    ax.set_title(
        f"{title_text} by {x_axis_mode}\n"
        f"Seasons: {start_year}–{end_year} | Months: {start_month} to {end_month}",
        fontsize=16,
        fontweight="bold",
        pad=20
    )

    ax.grid(axis="y", linestyle=":", linewidth=0.9, alpha=0.75)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.margins(x=0.04)

    y_low, y_high = ax.get_ylim()
    y_range = y_high - y_low
    top_offset = y_range * 0.015
    bottom_n = y_low + y_range * 0.062

    if show_model_mae:
        for i, (mb, bb, mv, bv, n) in enumerate(zip(model_bars, book_bars, model_vals, book_vals, sample_vals)):
            if pd.notna(mv):
                ax.text(
                    mb.get_x() + mb.get_width() / 2,
                    mv + top_offset,
                    f"{mv:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    fontweight="bold"
                )

            if pd.notna(bv):
                ax.text(
                    bb.get_x() + bb.get_width() / 2,
                    bv + top_offset,
                    f"{bv:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    fontweight="bold"
                )

            ax.text(
                x[i],
                bottom_n,
                f"n={int(n)}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )
    else:
        for i, (bb, bv, n) in enumerate(zip(book_bars, book_vals, sample_vals)):
            if pd.notna(bv):
                ax.text(
                    bb.get_x() + bb.get_width() / 2,
                    bv + top_offset,
                    f"{bv:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    fontweight="bold"
                )

            ax.text(
                x[i],
                bottom_n,
                f"n={int(n)}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )

    footer_text = (
        f"TOTAL UNIQUE SAMPLE: {int(totals_row['Sample'])} | "
        f"Book MAE: {totals_row['Book MAE']:.2f}"
    )

    if show_model_mae:
        footer_text += (
            f" | Model MAE: {totals_row['Model MAE']:.2f}"
            f" | % Diff: {totals_row['% Diff']:+.1f}%"
        )

    fig.text(
        0.5,
        0.018,
        footer_text,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        bbox=dict(facecolor="#F3F7F3", edgecolor="#B7D7B7", boxstyle="round,pad=0.45")
    )

    ax.legend(loc="upper right", fontsize=12, frameon=True)
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()

    show_df = results_df.copy()
    show_df = pd.concat([show_df, pd.DataFrame([totals_row])], ignore_index=True)

    show_df["Model MAE"] = show_df["Model MAE"].map(lambda v: "" if pd.isna(v) else f"{v:.2f}")
    show_df["Book MAE"] = show_df["Book MAE"].map(lambda v: "" if pd.isna(v) else f"{v:.2f}")
    show_df["% Diff"] = show_df["% Diff"].map(lambda v: "" if pd.isna(v) else f"{v:+.1f}%")

    show_df = show_df[["X", "Model MAE", "Book MAE", "% Diff", "Sample"]]

    def highlight_rows(row):
        styles = [""] * len(row)
        if best_diff_label is not None and row["X"] == best_diff_label:
            styles = ["background-color: #FFF3BF; font-weight: bold;"] * len(row)
        if row["X"] == "TOTAL":
            styles = ["background-color: #E8F5E9; font-weight: bold; border-top: 2px solid #666;"] * len(row)
        return styles

    display(show_df.style.hide(axis="index").apply(highlight_rows, axis=1))

# Widget setup
style = {"description_width": "95px"}

x_axis_dd = widgets.Dropdown(
    options=[
        "Year",
        "Month",
        "Pace Matchup",
        "Conf. Matchup Group",
        "Bet Type",
    ],
    value="Year",
    description="X Axis",
    style=style,
    layout=widgets.Layout(width="220px")
)

bet_type_dd = widgets.Dropdown(
    options=["ALL", "HOME", "AWAY"],
    value="ALL",
    description="Bet Type",
    style=style,
    layout=widgets.Layout(width="145px")
)

show_model_cb = widgets.Checkbox(
    value=False,
    description="Include Model MAE",
    indent=False,
    layout=widgets.Layout(width="170px")
)

start_year_dd = widgets.Dropdown(
    options=years,
    value=years[0],
    description="Start Season",
    style=style,
    layout=widgets.Layout(width="165px")
)

end_year_dd = widgets.Dropdown(
    options=years,
    value=years[-1],
    description="End Season",
    style=style,
    layout=widgets.Layout(width="165px")
)

start_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[0],
    description="Start Month",
    style=style,
    layout=widgets.Layout(width="155px")
)

end_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[-1],
    description="End Month",
    style=style,
    layout=widgets.Layout(width="155px")
)

submit_btn = widgets.Button(
    description="Run Report",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="140px")
)

conf_a_boxes, conf_a_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
conf_b_boxes, conf_b_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
pace_a_boxes, pace_a_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)
pace_b_boxes, pace_b_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)

output = widgets.Output()

def on_submit_clicked(_):
    with output:
        output.clear_output(wait=True)
        try:
            run_plot(
                x_axis_mode=x_axis_dd.value,
                start_year=start_year_dd.value,
                end_year=end_year_dd.value,
                start_month=start_month_dd.value,
                end_month=end_month_dd.value,
                conf_team_a=get_checked_values(conf_a_boxes),
                conf_team_b=get_checked_values(conf_b_boxes),
                pace_team_a=get_checked_values(pace_a_boxes),
                pace_team_b=get_checked_values(pace_b_boxes),
                bet_type=bet_type_dd.value,
                show_model_mae=show_model_cb.value
            )
        except Exception:
            import traceback
            print("RUN REPORT FAILED\n")
            traceback.print_exc()

submit_btn.on_click(on_submit_clicked)

controls_row_1 = widgets.HBox(
    [x_axis_dd, bet_type_dd, show_model_cb],
    layout=widgets.Layout(gap="12px", margin="0 0 8px 0")
)

controls_row_2 = widgets.HBox(
    [start_year_dd, end_year_dd, start_month_dd, end_month_dd, submit_btn],
    layout=widgets.Layout(gap="12px", margin="0 0 10px 0")
)

selection_row = widgets.HBox(
    [
        card("Conference Group — Team A", conf_a_ui, "260px"),
        card("Conference Group — Team B", conf_b_ui, "260px"),
        card("Pace Group — Team A", pace_a_ui, "220px"),
        card("Pace Group — Team B", pace_b_ui, "220px"),
    ],
    layout=widgets.Layout(gap="10px", margin="0 0 8px 0")
)

ui = widgets.VBox([
    widgets.HTML("<h3>Book Open Spread MAE Dashboard</h3>"),
    controls_row_1,
    controls_row_2,
    selection_row,
    output
])

display(ui)

In [55]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML

ROOT_DIR = Path(r"C:\Users\djbar\popalgo")
years = [2022, 2023, 2024, 2025, 2026]

# Months included in the dashboard
month_labels = ["DEC", "JAN", "FEB"]
month_to_num = {"DEC": 1, "JAN": 2, "FEB": 3}

# Load each season file and stack them together
dfs = []
for y in years:
    p = ROOT_DIR / f"season_results_{y}.csv"
    if not p.exists():
        print(f"Missing file: {p}")
        continue
    temp = pd.read_csv(p)
    temp["year"] = y
    dfs.append(temp)

if not dfs:
    raise FileNotFoundError("No season_results_YYYY.csv files were found.")

full_df = pd.concat(dfs, ignore_index=True)

# Clean up the month field if it already exists.
# If not, build it from an available date column.
if "MONTH" in full_df.columns:
    full_df["MONTH"] = full_df["MONTH"].astype(str).str.upper().str.strip()
    full_df["MONTH"] = full_df["MONTH"].replace({
        "FEB/MAR": "FEB",
        "MARCH": "FEB",
        "MAR": "FEB"
    })
else:
    date_col = next((c for c in ["slate_date", "game_date", "date"] if c in full_df.columns), None)
    if date_col is None:
        raise KeyError("Could not find MONTH column or a usable date column.")
    dt = pd.to_datetime(full_df[date_col], errors="coerce")
    m = dt.dt.month
    full_df["MONTH"] = np.select(
        [m == 12, m == 1, m.isin([2, 3])],
        ["DEC", "JAN", "FEB"],
        default="OTHER"
    )

full_df = full_df[full_df["MONTH"].isin(month_labels)].copy()
full_df["MONTH_NUM"] = full_df["MONTH"].map(month_to_num)

# Standardize conference group fields and map them to readable labels
if not {"home_conf_grp", "away_conf_grp"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_conf_grp and away_conf_grp")

full_df["home_conf_grp"] = full_df["home_conf_grp"].astype(str).str.upper().str.strip()
full_df["away_conf_grp"] = full_df["away_conf_grp"].astype(str).str.upper().str.strip()

conf_label_map = {
    "P6": "Power 5",
    "HM2": "High Mid-Majors",
    "MM": "Mid-Majors",
    "LM": "Low Majors",
}

full_df["home_conf_label"] = full_df["home_conf_grp"].map(conf_label_map)
full_df["away_conf_label"] = full_df["away_conf_grp"].map(conf_label_map)

valid_conf_options = ["Power 5", "High Mid-Majors", "Mid-Majors", "Low Majors"]

full_df = full_df[
    full_df["home_conf_label"].isin(valid_conf_options) &
    full_df["away_conf_label"].isin(valid_conf_options)
].copy()

# Clean up pace labels and drop unusable values
if not {"home_pace_group", "away_pace_group"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_pace_group and away_pace_group")

full_df["home_pace_group"] = full_df["home_pace_group"].astype(str).str.upper().str.strip()
full_df["away_pace_group"] = full_df["away_pace_group"].astype(str).str.upper().str.strip()

bad_pace_vals = {"", "NAN", "NONE", "NULL", "OTHER"}
full_df = full_df[
    ~full_df["home_pace_group"].isin(bad_pace_vals) &
    ~full_df["away_pace_group"].isin(bad_pace_vals)
].copy()

pace_options = sorted(
    set(full_df["home_pace_group"].dropna().unique()).union(
        set(full_df["away_pace_group"].dropna().unique())
    )
)

# Make sure the key totals columns are present
required_cols = [
    "open_total",
    "close_total",
    "projected_total",
    "totals_conf",
    "total_play",
    "total_result",
]
missing = [c for c in required_cols if c not in full_df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

full_df["open_total"] = pd.to_numeric(full_df["open_total"], errors="coerce")
full_df["close_total"] = pd.to_numeric(full_df["close_total"], errors="coerce")
full_df["projected_total"] = pd.to_numeric(full_df["projected_total"], errors="coerce")
full_df["totals_conf"] = pd.to_numeric(full_df["totals_conf"], errors="coerce")
full_df["total_result"] = pd.to_numeric(full_df["total_result"], errors="coerce")
full_df["total_play"] = full_df["total_play"].astype(str).str.upper().str.strip()

# Build a checkbox group and return both the boxes and the UI container
def make_checkbox_group(options, selected=None, box_width="220px"):
    if selected is None:
        selected = options
    boxes = []
    for opt in options:
        cb = widgets.Checkbox(
            value=(opt in selected),
            description=opt,
            indent=False,
            layout=widgets.Layout(width=box_width)
        )
        boxes.append(cb)
    return boxes, widgets.VBox(boxes)

# Pull the checked values out of a checkbox group
def get_checked_values(boxes):
    return [b.description for b in boxes if b.value]

# Simple card layout used for the filter panels
def card(title, body, width="250px"):
    return widgets.VBox(
        [widgets.HTML(f"<b>{title}</b>"), body],
        layout=widgets.Layout(
            width=width,
            border="1px solid #d9d9d9",
            padding="10px",
            margin="4px"
        )
    )

# Reset the run button anytime a filter changes
def reset_run_button(change=None):
    submit_btn.button_style = "primary"
    submit_btn.description = "Run Scatter"
    submit_btn.icon = "bar-chart"

# Hide the open total value box whenever the filter is set to ALL
def update_open_total_state(change=None):
    is_all = (open_total_op_dd.value == "ALL")
    open_total_val.disabled = is_all
    open_total_val.layout.display = "none" if is_all else "block"
    reset_run_button()

# Measure how much the market moved toward the model from open to close
def compute_pct_toward_model(df):
    d = df.copy()

    d["open_gap"] = (d["projected_total"] - d["open_total"]).abs()
    d["close_gap"] = (d["projected_total"] - d["close_total"]).abs()

    d = d[d["open_gap"].notna() & d["close_gap"].notna()].copy()
    d = d[d["open_gap"] > 0].copy()
    d = d[d["open_gap"] >= 0.5].copy()

    d["pct_toward_model"] = ((d["open_gap"] - d["close_gap"]) / d["open_gap"]) * 100.0
    d["pct_toward_model"] = d["pct_toward_model"].clip(-200, 200)

    return d

# Render the summary table in a centered HTML block
def render_centered_table(df_show):
    if df_show.empty:
        display(HTML("<div style='text-align:center; margin-top:25px; font-size:16px; font-weight:600;'>No table rows to display.</div>"))
        return

    cols = list(df_show.columns)
    metric_col = cols[1]

    temp = df_show[df_show["Confidence Score"] != "TOTAL"].copy()

    try:
        metric_vals = (
            temp[metric_col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float)
        )
        best_idx = metric_vals.idxmax()
    except Exception:
        best_idx = None

    html = """
    <div style="display:flex; justify-content:center; margin-top:34px; margin-bottom:10px; width:100%;">
        <table style="
            min-width:950px;
            border-collapse:separate;
            border-spacing:0;
            font-family:Arial, Helvetica, sans-serif;
            background:white;
            border:1px solid #DADADA;
            border-radius:12px;
            overflow:hidden;
            box-shadow:0 4px 14px rgba(0,0,0,0.08);
            text-align:center;
        ">
            <thead>
                <tr>
    """

    for c in cols:
        html += f"""
            <th style="background:#1F1F1F; color:white; padding:15px 20px; font-size:17px; font-weight:700; border-bottom:2px solid #111;">{c}</th>
        """

    html += """
                </tr>
            </thead>
            <tbody>
    """

    for i, (idx, row) in enumerate(df_show.iterrows()):
        if row["Confidence Score"] == "TOTAL":
            bg = "#E8E8E8"
            font_weight = "700"
        elif idx == best_idx:
            bg = "#DFF5E1"
            font_weight = "700"
        else:
            bg = "#FAFAFA" if i % 2 == 0 else "white"
            font_weight = "600"

        html += f"""<tr style="background:{bg}; font-weight:{font_weight};">"""
        for c in cols:
            html += f"""
                <td style="padding:15px 20px; font-size:17px;">{row[c]}</td>
            """
        html += "</tr>"

    html += """
            </tbody>
        </table>
    </div>
    """
    display(HTML(html))

# Main function that filters the data, builds the chart, and shows the table
def run_scatter(
    start_year,
    end_year,
    start_month,
    end_month,
    conf_team_a,
    conf_team_b,
    pace_team_a,
    pace_team_b,
    bet_type,
    total_filter_op,
    total_filter_value,
    min_conf,
    y_axis_metric
):
    if start_year > end_year:
        print("Start season must be <= end season.")
        return False

    if month_to_num[start_month] > month_to_num[end_month]:
        print("Start month must be <= end month.")
        return False

    if len(conf_team_a) == 0 or len(conf_team_b) == 0:
        print("Pick at least one conference group for Team A and Team B.")
        return False

    if len(pace_team_a) == 0 or len(pace_team_b) == 0:
        print("Pick at least one pace group for Team A and Team B.")
        return False

    df = full_df[
        (full_df["year"] >= start_year) &
        (full_df["year"] <= end_year) &
        (full_df["MONTH_NUM"] >= month_to_num[start_month]) &
        (full_df["MONTH_NUM"] <= month_to_num[end_month])
    ].copy()

    conf_mask = (
        (df["home_conf_label"].isin(conf_team_a) & df["away_conf_label"].isin(conf_team_b)) |
        (df["home_conf_label"].isin(conf_team_b) & df["away_conf_label"].isin(conf_team_a))
    )
    df = df[conf_mask].copy()

    pace_mask = (
        (df["home_pace_group"].isin(pace_team_a) & df["away_pace_group"].isin(pace_team_b)) |
        (df["home_pace_group"].isin(pace_team_b) & df["away_pace_group"].isin(pace_team_a))
    )
    df = df[pace_mask].copy()

    df = df[
        df["total_play"].isin(["OVER", "UNDER"]) &
        df["total_result"].isin([1, -1]) &
        df["totals_conf"].notna() &
        df["open_total"].notna() &
        df["close_total"].notna() &
        df["projected_total"].notna()
    ].copy()

    if bet_type != "ALL":
        df = df[df["total_play"] == bet_type].copy()

    if total_filter_op != "ALL":
        if total_filter_op == ">=":
            df = df[df["open_total"] >= total_filter_value].copy()
        elif total_filter_op == "<=":
            df = df[df["open_total"] <= total_filter_value].copy()

    if min_conf > 0:
        df = df[df["totals_conf"] >= min_conf].copy()

    if df.empty:
        print("No rows for this filter.")
        return False

    # Drop the very top end of confidence scores so a few extremes do not distort the view
    cutoff = df["totals_conf"].quantile(0.97)
    df = df[df["totals_conf"] <= cutoff].copy()

    if df.empty:
        print("No rows remain after top-3% confidence removal.")
        return False

    df = compute_pct_toward_model(df)

    if df.empty:
        print("No rows remain after % toward model calculation.")
        return False

    # Build the year-by-year threshold summary.
    # Each point is cumulative, so 4 means all plays with confidence 4 and up.
    thresholds = sorted(pd.Series(df["totals_conf"].dropna().unique()).astype(float).astype(int).unique())
    thresholds = [t for t in thresholds if 1 <= t <= 6]

    if not thresholds:
        print("No usable totals_conf thresholds found in the 1% to 6% range.")
        return False

    rows = []
    years_in_range = [y for y in years if start_year <= y <= end_year]

    for yr in years_in_range:
        yr_df = df[df["year"] == yr].copy()

        for t in thresholds:
            subset = yr_df[yr_df["totals_conf"] >= t].copy()
            if subset.empty:
                continue

            wins = int((subset["total_result"] == 1).sum())
            losses = int((subset["total_result"] == -1).sum())
            sample = len(subset)
            win_pct = (wins / sample) * 100.0 if sample > 0 else np.nan
            line_move = subset["pct_toward_model"].mean()
            net_units = wins - (losses * 1.1)

            if y_axis_metric == "Line % Movement":
                y_value = line_move
            elif y_axis_metric == "Winning %":
                y_value = win_pct
            else:
                y_value = net_units

            rows.append({
                "Confidence Score": t,
                "Year": yr,
                "Y Value": y_value,
                "Line % Movement": line_move,
                "Wins": wins,
                "Losses": losses,
                "Win %": win_pct,
                "Net Units": net_units,
                "Total Sample Size": sample
            })

    scatter_df = pd.DataFrame(rows)

    if scatter_df.empty:
        print("No valid results after filtering.")
        return False

    # Build the summary table across all selected years straight from the raw filtered rows
    table_rows = []

    for t in thresholds:
        subset_all = df[df["totals_conf"] >= t].copy()
        if subset_all.empty:
            continue

        total_wins = int((subset_all["total_result"] == 1).sum())
        total_losses = int((subset_all["total_result"] == -1).sum())
        total_sample = len(subset_all)
        total_net_units = total_wins - (total_losses * 1.1)
        total_win_pct = (total_wins / total_sample) * 100.0 if total_sample > 0 else np.nan
        avg_line_move = subset_all["pct_toward_model"].mean()

        if y_axis_metric == "Line % Movement":
            row = {
                "Confidence Score": f"{int(t)}%",
                "Line % Movement": "" if pd.isna(avg_line_move) else f"{avg_line_move:.1f}%",
                "Wins": f"{total_wins:,}",
                "Losses": f"{total_losses:,}",
                "Win %": "" if pd.isna(total_win_pct) else f"{total_win_pct:.1f}%",
                "Net Units": f"{total_net_units:.1f}",
                "Total Sample Size": f"{total_sample:,}"
            }
        elif y_axis_metric == "Winning %":
            row = {
                "Confidence Score": f"{int(t)}%",
                "Winning %": "" if pd.isna(total_win_pct) else f"{total_win_pct:.1f}%",
                "Wins": f"{total_wins:,}",
                "Losses": f"{total_losses:,}",
                "Net Units": f"{total_net_units:.1f}",
                "Total Sample Size": f"{total_sample:,}"
            }
        else:
            row = {
                "Confidence Score": f"{int(t)}%",
                "Net Units": f"{total_net_units:.1f}",
                "Wins": f"{total_wins:,}",
                "Losses": f"{total_losses:,}",
                "Win %": "" if pd.isna(total_win_pct) else f"{total_win_pct:.1f}%",
                "Total Sample Size": f"{total_sample:,}"
            }

        table_rows.append(row)

    table_df = pd.DataFrame(table_rows)

    # Add one total row using the full filtered sample
    total_wins = int((df["total_result"] == 1).sum())
    total_losses = int((df["total_result"] == -1).sum())
    total_sample = len(df)
    total_net_units = total_wins - (total_losses * 1.1)
    total_win_pct = (total_wins / total_sample) * 100.0 if total_sample > 0 else np.nan
    avg_line_move_total = df["pct_toward_model"].mean()

    if y_axis_metric == "Line % Movement":
        totals_row = {
            "Confidence Score": "TOTAL",
            "Line % Movement": f"{avg_line_move_total:.1f}%",
            "Wins": f"{total_wins:,}",
            "Losses": f"{total_losses:,}",
            "Win %": f"{total_win_pct:.1f}%",
            "Net Units": f"{total_net_units:.1f}",
            "Total Sample Size": f"{total_sample:,}"
        }
    elif y_axis_metric == "Winning %":
        totals_row = {
            "Confidence Score": "TOTAL",
            "Winning %": f"{total_win_pct:.1f}%",
            "Wins": f"{total_wins:,}",
            "Losses": f"{total_losses:,}",
            "Net Units": f"{total_net_units:.1f}",
            "Total Sample Size": f"{total_sample:,}"
        }
    else:
        totals_row = {
            "Confidence Score": "TOTAL",
            "Net Units": f"{total_net_units:.1f}",
            "Wins": f"{total_wins:,}",
            "Losses": f"{total_losses:,}",
            "Win %": f"{total_win_pct:.1f}%",
            "Total Sample Size": f"{total_sample:,}"
        }

    table_df = pd.concat([table_df, pd.DataFrame([totals_row])], ignore_index=True)

    # Build the combined all-years line from the same raw filtered sample
    all_years_rows = []
    for t in thresholds:
        subset_all = df[df["totals_conf"] >= t].copy()
        if subset_all.empty:
            continue

        total_sample_t = len(subset_all)
        total_wins_t = int((subset_all["total_result"] == 1).sum())
        total_losses_t = int((subset_all["total_result"] == -1).sum())
        total_net_units_t = total_wins_t - (total_losses_t * 1.1)

        if y_axis_metric == "Line % Movement":
            all_val = subset_all["pct_toward_model"].mean()
        elif y_axis_metric == "Winning %":
            all_val = (total_wins_t / total_sample_t) * 100.0 if total_sample_t > 0 else np.nan
        else:
            all_val = total_net_units_t

        all_years_rows.append({
            "Confidence Score": t,
            "Y Value": all_val,
            "Total Sample Size": total_sample_t
        })

    all_years_df = pd.DataFrame(all_years_rows).sort_values("Confidence Score").reset_index(drop=True)

    # Plot each season separately, then overlay the combined line
    year_colors = {
        2022: "#1f77b4",
        2023: "#d62728",
        2024: "#2ca02c",
        2025: "#9467bd",
        2026: "#ff7f0e",
    }

    fig, ax = plt.subplots(figsize=(13.5, 8.8))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("#FCFCFC")

    for yr in years_in_range:
        yr_plot = scatter_df[scatter_df["Year"] == yr].sort_values("Confidence Score")
        if yr_plot.empty:
            continue

        ax.plot(
            yr_plot["Confidence Score"],
            yr_plot["Y Value"],
            linewidth=1.8,
            alpha=0.32,
            color=year_colors.get(yr, "gray"),
            zorder=2
        )

        ax.scatter(
            yr_plot["Confidence Score"],
            yr_plot["Y Value"],
            s=135,
            color=year_colors.get(yr, "gray"),
            edgecolor="#2D2D2D",
            linewidth=1.0,
            alpha=0.92,
            label=str(yr),
            zorder=3
        )

    ax.plot(
        all_years_df["Confidence Score"],
        all_years_df["Y Value"],
        color="black",
        linewidth=3.4,
        alpha=0.65,
        zorder=4.2,
        label="All Years"
    )

    ax.scatter(
        all_years_df["Confidence Score"],
        all_years_df["Y Value"],
        s=180,
        color="black",
        edgecolor="white",
        linewidth=1.1,
        alpha=0.95,
        zorder=4.4
    )

    if y_axis_metric == "Line % Movement":
        ax.axhline(
            0,
            linestyle="--",
            linewidth=1.6,
            color="black",
            alpha=0.70,
            zorder=1
        )

    if y_axis_metric == "Winning %":
        ax.axhline(
            52.39,
            linestyle=":",
            linewidth=2.0,
            color="black",
            alpha=0.65,
            zorder=1.5,
            label="Breakeven 52.39%"
        )

    open_total_text = "ALL" if total_filter_op == "ALL" else f"{total_filter_op} {total_filter_value:g}"

    if y_axis_metric == "Line % Movement":
        chart_title_metric = "Avg % Toward Model"
    elif y_axis_metric == "Winning %":
        chart_title_metric = "Winning %"
    else:
        chart_title_metric = "Net Units"

    min_conf_text = "ALL (0%)" if min_conf == 0 else f"{min_conf}%+"

    ax.set_title(
        f"Totals {chart_title_metric} by Confidence Score and Year\n"
        f"Seasons: {start_year}–{end_year} | Months: {start_month} to {end_month} | "
        f"Bet Type: {bet_type} | Open Total: {open_total_text} | Min Included Confidence: {min_conf_text}",
        fontsize=15,
        fontweight="bold",
        pad=18
    )

    ax.set_xlabel("Min Confidence Included", fontsize=16, fontweight="bold", labelpad=10)
    ax.set_ylabel(chart_title_metric, fontsize=16, fontweight="bold", labelpad=10)

    conf_ticks = [t for t in sorted(scatter_df["Confidence Score"].unique()) if 1 <= t <= 6]
    ax.set_xticks(conf_ticks)
    ax.set_xticklabels([f"{int(x)}%" for x in conf_ticks], fontsize=12, fontweight="bold")
    ax.set_xlim(0.8, 6.2)

    ax.tick_params(axis="y", labelsize=12)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")

    if y_axis_metric == "Winning %":
        y_min = min(scatter_df["Y Value"].min(), 52.39) - 2
        y_max = max(scatter_df["Y Value"].max(), 52.39) + 2
        ax.set_ylim(y_min, y_max)

    ax.grid(True, linestyle=":", linewidth=0.9, alpha=0.70)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#999999")
    ax.spines["bottom"].set_color("#999999")

    ax.legend(title="Year / Series", loc="best", fontsize=11, title_fontsize=12, frameon=True)

    plt.tight_layout()
    plt.show()

    display(HTML("<div style='height:30px;'></div>"))
    render_centered_table(table_df)

    return True

# Widget styling
style = {"description_width": "145px"}

med_dd = widgets.Layout(width="260px")
month_dd = widgets.Layout(width="220px")
wide_dd = widgets.Layout(width="320px")
value_box_layout = widgets.Layout(width="120px")

y_axis_dd = widgets.Dropdown(
    options=["Line % Movement", "Winning %", "Net Units"],
    value="Line % Movement",
    description="Y Axis",
    style=style,
    layout=med_dd
)

bet_type_dd = widgets.Dropdown(
    options=["ALL", "OVER", "UNDER"],
    value="ALL",
    description="Bet Type",
    style=style,
    layout=med_dd
)

conf_min_dd = widgets.Dropdown(
    options=[
        ("ALL (0%)", 0),
        ("1%", 1),
        ("2%", 2),
        ("3%", 3),
        ("4%", 4),
        ("5%", 5),
        ("6%", 6),
    ],
    value=0,
    description="Min Included Conf",
    style=style,
    layout=wide_dd
)

start_year_dd = widgets.Dropdown(
    options=years,
    value=years[0],
    description="Start Season",
    style=style,
    layout=med_dd
)

end_year_dd = widgets.Dropdown(
    options=years,
    value=years[-1],
    description="End Season",
    style=style,
    layout=med_dd
)

start_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[0],
    description="Start Month",
    style=style,
    layout=month_dd
)

end_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[-1],
    description="End Month",
    style=style,
    layout=month_dd
)

open_total_op_dd = widgets.Dropdown(
    options=["ALL", ">=", "<="],
    value="ALL",
    description="Open Total",
    style=style,
    layout=med_dd
)

open_total_val = widgets.FloatText(
    value=140.0,
    description="",
    layout=value_box_layout
)

submit_btn = widgets.Button(
    description="Run Scatter",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="170px", height="40px")
)

conf_a_boxes, conf_a_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
conf_b_boxes, conf_b_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
pace_a_boxes, pace_a_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)
pace_b_boxes, pace_b_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)

output = widgets.Output()

# Wire up the widget behavior
open_total_op_dd.observe(update_open_total_state, names="value")
update_open_total_state()

all_reset_widgets = [
    y_axis_dd,
    bet_type_dd,
    conf_min_dd,
    start_year_dd,
    end_year_dd,
    start_month_dd,
    end_month_dd,
    open_total_op_dd,
    open_total_val,
]

for w in all_reset_widgets:
    w.observe(reset_run_button, names="value")

for box in conf_a_boxes + conf_b_boxes + pace_a_boxes + pace_b_boxes:
    box.observe(reset_run_button, names="value")

def on_submit_clicked(_):
    submit_btn.button_style = ""
    submit_btn.description = "Running..."
    submit_btn.icon = "spinner"

    with output:
        output.clear_output(wait=True)

        try:
            ok = run_scatter(
                start_year=start_year_dd.value,
                end_year=end_year_dd.value,
                start_month=start_month_dd.value,
                end_month=end_month_dd.value,
                conf_team_a=get_checked_values(conf_a_boxes),
                conf_team_b=get_checked_values(conf_b_boxes),
                pace_team_a=get_checked_values(pace_a_boxes),
                pace_team_b=get_checked_values(pace_b_boxes),
                bet_type=bet_type_dd.value,
                total_filter_op=open_total_op_dd.value,
                total_filter_value=open_total_val.value,
                min_conf=conf_min_dd.value,
                y_axis_metric=y_axis_dd.value
            )

            if ok:
                submit_btn.button_style = "success"
                submit_btn.description = "Scatter Ready"
                submit_btn.icon = "check"
            else:
                submit_btn.button_style = "warning"
                submit_btn.description = "No Results"
                submit_btn.icon = "info-circle"

        except Exception:
            import traceback
            print("RUN SCATTER FAILED\n")
            traceback.print_exc()

            submit_btn.button_style = "danger"
            submit_btn.description = "Error"
            submit_btn.icon = "warning"

submit_btn.on_click(on_submit_clicked)

# Lay out the controls and display the full UI
controls_row_1 = widgets.HBox(
    [y_axis_dd, bet_type_dd, conf_min_dd, start_year_dd, end_year_dd, start_month_dd, end_month_dd],
    layout=widgets.Layout(
        gap="12px",
        margin="0 0 10px 0",
        flex_flow="row wrap",
        justify_content="flex-start",
        align_items="center"
    )
)

controls_row_2 = widgets.HBox(
    [open_total_op_dd, open_total_val],
    layout=widgets.Layout(
        gap="12px",
        margin="0 0 10px 0",
        align_items="center"
    )
)

controls_row_3 = widgets.HBox(
    [submit_btn],
    layout=widgets.Layout(
        justify_content="center",
        margin="6px 0 12px 0"
    )
)

selection_row = widgets.HBox(
    [
        card("Conference Group — Team A", conf_a_ui, "260px"),
        card("Conference Group — Team B", conf_b_ui, "260px"),
        card("Pace Group — Team A", pace_a_ui, "220px"),
        card("Pace Group — Team B", pace_b_ui, "220px"),
    ],
    layout=widgets.Layout(
        justify_content="center",
        align_items="flex-start",
        gap="8px",
        flex_flow="row wrap"
    )
)

ui = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 10px 0;'>Totals by Confidence Score and Year</h3>"),
    controls_row_1,
    controls_row_2,
    controls_row_3,
    selection_row,
    output
])

display(ui)

In [56]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML

ROOT_DIR = Path(r"C:\Users\djbar\popalgo")
years = [2022, 2023, 2024, 2025, 2026]

# Months included in the analysis window
month_labels = ["DEC", "JAN", "FEB"]
month_to_num = {"DEC": 1, "JAN": 2, "FEB": 3}

# Load each season file and combine everything into one dataframe
dfs = []
for y in years:
    p = ROOT_DIR / f"season_results_{y}.csv"
    if not p.exists():
        print(f"Missing file: {p}")
        continue
    temp = pd.read_csv(p)
    temp["year"] = y
    dfs.append(temp)

if not dfs:
    raise FileNotFoundError("No season_results_YYYY.csv files were found.")

full_df = pd.concat(dfs, ignore_index=True)

# Clean the month field if it exists, otherwise build it from a date column
if "MONTH" in full_df.columns:
    full_df["MONTH"] = full_df["MONTH"].astype(str).str.upper().str.strip()
    full_df["MONTH"] = full_df["MONTH"].replace({
        "FEB/MAR": "FEB",
        "MARCH": "FEB",
        "MAR": "FEB"
    })
else:
    date_col = next((c for c in ["slate_date", "game_date", "date"] if c in full_df.columns), None)
    if date_col is None:
        raise KeyError("Could not find MONTH column or a usable date column.")

    dt = pd.to_datetime(full_df[date_col], errors="coerce")
    m = dt.dt.month

    full_df["MONTH"] = np.select(
        [m == 12, m == 1, m.isin([2, 3])],
        ["DEC", "JAN", "FEB"],
        default="OTHER"
    )

# Keep only the months used for this tool
full_df = full_df[full_df["MONTH"].isin(month_labels)].copy()
full_df["MONTH_NUM"] = full_df["MONTH"].map(month_to_num)

# Standardize conference group fields
if not {"home_conf_grp", "away_conf_grp"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_conf_grp and away_conf_grp")

full_df["home_conf_grp"] = full_df["home_conf_grp"].astype(str).str.upper().str.strip()
full_df["away_conf_grp"] = full_df["away_conf_grp"].astype(str).str.upper().str.strip()

# Turn conference codes into readable labels
conf_label_map = {
    "P6": "Power 5",
    "HM2": "High Mid-Majors",
    "MM": "Mid-Majors",
    "LM": "Low Majors",
}

full_df["home_conf_label"] = full_df["home_conf_grp"].map(conf_label_map)
full_df["away_conf_label"] = full_df["away_conf_grp"].map(conf_label_map)

valid_conf_options = ["Power 5", "High Mid-Majors", "Mid-Majors", "Low Majors"]

full_df = full_df[
    full_df["home_conf_label"].isin(valid_conf_options) &
    full_df["away_conf_label"].isin(valid_conf_options)
].copy()

# Standardize pace group fields
if not {"home_pace_group", "away_pace_group"}.issubset(full_df.columns):
    raise KeyError("Need columns: home_pace_group and away_pace_group")

full_df["home_pace_group"] = full_df["home_pace_group"].astype(str).str.upper().str.strip()
full_df["away_pace_group"] = full_df["away_pace_group"].astype(str).str.upper().str.strip()

bad_pace_vals = {"", "NAN", "NONE", "NULL", "OTHER"}
full_df = full_df[
    ~full_df["home_pace_group"].isin(bad_pace_vals) &
    ~full_df["away_pace_group"].isin(bad_pace_vals)
].copy()

# Build the list of pace options that appear in the data
pace_options = sorted(
    set(full_df["home_pace_group"].dropna().unique()).union(
        set(full_df["away_pace_group"].dropna().unique())
    )
)

# Try to find the model spread column automatically
spread_proj_candidates = [
    "projected_spread",
    "proj_spread",
    "model_spread",
    "spread_projection",
    "home_proj_spread",
    "predicted_spread",
]

spread_proj_col = next((c for c in spread_proj_candidates if c in full_df.columns), None)

# If that column is missing, build one from projected scores
if spread_proj_col is None:
    if {"away_projected", "home_projected"}.issubset(full_df.columns):
        full_df["projected_spread_auto"] = (
            pd.to_numeric(full_df["away_projected"], errors="coerce")
            - pd.to_numeric(full_df["home_projected"], errors="coerce")
        )
        spread_proj_col = "projected_spread_auto"
    elif {"away_proj", "home_proj"}.issubset(full_df.columns):
        full_df["projected_spread_auto"] = (
            pd.to_numeric(full_df["away_proj"], errors="coerce")
            - pd.to_numeric(full_df["home_proj"], errors="coerce")
        )
        spread_proj_col = "projected_spread_auto"
    else:
        raise KeyError(
            "Could not find a spread projection column. Need one of: "
            "projected_spread, proj_spread, model_spread, spread_projection, "
            "home_proj_spread, predicted_spread, or home/away projected score columns."
        )

# Make sure the key columns for the spread analysis exist
required_cols = [
    "open_spread_home",
    "close_spread_home",
    spread_proj_col,
    "spread_conf",
    "spread_play",
    "spread_result"
]
missing = [c for c in required_cols if c not in full_df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

# Convert analysis columns to the right types
full_df["open_spread_home"] = pd.to_numeric(full_df["open_spread_home"], errors="coerce")
full_df["close_spread_home"] = pd.to_numeric(full_df["close_spread_home"], errors="coerce")
full_df[spread_proj_col] = pd.to_numeric(full_df[spread_proj_col], errors="coerce")
full_df["spread_conf"] = pd.to_numeric(full_df["spread_conf"], errors="coerce")
full_df["spread_result"] = pd.to_numeric(full_df["spread_result"], errors="coerce")
full_df["spread_play"] = full_df["spread_play"].astype(str).str.upper().str.strip()

# Small helper to build a vertical group of checkboxes
def make_checkbox_group(options, selected=None, box_width="220px"):
    if selected is None:
        selected = options
    boxes = []
    for opt in options:
        cb = widgets.Checkbox(
            value=(opt in selected),
            description=opt,
            indent=False,
            layout=widgets.Layout(width=box_width)
        )
        boxes.append(cb)
    return boxes, widgets.VBox(boxes)

# Return only the checkbox values that are selected
def get_checked_values(boxes):
    return [b.description for b in boxes if b.value]

# Simple card-style container used in the layout
def card(title, body, width="250px"):
    return widgets.VBox(
        [widgets.HTML(f"<b>{title}</b>"), body],
        layout=widgets.Layout(
            width=width,
            border="1px solid #d9d9d9",
            padding="10px",
            margin="4px"
        )
    )

# Reset the button any time a filter changes
def reset_run_button(change=None):
    submit_btn.button_style = "primary"
    submit_btn.description = "Run Scatter"
    submit_btn.icon = "bar-chart"

# Hide the open spread value box when the dropdown is set to ALL
def update_open_spread_state(change=None):
    is_all = (open_spread_op_dd.value == "ALL")
    open_spread_val.disabled = is_all
    open_spread_val.layout.display = "none" if is_all else "block"
    reset_run_button()

# Measure how much the closing line moved toward the model from the open
def compute_pct_toward_model_spread(df, proj_col):
    d = df.copy()

    d["open_gap"] = (d[proj_col] - d["open_spread_home"]).abs()
    d["close_gap"] = (d[proj_col] - d["close_spread_home"]).abs()

    d = d[d["open_gap"].notna() & d["close_gap"].notna()].copy()
    d = d[d["open_gap"] > 0].copy()

    # Ignore tiny gaps so the percentage move is not misleading
    d = d[d["open_gap"] >= 1.0].copy()

    d["pct_toward_model"] = ((d["open_gap"] - d["close_gap"]) / d["open_gap"]) * 100.0
    d["pct_toward_model"] = d["pct_toward_model"].clip(-200, 200)

    return d

# Render the summary table with a cleaner centered HTML layout
def render_centered_table(df_show):
    if df_show.empty:
        display(HTML("<div style='text-align:center; margin-top:25px; font-size:16px; font-weight:600;'>No table rows to display.</div>"))
        return

    cols = list(df_show.columns)
    metric_col = cols[1]

    temp = df_show[df_show["Confidence Score"] != "TOTAL"].copy()

    try:
        metric_vals = (
            temp[metric_col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float)
        )
        best_idx = metric_vals.idxmax()
    except Exception:
        best_idx = None

    html = """
    <div style="display:flex; justify-content:center; margin-top:34px; margin-bottom:10px; width:100%;">
        <table style="
            min-width:950px;
            border-collapse:separate;
            border-spacing:0;
            font-family:Arial, Helvetica, sans-serif;
            background:white;
            border:1px solid #DADADA;
            border-radius:12px;
            overflow:hidden;
            box-shadow:0 4px 14px rgba(0,0,0,0.08);
            text-align:center;
        ">
            <thead>
                <tr>
    """

    for c in cols:
        html += f"""
            <th style="background:#1F1F1F; color:white; padding:15px 20px; font-size:17px; font-weight:700; border-bottom:2px solid #111;">{c}</th>
        """

    html += """
                </tr>
            </thead>
            <tbody>
    """

    for i, (idx, row) in enumerate(df_show.iterrows()):
        if row["Confidence Score"] == "TOTAL":
            bg = "#E8E8E8"
            font_weight = "700"
        elif idx == best_idx:
            bg = "#FFF3A3"
            font_weight = "700"
        else:
            bg = "#FAFAFA" if i % 2 == 0 else "white"
            font_weight = "600"

        html += f"""<tr style="background:{bg}; font-weight:{font_weight};">"""
        for c in cols:
            html += f"""
                <td style="padding:15px 20px; font-size:17px;">{row[c]}</td>
            """
        html += "</tr>"

    html += """
            </tbody>
        </table>
    </div>
    """
    display(HTML(html))

# Main function that filters the data, builds the summary, and draws the chart
def run_scatter(
    start_year,
    end_year,
    start_month,
    end_month,
    conf_team_a,
    conf_team_b,
    pace_team_a,
    pace_team_b,
    bet_type,
    spread_filter_op,
    spread_filter_value,
    min_conf,
    y_axis_metric
):
    if start_year > end_year:
        print("Start season must be <= end season.")
        return False

    if month_to_num[start_month] > month_to_num[end_month]:
        print("Start month must be <= end month.")
        return False

    if len(conf_team_a) == 0 or len(conf_team_b) == 0:
        print("Pick at least one conference group for Team A and Team B.")
        return False

    if len(pace_team_a) == 0 or len(pace_team_b) == 0:
        print("Pick at least one pace group for Team A and Team B.")
        return False

    df = full_df[
        (full_df["year"] >= start_year) &
        (full_df["year"] <= end_year) &
        (full_df["MONTH_NUM"] >= month_to_num[start_month]) &
        (full_df["MONTH_NUM"] <= month_to_num[end_month])
    ].copy()

    # Keep matchups that fit the selected conference group combinations
    conf_mask = (
        (df["home_conf_label"].isin(conf_team_a) & df["away_conf_label"].isin(conf_team_b)) |
        (df["home_conf_label"].isin(conf_team_b) & df["away_conf_label"].isin(conf_team_a))
    )
    df = df[conf_mask].copy()

    # Keep matchups that fit the selected pace group combinations
    pace_mask = (
        (df["home_pace_group"].isin(pace_team_a) & df["away_pace_group"].isin(pace_team_b)) |
        (df["home_pace_group"].isin(pace_team_b) & df["away_pace_group"].isin(pace_team_a))
    )
    df = df[pace_mask].copy()

    # Drop rows missing any of the core fields needed for the scatter
    df = df[
        df["spread_play"].notna() &
        df["spread_result"].isin([1, -1]) &
        df["spread_conf"].notna() &
        df["open_spread_home"].notna() &
        df["close_spread_home"].notna() &
        df[spread_proj_col].notna()
    ].copy()

    if bet_type != "ALL":
        df = df[df["spread_play"] == bet_type].copy()

    if spread_filter_op != "ALL":
        if spread_filter_op == ">=":
            df = df[df["open_spread_home"] >= spread_filter_value].copy()
        elif spread_filter_op == "<=":
            df = df[df["open_spread_home"] <= spread_filter_value].copy()

    if min_conf > 0:
        df = df[df["spread_conf"] >= min_conf].copy()

    if df.empty:
        print("No rows for this filter.")
        return False

    # Trim off the top 5% of confidence values
    cutoff = df["spread_conf"].quantile(0.95)
    df = df[df["spread_conf"] <= cutoff].copy()

    if df.empty:
        print("No rows remain after top-5% confidence removal.")
        return False

    df = compute_pct_toward_model_spread(df, spread_proj_col)

    if df.empty:
        print("No rows remain after % toward model calculation.")
        return False

    # Build year-by-year points for each confidence threshold
    thresholds = sorted(pd.Series(df["spread_conf"].dropna().unique()).astype(float).astype(int).unique())
    thresholds = [t for t in thresholds if 1 <= t <= 7]

    if not thresholds:
        print("No usable spread_conf thresholds found in the 1 to 7 range.")
        return False

    rows = []
    years_in_range = [y for y in years if start_year <= y <= end_year]

    for yr in years_in_range:
        yr_df = df[df["year"] == yr].copy()

        for t in thresholds:
            subset = yr_df[yr_df["spread_conf"] >= t].copy()
            if subset.empty:
                continue

            wins = int((subset["spread_result"] == 1).sum())
            losses = int((subset["spread_result"] == -1).sum())
            sample = len(subset)
            win_pct = (wins / sample) * 100.0 if sample > 0 else np.nan
            line_move = subset["pct_toward_model"].mean()
            net_units = wins - (losses * 1.1)

            if y_axis_metric == "Line % Movement":
                y_value = line_move
            elif y_axis_metric == "Winning %":
                y_value = win_pct
            else:
                y_value = net_units

            rows.append({
                "Confidence Score": t,
                "Year": yr,
                "Y Value": y_value,
                "Line % Movement": line_move,
                "Wins": wins,
                "Losses": losses,
                "Win %": win_pct,
                "Net Units": net_units,
                "Total Sample Size": sample
            })

    scatter_df = pd.DataFrame(rows)

    if scatter_df.empty:
        print("No valid results after filtering.")
        return False

    # Build the summary table directly from the filtered game rows
    table_rows = []

    for t in thresholds:
        subset_all = df[df["spread_conf"] >= t].copy()
        if subset_all.empty:
            continue

        total_wins = int((subset_all["spread_result"] == 1).sum())
        total_losses = int((subset_all["spread_result"] == -1).sum())
        total_sample = len(subset_all)
        total_net_units = total_wins - (total_losses * 1.1)
        total_win_pct = (total_wins / total_sample) * 100.0 if total_sample > 0 else np.nan
        avg_line_move = subset_all["pct_toward_model"].mean()

        if y_axis_metric == "Line % Movement":
            row = {
                "Confidence Score": f"{int(t)}",
                "Line % Movement": "" if pd.isna(avg_line_move) else f"{avg_line_move:.1f}%",
                "Wins": f"{total_wins:,}",
                "Losses": f"{total_losses:,}",
                "Win %": "" if pd.isna(total_win_pct) else f"{total_win_pct:.1f}%",
                "Net Units": f"{total_net_units:.1f}",
                "Total Sample Size": f"{total_sample:,}"
            }
        elif y_axis_metric == "Winning %":
            row = {
                "Confidence Score": f"{int(t)}",
                "Winning %": "" if pd.isna(total_win_pct) else f"{total_win_pct:.1f}%",
                "Wins": f"{total_wins:,}",
                "Losses": f"{total_losses:,}",
                "Net Units": f"{total_net_units:.1f}",
                "Total Sample Size": f"{total_sample:,}"
            }
        else:
            row = {
                "Confidence Score": f"{int(t)}",
                "Net Units": f"{total_net_units:.1f}",
                "Wins": f"{total_wins:,}",
                "Losses": f"{total_losses:,}",
                "Win %": "" if pd.isna(total_win_pct) else f"{total_win_pct:.1f}%",
                "Total Sample Size": f"{total_sample:,}"
            }

        table_rows.append(row)

    table_df = pd.DataFrame(table_rows)

    # Add one overall total row using the filtered base dataframe once
    total_wins = int((df["spread_result"] == 1).sum())
    total_losses = int((df["spread_result"] == -1).sum())
    total_sample = len(df)
    total_net_units = total_wins - (total_losses * 1.1)
    total_win_pct = (total_wins / total_sample) * 100.0 if total_sample > 0 else np.nan
    avg_line_move_total = df["pct_toward_model"].mean()

    if y_axis_metric == "Line % Movement":
        totals_row = {
            "Confidence Score": "TOTAL",
            "Line % Movement": f"{avg_line_move_total:.1f}%",
            "Wins": f"{total_wins:,}",
            "Losses": f"{total_losses:,}",
            "Win %": f"{total_win_pct:.1f}%",
            "Net Units": f"{total_net_units:.1f}",
            "Total Sample Size": f"{total_sample:,}"
        }
    elif y_axis_metric == "Winning %":
        totals_row = {
            "Confidence Score": "TOTAL",
            "Winning %": f"{total_win_pct:.1f}%",
            "Wins": f"{total_wins:,}",
            "Losses": f"{total_losses:,}",
            "Net Units": f"{total_net_units:.1f}",
            "Total Sample Size": f"{total_sample:,}"
        }
    else:
        totals_row = {
            "Confidence Score": "TOTAL",
            "Net Units": f"{total_net_units:.1f}",
            "Wins": f"{total_wins:,}",
            "Losses": f"{total_losses:,}",
            "Win %": f"{total_win_pct:.1f}%",
            "Total Sample Size": f"{total_sample:,}"
        }

    table_df = pd.concat([table_df, pd.DataFrame([totals_row])], ignore_index=True)

    # Build the all-years line directly from the filtered rows
    all_years_rows = []
    for t in thresholds:
        subset_all = df[df["spread_conf"] >= t].copy()
        if subset_all.empty:
            continue

        total_sample_t = len(subset_all)
        total_wins_t = int((subset_all["spread_result"] == 1).sum())
        total_losses_t = int((subset_all["spread_result"] == -1).sum())
        total_net_units_t = total_wins_t - (total_losses_t * 1.1)

        if y_axis_metric == "Line % Movement":
            all_val = subset_all["pct_toward_model"].mean()
        elif y_axis_metric == "Winning %":
            all_val = (total_wins_t / total_sample_t) * 100.0 if total_sample_t > 0 else np.nan
        else:
            all_val = total_net_units_t

        all_years_rows.append({
            "Confidence Score": t,
            "Y Value": all_val,
            "Total Sample Size": total_sample_t
        })

    all_years_df = pd.DataFrame(all_years_rows).sort_values("Confidence Score").reset_index(drop=True)

    # Draw the scatter and line chart
    year_colors = {
        2022: "#1f77b4",
        2023: "#d62728",
        2024: "#2ca02c",
        2025: "#9467bd",
        2026: "#ff7f0e",
    }

    fig, ax = plt.subplots(figsize=(13.5, 8.8))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("#FCFCFC")

    for yr in years_in_range:
        yr_plot = scatter_df[scatter_df["Year"] == yr].sort_values("Confidence Score")
        if yr_plot.empty:
            continue

        ax.plot(
            yr_plot["Confidence Score"],
            yr_plot["Y Value"],
            linewidth=1.8,
            alpha=0.32,
            color=year_colors.get(yr, "gray"),
            zorder=2
        )

        ax.scatter(
            yr_plot["Confidence Score"],
            yr_plot["Y Value"],
            s=135,
            color=year_colors.get(yr, "gray"),
            edgecolor="#2D2D2D",
            linewidth=1.0,
            alpha=0.92,
            label=str(yr),
            zorder=3
        )

    # Add the combined all-years series on top
    ax.plot(
        all_years_df["Confidence Score"],
        all_years_df["Y Value"],
        color="black",
        linewidth=3.4,
        alpha=0.65,
        zorder=4.2,
        label="All Years"
    )

    ax.scatter(
        all_years_df["Confidence Score"],
        all_years_df["Y Value"],
        s=180,
        color="black",
        edgecolor="white",
        linewidth=1.1,
        alpha=0.95,
        zorder=4.4
    )

    if y_axis_metric == "Line % Movement":
        ax.axhline(
            0,
            linestyle="--",
            linewidth=1.6,
            color="black",
            alpha=0.70,
            zorder=1
        )

    if y_axis_metric == "Winning %":
        ax.axhline(
            52.39,
            linestyle=":",
            linewidth=2.0,
            color="black",
            alpha=0.65,
            zorder=1.5,
            label="Breakeven 52.39%"
        )

    open_spread_text = "ALL" if spread_filter_op == "ALL" else f"{spread_filter_op} {spread_filter_value:g}"

    if y_axis_metric == "Line % Movement":
        chart_title_metric = "Avg % Toward Model"
    elif y_axis_metric == "Winning %":
        chart_title_metric = "Winning %"
    else:
        chart_title_metric = "Net Units"

    min_conf_text = "ALL (0)" if min_conf == 0 else f"{min_conf}+"

    ax.set_title(
        f"Spreads {chart_title_metric} by Confidence Score and Year\n"
        f"Seasons: {start_year}–{end_year} | Months: {start_month} to {end_month} | "
        f"Bet Type: {bet_type} | Open Spread: {open_spread_text} | Min Included Confidence: {min_conf_text}",
        fontsize=15,
        fontweight="bold",
        pad=18
    )

    ax.set_xlabel("Confidence Score Threshold", fontsize=16, fontweight="bold", labelpad=10)
    ax.set_ylabel(chart_title_metric, fontsize=16, fontweight="bold", labelpad=10)

    conf_ticks = [t for t in sorted(scatter_df["Confidence Score"].unique()) if 1 <= t <= 7]
    ax.set_xticks(conf_ticks)
    ax.set_xticklabels([str(int(x)) for x in conf_ticks], fontsize=12, fontweight="bold")
    ax.set_xlim(0.8, 7.2)

    ax.tick_params(axis="y", labelsize=12)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")

    if y_axis_metric == "Winning %":
        y_min = min(scatter_df["Y Value"].min(), 52.39) - 2
        y_max = max(scatter_df["Y Value"].max(), 52.39) + 2
        ax.set_ylim(y_min, y_max)

    ax.grid(True, linestyle=":", linewidth=0.9, alpha=0.70)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#999999")
    ax.spines["bottom"].set_color("#999999")

    ax.legend(title="Year / Series", loc="best", fontsize=11, title_fontsize=12, frameon=True)

    plt.tight_layout()
    plt.show()

    # Show the table underneath the chart
    display(HTML("<div style='height:30px;'></div>"))
    render_centered_table(table_df)

    return True

# Widget styling and sizing
style = {"description_width": "145px"}

med_dd = widgets.Layout(width="260px")
month_dd = widgets.Layout(width="220px")
wide_dd = widgets.Layout(width="320px")
value_box_layout = widgets.Layout(width="120px")

# Choose what goes on the y-axis
y_axis_dd = widgets.Dropdown(
    options=["Line % Movement", "Winning %", "Net Units"],
    value="Line % Movement",
    description="Y Axis",
    style=style,
    layout=med_dd
)

# Build bet type choices from the data
bet_type_options = ["ALL"] + sorted([
    x for x in full_df["spread_play"].dropna().astype(str).str.upper().str.strip().unique()
    if x not in {"", "NAN"}
])

bet_type_dd = widgets.Dropdown(
    options=bet_type_options,
    value="ALL",
    description="Bet Type",
    style=style,
    layout=med_dd
)

# Confidence threshold dropdown
spread_conf_vals = sorted(pd.Series(full_df["spread_conf"].dropna().unique()).astype(float).astype(int).unique())
spread_conf_options = [("ALL (0)", 0)] + [(str(int(x)), int(x)) for x in spread_conf_vals if 1 <= x <= 7]

conf_min_dd = widgets.Dropdown(
    options=spread_conf_options,
    value=0,
    description="Min Included Conf",
    style=style,
    layout=wide_dd
)

# Season and month filters
start_year_dd = widgets.Dropdown(
    options=years,
    value=years[0],
    description="Start Season",
    style=style,
    layout=med_dd
)

end_year_dd = widgets.Dropdown(
    options=years,
    value=years[-1],
    description="End Season",
    style=style,
    layout=med_dd
)

start_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[0],
    description="Start Month",
    style=style,
    layout=month_dd
)

end_month_dd = widgets.Dropdown(
    options=month_labels,
    value=month_labels[-1],
    description="End Month",
    style=style,
    layout=month_dd
)

# Open spread filter
open_spread_op_dd = widgets.Dropdown(
    options=["ALL", ">=", "<="],
    value="ALL",
    description="Open Spread",
    style=style,
    layout=med_dd
)

open_spread_val = widgets.FloatText(
    value=0.0,
    description="",
    layout=value_box_layout
)

# Main run button
submit_btn = widgets.Button(
    description="Run Scatter",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="170px", height="40px")
)

# Checkbox groups for conference and pace filters
conf_a_boxes, conf_a_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
conf_b_boxes, conf_b_ui = make_checkbox_group(
    valid_conf_options,
    selected=valid_conf_options,
    box_width="220px"
)
pace_a_boxes, pace_a_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)
pace_b_boxes, pace_b_ui = make_checkbox_group(
    pace_options,
    selected=pace_options,
    box_width="180px"
)

output = widgets.Output()

# Hook up widget events
open_spread_op_dd.observe(update_open_spread_state, names="value")
update_open_spread_state()

all_reset_widgets = [
    y_axis_dd,
    bet_type_dd,
    conf_min_dd,
    start_year_dd,
    end_year_dd,
    start_month_dd,
    end_month_dd,
    open_spread_op_dd,
    open_spread_val,
]

for w in all_reset_widgets:
    w.observe(reset_run_button, names="value")

for box in conf_a_boxes + conf_b_boxes + pace_a_boxes + pace_b_boxes:
    box.observe(reset_run_button, names="value")

# What happens when the run button is clicked
def on_submit_clicked(_):
    submit_btn.button_style = ""
    submit_btn.description = "Running..."
    submit_btn.icon = "spinner"

    with output:
        output.clear_output(wait=True)

        try:
            ok = run_scatter(
                start_year=start_year_dd.value,
                end_year=end_year_dd.value,
                start_month=start_month_dd.value,
                end_month=end_month_dd.value,
                conf_team_a=get_checked_values(conf_a_boxes),
                conf_team_b=get_checked_values(conf_b_boxes),
                pace_team_a=get_checked_values(pace_a_boxes),
                pace_team_b=get_checked_values(pace_b_boxes),
                bet_type=bet_type_dd.value,
                spread_filter_op=open_spread_op_dd.value,
                spread_filter_value=open_spread_val.value,
                min_conf=conf_min_dd.value,
                y_axis_metric=y_axis_dd.value
            )

            if ok:
                submit_btn.button_style = "success"
                submit_btn.description = "Scatter Ready"
                submit_btn.icon = "check"
            else:
                submit_btn.button_style = "warning"
                submit_btn.description = "No Results"
                submit_btn.icon = "info-circle"

        except Exception:
            import traceback
            print("RUN SCATTER FAILED\n")
            traceback.print_exc()

            submit_btn.button_style = "danger"
            submit_btn.description = "Error"
            submit_btn.icon = "warning"

submit_btn.on_click(on_submit_clicked)

# Arrange the controls across a few rows
controls_row_1 = widgets.HBox(
    [y_axis_dd, bet_type_dd, conf_min_dd, start_year_dd, end_year_dd, start_month_dd, end_month_dd],
    layout=widgets.Layout(
        gap="12px",
        margin="0 0 10px 0",
        flex_flow="row wrap",
        justify_content="flex-start",
        align_items="center"
    )
)

controls_row_2 = widgets.HBox(
    [open_spread_op_dd, open_spread_val],
    layout=widgets.Layout(
        gap="12px",
        margin="0 0 10px 0",
        align_items="center"
    )
)

controls_row_3 = widgets.HBox(
    [submit_btn],
    layout=widgets.Layout(
        justify_content="center",
        margin="6px 0 12px 0"
    )
)

selection_row = widgets.HBox(
    [
        card("Conference Group — Team A", conf_a_ui, "260px"),
        card("Conference Group — Team B", conf_b_ui, "260px"),
        card("Pace Group — Team A", pace_a_ui, "220px"),
        card("Pace Group — Team B", pace_b_ui, "220px"),
    ],
    layout=widgets.Layout(
        justify_content="center",
        align_items="flex-start",
        gap="8px",
        flex_flow="row wrap"
    )
)

# Put the full UI on the page
ui = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 10px 0;'>Spreads by Confidence Score and Year</h3>"),
    controls_row_1,
    controls_row_2,
    controls_row_3,
    selection_row,
    output
])

display(ui)

In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML

ROOT_DIR = Path(r"C:\Users\djbar\popalgo")
years = [2022, 2023, 2024, 2025, 2026]

# Load each season file and stack them into one dataframe
dfs = []
for y in years:
    p = ROOT_DIR / f"season_results_{y}.csv"
    if not p.exists():
        print(f"Missing file: {p}")
        continue
    temp = pd.read_csv(p)
    if "SEASON" not in temp.columns:
        temp["SEASON"] = y
    dfs.append(temp)

if not dfs:
    raise FileNotFoundError("No season_results_YYYY.csv files were found.")

full_df = pd.concat(dfs, ignore_index=True)

if "SEASON" not in full_df.columns:
    raise KeyError("Missing SEASON column.")

# Check that the main fields needed for the totals report exist
required_cols = [
    "SEASON", "totals_conf", "total_result", "total_play",
    "home_conf_grp", "away_conf_grp",
    "home_pace_group", "away_pace_group"
]
missing = [c for c in required_cols if c not in full_df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

full_df["SEASON"] = pd.to_numeric(full_df["SEASON"], errors="coerce")
full_df["totals_conf"] = pd.to_numeric(full_df["totals_conf"], errors="coerce")
full_df["total_result"] = pd.to_numeric(full_df["total_result"], errors="coerce")
full_df["total_play"] = full_df["total_play"].astype(str).str.upper().str.strip()

# Clean the month field if it exists, otherwise build it from a date column
month_labels = ["DEC", "JAN", "FEB"]

if "MONTH" in full_df.columns:
    full_df["MONTH"] = full_df["MONTH"].astype(str).str.upper().str.strip()
    full_df["MONTH"] = full_df["MONTH"].replace({
        "FEB/MAR": "FEB",
        "MARCH": "FEB",
        "MAR": "FEB"
    })
else:
    date_col = next((c for c in ["slate_date", "game_date", "date"] if c in full_df.columns), None)
    if date_col is None:
        raise KeyError("Could not find MONTH column or a usable date column.")
    dt = pd.to_datetime(full_df[date_col], errors="coerce")
    m = dt.dt.month
    full_df["MONTH"] = np.select(
        [m == 12, m == 1, m.isin([2, 3])],
        ["DEC", "JAN", "FEB"],
        default="OTHER"
    )

# Standardize conference group fields
full_df["home_conf_grp"] = full_df["home_conf_grp"].astype(str).str.upper().str.strip()
full_df["away_conf_grp"] = full_df["away_conf_grp"].astype(str).str.upper().str.strip()

# Map conference codes to readable labels
conf_label_map = {
    "P6": "Power 5",
    "HM2": "High Mid-Majors",
    "MM": "Mid-Majors",
    "LM": "Low Majors",
}

valid_conf_options = ["Power 5", "High Mid-Majors", "Mid-Majors", "Low Majors"]

full_df["home_conf_label"] = full_df["home_conf_grp"].map(conf_label_map)
full_df["away_conf_label"] = full_df["away_conf_grp"].map(conf_label_map)

full_df = full_df[
    full_df["home_conf_label"].isin(valid_conf_options) &
    full_df["away_conf_label"].isin(valid_conf_options)
].copy()

# Prebuilt conference matchup buckets for the dropdown
custom_conf_group_options = [
    "",
    "Top 2 vs Top 2",
    "P5 vs P5",
    "Mid 2 vs Mid 2",
    "Top 2 vs Bottom 2",
    "HM2 vs HM2",
    "MM vs MM",
    "LM vs LM",
    "Bottom 2 vs Bottom 2",
    "No P5",
    "No LM",
]

top2_labels = {"Power 5", "High Mid-Majors"}
bottom2_labels = {"Mid-Majors", "Low Majors"}
mid2_labels = {"High Mid-Majors", "Mid-Majors"}
no_p5_labels = {"High Mid-Majors", "Mid-Majors", "Low Majors"}
no_lm_labels = {"Power 5", "High Mid-Majors", "Mid-Majors"}

# Return the mask for the selected conference matchup bucket
def conf_group_matchup_mask(df, selected_option):
    if selected_option == "":
        return pd.Series(True, index=df.index)

    home = df["home_conf_label"]
    away = df["away_conf_label"]

    if selected_option == "Top 2 vs Top 2":
        return home.isin(top2_labels) & away.isin(top2_labels)

    if selected_option == "P5 vs P5":
        return (home == "Power 5") & (away == "Power 5")

    if selected_option == "Mid 2 vs Mid 2":
        return home.isin(mid2_labels) & away.isin(mid2_labels)

    if selected_option == "Top 2 vs Bottom 2":
        return (
            (home.isin(top2_labels) & away.isin(bottom2_labels)) |
            (home.isin(bottom2_labels) & away.isin(top2_labels))
        )

    if selected_option == "HM2 vs HM2":
        return (home == "High Mid-Majors") & (away == "High Mid-Majors")

    if selected_option == "MM vs MM":
        return (home == "Mid-Majors") & (away == "Mid-Majors")

    if selected_option == "LM vs LM":
        return (home == "Low Majors") & (away == "Low Majors")

    if selected_option == "Bottom 2 vs Bottom 2":
        return home.isin(bottom2_labels) & away.isin(bottom2_labels)

    if selected_option == "No P5":
        return home.isin(no_p5_labels) & away.isin(no_p5_labels)

    if selected_option == "No LM":
        return home.isin(no_lm_labels) & away.isin(no_lm_labels)

    return pd.Series(True, index=df.index)

# Standardize the pace fields
full_df["home_pace_group"] = full_df["home_pace_group"].astype(str).str.upper().str.strip()
full_df["away_pace_group"] = full_df["away_pace_group"].astype(str).str.upper().str.strip()

# Convert pace values into cleaner display labels
def normalize_pace_value(x):
    if pd.isna(x):
        return np.nan

    s = str(x).upper().strip()

    mapping = {
        "FAST": "Fast",
        "MID": "Mid",
        "SLOW": "Slow",
    }
    return mapping.get(s, np.nan)

full_df["home_pace_label"] = full_df["home_pace_group"].map(normalize_pace_value)
full_df["away_pace_label"] = full_df["away_pace_group"].map(normalize_pace_value)

# Pace matchup choices for the dropdown
pace_matchup_options = [
    "",
    "Fast vs Fast",
    "Fast vs Mid",
    "Fast vs Slow",
    "Mid vs Mid",
    "Mid vs Slow",
    "Slow vs Slow",
    "No Fast",
    "No Slow",
]

# Return the mask for the selected pace matchup bucket
def pace_matchup_mask(df, selected_option):
    if selected_option == "":
        return pd.Series(True, index=df.index)

    home = df["home_pace_label"]
    away = df["away_pace_label"]

    if selected_option == "Fast vs Fast":
        return (home == "Fast") & (away == "Fast")

    if selected_option == "Fast vs Mid":
        return (
            ((home == "Fast") & (away == "Mid")) |
            ((home == "Mid") & (away == "Fast"))
        )

    if selected_option == "Fast vs Slow":
        return (
            ((home == "Fast") & (away == "Slow")) |
            ((home == "Slow") & (away == "Fast"))
        )

    if selected_option == "Mid vs Mid":
        return (home == "Mid") & (away == "Mid")

    if selected_option == "Mid vs Slow":
        return (
            ((home == "Mid") & (away == "Slow")) |
            ((home == "Slow") & (away == "Mid"))
        )

    if selected_option == "Slow vs Slow":
        return (home == "Slow") & (away == "Slow")

    if selected_option == "No Fast":
        return home.isin(["Mid", "Slow"]) & away.isin(["Mid", "Slow"])

    if selected_option == "No Slow":
        return home.isin(["Fast", "Mid"]) & away.isin(["Fast", "Mid"])

    return pd.Series(True, index=df.index)

# Final cleanup before running the report
full_df = full_df[full_df["MONTH"].isin(month_labels)].copy()
full_df = full_df[full_df["SEASON"].isin(years)].copy()
full_df = full_df[full_df["total_result"].isin([1, -1])].copy()
full_df = full_df[
    full_df["home_pace_label"].notna() &
    full_df["away_pace_label"].notna()
].copy()
full_df = full_df.reset_index(drop=True)
full_df["_row_id"] = np.arange(len(full_df))

# Build a checkbox group for season selection
def make_checkbox_group(options, selected=None, width="110px"):
    if selected is None:
        selected = options

    boxes = []
    for opt in options:
        cb = widgets.Checkbox(
            value=(opt in selected),
            description=str(opt),
            indent=False,
            layout=widgets.Layout(width=width)
        )
        boxes.append(cb)

    return boxes, widgets.HBox(
        boxes,
        layout=widgets.Layout(
            gap="12px",
            flex_flow="row wrap",
            align_items="center"
        )
    )

# Return the checked season values as integers
def get_checked_values(boxes):
    vals = []
    for b in boxes:
        if b.value:
            try:
                vals.append(int(b.description))
            except:
                pass
    return vals

# Reset the run button whenever something changes
def reset_run_button(change=None):
    submit_btn.button_style = "primary"
    submit_btn.description = "Run Report"
    submit_btn.icon = "table"

# Summarize a filtered subset into wins, losses, win rate, and net units
def summarize_subset(subset):
    subset = subset[subset["total_result"].isin([1, -1])].copy()

    wins = int((subset["total_result"] == 1).sum())
    losses = int((subset["total_result"] == -1).sum())
    n = wins + losses

    if n == 0:
        win_pct = np.nan
        net_units = np.nan
    else:
        win_pct = wins / n
        net_units = wins - (losses * 1.1)

    return {
        "Wins": wins,
        "Losses": losses,
        "Win %": win_pct,
        "Net Units": net_units,
        "N": n
    }

# Translate the month dropdown into the actual month list to filter on
def month_value_to_list(month_value):
    month_map = {
        "DEC": ["DEC"],
        "JAN": ["JAN"],
        "FEB": ["FEB"],
        "DEC/JAN": ["DEC", "JAN"],
        "JAN/FEB": ["JAN", "FEB"],
    }
    return month_map.get(month_value, [])

# Apply one metric's filters and return the filtered dataframe
def apply_metric(df, bet_type, min_conf, month_value, conf_group_matchup_value, pace_matchup_value):
    d = df.copy()

    if bet_type not in ["", "ALL"]:
        d = d[d["total_play"] == bet_type].copy()

    if min_conf != "":
        d = d[d["totals_conf"] >= float(min_conf)].copy()

    if month_value != "":
        months = month_value_to_list(month_value)
        d = d[d["MONTH"].isin(months)].copy()

    if conf_group_matchup_value != "":
        d = d[conf_group_matchup_mask(d, conf_group_matchup_value)].copy()

    if pace_matchup_value != "":
        d = d[pace_matchup_mask(d, pace_matchup_value)].copy()

    return d

# Style the table output
def style_single_table(show_df):
    def color_units(val):
        try:
            v = float(val)
        except:
            return ""
        if v > 0:
            return "color:#0B7D3E; font-weight:700;"
        if v < 0:
            return "color:#B42318; font-weight:700;"
        return ""

    def highlight_rows(row):
        if str(row["Totals"]) == "Summary of All Unique":
            return ["background-color:#E8F5E9; font-weight:800; border-top:4px solid #333;"] * len(row)
        return ["background-color:#FFF8E8; font-weight:700;"] * len(row)

    styled_table = (
        show_df.style
        .hide(axis="index")
        .apply(highlight_rows, axis=1)
        .map(color_units, subset=["Net Units"])
        .set_properties(**{
            "text-align": "center",
            "font-size": "18px",
            "padding": "14px 18px",
            "border": "none"
        })
        .set_table_styles([
            {
                "selector": "table",
                "props": [
                    ("margin-left", "auto"),
                    ("margin-right", "auto"),
                    ("margin-top", "8px"),
                    ("margin-bottom", "28px"),
                    ("min-width", "1320px"),
                    ("border-collapse", "separate"),
                    ("border-spacing", "0"),
                    ("font-family", "Arial, Helvetica, sans-serif"),
                    ("background-color", "white"),
                    ("border", "1px solid #DADADA"),
                    ("border-radius", "10px"),
                    ("overflow", "hidden"),
                    ("box-shadow", "0 4px 14px rgba(0,0,0,0.08)")
                ]
            },
            {
                "selector": "thead th",
                "props": [
                    ("background-color", "#1F1F1F"),
                    ("color", "white"),
                    ("font-size", "18px"),
                    ("font-weight", "700"),
                    ("text-align", "center"),
                    ("padding", "14px 18px"),
                    ("border-bottom", "2px solid #111")
                ]
            },
            {
                "selector": "tbody td",
                "props": [
                    ("text-align", "center"),
                    ("padding", "14px 18px"),
                    ("font-size", "18px")
                ]
            },
            {
                "selector": "tbody td.col0",
                "props": [
                    ("font-weight", "700"),
                    ("text-align", "left"),
                    ("padding-left", "20px")
                ]
            }
        ])
    )
    return styled_table

# Main report function
def run_report(selected_seasons, metric_inputs):
    if not selected_seasons:
        print("Pick at least one SEASON checkbox.")
        return False

    active_metrics = []
    for include_flag, label, bet_type, min_conf, month_value, conf_group_matchup_value, pace_matchup_value in metric_inputs:
        if include_flag:
            active_metrics.append((label, bet_type, min_conf, month_value, conf_group_matchup_value, pace_matchup_value))

    if not active_metrics:
        print("Check at least one Include box.")
        return False

    base_df = full_df[full_df["SEASON"].isin(selected_seasons)].copy()

    if base_df.empty:
        print("No rows after season filters.")
        return False

    display(HTML(
        f"""
        <div style="
            margin: 8px 0 20px 0;
            padding: 12px 16px;
            border: 1px solid #DADADA;
            border-radius: 10px;
            background: #F8F8F8;
            font-family: Arial, Helvetica, sans-serif;
        ">
            <div style="font-size:20px; font-weight:700; margin-bottom:6px;">
                Totals Results
            </div>
            <div style="font-size:15px;">
                <b>Selected Seasons:</b> {", ".join(map(str, [y for y in years if y in selected_seasons]))}
            </div>
            <div style="font-size:14px; margin-top:6px; color:#555;">
                Each metric row is calculated individually. The last row, Summary of All Unique, removes duplicate rows across all included metrics.
            </div>
        </div>
        """
    ))

    result_rows = []
    chart_rows = []
    all_unique_parts = []

    for label, bet_type, min_conf, month_value, conf_group_matchup_value, pace_matchup_value in active_metrics:
        metric_df = apply_metric(
            base_df,
            bet_type=bet_type,
            min_conf=min_conf,
            month_value=month_value,
            conf_group_matchup_value=conf_group_matchup_value,
            pace_matchup_value=pace_matchup_value
        ).copy()

        if not metric_df.empty:
            all_unique_parts.append(metric_df.copy())

        summary = summarize_subset(metric_df)

        bt_label = "ALL" if bet_type == "" else bet_type
        conf_label = "ALL" if min_conf == "" else f"{min_conf}+"
        month_label = "ALL" if month_value == "" else month_value
        conf_group_label = "ALL" if conf_group_matchup_value == "" else conf_group_matchup_value
        pace_label = "ALL" if pace_matchup_value == "" else pace_matchup_value

        totals_label = (
            f"{label} ({bt_label} | Conf {conf_label} | {month_label} | "
            f"Conf Group {conf_group_label} | Pace {pace_label})"
        )

        result_rows.append({
            "Totals": totals_label,
            "Wins": summary["Wins"],
            "Losses": summary["Losses"],
            "Win %": "" if pd.isna(summary["Win %"]) else f'{summary["Win %"]:.1%}',
            "Net Units": "" if pd.isna(summary["Net Units"]) else f'{summary["Net Units"]:.1f}',
            "N": summary["N"]
        })

        chart_rows.append({
            "Metric": label,
            "Net Units": 0 if pd.isna(summary["Net Units"]) else float(summary["Net Units"]),
            "N": summary["N"]
        })

    if all_unique_parts:
        all_unique_df = pd.concat(all_unique_parts, ignore_index=True)
        all_unique_df = all_unique_df.drop_duplicates(subset=["_row_id"]).copy()
    else:
        all_unique_df = base_df.iloc[0:0].copy()

    summary_all_unique = summarize_subset(all_unique_df)
    result_rows.append({
        "Totals": "Summary of All Unique",
        "Wins": summary_all_unique["Wins"],
        "Losses": summary_all_unique["Losses"],
        "Win %": "" if pd.isna(summary_all_unique["Win %"]) else f'{summary_all_unique["Win %"]:.1%}',
        "Net Units": "" if pd.isna(summary_all_unique["Net Units"]) else f'{summary_all_unique["Net Units"]:.1f}',
        "N": summary_all_unique["N"]
    })

    show_df = pd.DataFrame(result_rows, columns=["Totals", "Wins", "Losses", "Win %", "Net Units", "N"])
    display(style_single_table(show_df))

    # Build the net-units bar chart by metric
    if chart_rows:
        chart_df = pd.DataFrame(chart_rows)
        bar_colors = ["#0B7D3E" if x >= 0 else "#B42318" for x in chart_df["Net Units"]]

        fig, ax = plt.subplots(figsize=(10.5, 6))
        bars = ax.bar(chart_df["Metric"], chart_df["Net Units"], width=0.6, color=bar_colors)

        ax.axhline(0, linestyle="--", linewidth=1, color="black")
        ax.set_title("Net Units by Metric", fontsize=16, fontweight="bold", pad=12)
        ax.set_xlabel("Metric", fontsize=12)
        ax.set_ylabel("Net Units", fontsize=12)
        ax.tick_params(axis="x", labelsize=11)
        ax.tick_params(axis="y", labelsize=11)

        max_abs = chart_df["Net Units"].abs().max()
        pad = max(1.0, max_abs * 0.10 if pd.notna(max_abs) else 1.0)

        ymin = min(chart_df["Net Units"].min() - pad * 2.1, -pad * 2.1)
        ymax = max(chart_df["Net Units"].max() + pad, pad)
        ax.set_ylim(ymin, ymax)

        for bar, val in zip(bars, chart_df["Net Units"]):
            x = bar.get_x() + bar.get_width() / 2
            if val >= 0:
                ax.text(x, val + pad * 0.20, f"{val:.1f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
            else:
                ax.text(x, val - pad * 0.20, f"{val:.1f}", ha="center", va="top", fontsize=11, fontweight="bold")

        sample_y = ymin + (ymax - ymin) * 0.05
        for bar, n in zip(bars, chart_df["N"]):
            x = bar.get_x() + bar.get_width() / 2
            ax.text(x, sample_y, f"N={int(n)}", ha="center", va="bottom", fontsize=10, fontweight="bold", color="#444444")

        plt.tight_layout()
        plt.show()

    return True

# Dropdown options used in the widget controls
bet_type_options = ["", "ALL", "OVER", "UNDER"]
conf_score_options = ["", 0, 1, 2, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8]
month_options = ["", "DEC", "JAN", "FEB", "DEC/JAN", "JAN/FEB"]
conf_group_matchup_options = custom_conf_group_options

style = {"description_width": "95px"}
dd_layout = widgets.Layout(width="180px")
wide_dd_layout = widgets.Layout(width="230px")
label_layout = widgets.Layout(width="85px")

season_boxes, season_ui = make_checkbox_group(
    years,
    selected=years,
    width="95px"
)

include_1_cb = widgets.Checkbox(
    value=False,
    description="Include",
    indent=False,
    layout=widgets.Layout(width="90px")
)
row_1_label = widgets.HTML("<b>Metric 1</b>", layout=label_layout)

bet_type_1_dd = widgets.Dropdown(
    options=bet_type_options,
    value="",
    description="Bet Type",
    style=style,
    layout=dd_layout
)

conf_score_1_dd = widgets.Dropdown(
    options=conf_score_options,
    value="",
    description="Conf +",
    style=style,
    layout=dd_layout
)

month_1_dd = widgets.Dropdown(
    options=month_options,
    value="",
    description="Month",
    style=style,
    layout=dd_layout
)

conf_group_1_dd = widgets.Dropdown(
    options=conf_group_matchup_options,
    value="",
    description="Conf Grp",
    style=style,
    layout=wide_dd_layout
)

pace_1_dd = widgets.Dropdown(
    options=pace_matchup_options,
    value="",
    description="Pace",
    style=style,
    layout=wide_dd_layout
)

include_2_cb = widgets.Checkbox(
    value=False,
    description="Include",
    indent=False,
    layout=widgets.Layout(width="90px")
)
row_2_label = widgets.HTML("<b>Metric 2</b>", layout=label_layout)

bet_type_2_dd = widgets.Dropdown(
    options=bet_type_options,
    value="",
    description="Bet Type",
    style=style,
    layout=dd_layout
)

conf_score_2_dd = widgets.Dropdown(
    options=conf_score_options,
    value="",
    description="Conf +",
    style=style,
    layout=dd_layout
)

month_2_dd = widgets.Dropdown(
    options=month_options,
    value="",
    description="Month",
    style=style,
    layout=dd_layout
)

conf_group_2_dd = widgets.Dropdown(
    options=conf_group_matchup_options,
    value="",
    description="Conf Grp",
    style=style,
    layout=wide_dd_layout
)

pace_2_dd = widgets.Dropdown(
    options=pace_matchup_options,
    value="",
    description="Pace",
    style=style,
    layout=wide_dd_layout
)

include_3_cb = widgets.Checkbox(
    value=False,
    description="Include",
    indent=False,
    layout=widgets.Layout(width="90px")
)
row_3_label = widgets.HTML("<b>Metric 3</b>", layout=label_layout)

bet_type_3_dd = widgets.Dropdown(
    options=bet_type_options,
    value="",
    description="Bet Type",
    style=style,
    layout=dd_layout
)

conf_score_3_dd = widgets.Dropdown(
    options=conf_score_options,
    value="",
    description="Conf +",
    style=style,
    layout=dd_layout
)

month_3_dd = widgets.Dropdown(
    options=month_options,
    value="",
    description="Month",
    style=style,
    layout=dd_layout
)

conf_group_3_dd = widgets.Dropdown(
    options=conf_group_matchup_options,
    value="",
    description="Conf Grp",
    style=style,
    layout=wide_dd_layout
)

pace_3_dd = widgets.Dropdown(
    options=pace_matchup_options,
    value="",
    description="Pace",
    style=style,
    layout=wide_dd_layout
)

submit_btn = widgets.Button(
    description="Run Report",
    button_style="primary",
    icon="table",
    layout=widgets.Layout(width="170px", height="42px")
)

output = widgets.Output()

# Reset the button whenever a control changes
for w in [
    include_1_cb, bet_type_1_dd, conf_score_1_dd, month_1_dd, conf_group_1_dd, pace_1_dd,
    include_2_cb, bet_type_2_dd, conf_score_2_dd, month_2_dd, conf_group_2_dd, pace_2_dd,
    include_3_cb, bet_type_3_dd, conf_score_3_dd, month_3_dd, conf_group_3_dd, pace_3_dd
]:
    w.observe(reset_run_button, names="value")

for box in season_boxes:
    box.observe(reset_run_button, names="value")

# Run the report when the button is clicked
def on_submit_clicked(_):
    submit_btn.button_style = ""
    submit_btn.description = "Running..."
    submit_btn.icon = "spinner"

    with output:
        output.clear_output(wait=True)
        try:
            ok = run_report(
                selected_seasons=get_checked_values(season_boxes),
                metric_inputs=[
                    (include_1_cb.value, "Metric 1", bet_type_1_dd.value, conf_score_1_dd.value, month_1_dd.value, conf_group_1_dd.value, pace_1_dd.value),
                    (include_2_cb.value, "Metric 2", bet_type_2_dd.value, conf_score_2_dd.value, month_2_dd.value, conf_group_2_dd.value, pace_2_dd.value),
                    (include_3_cb.value, "Metric 3", bet_type_3_dd.value, conf_score_3_dd.value, month_3_dd.value, conf_group_3_dd.value, pace_3_dd.value),
                ]
            )

            if ok:
                submit_btn.button_style = "success"
                submit_btn.description = "Report Ready"
                submit_btn.icon = "check"
            else:
                submit_btn.button_style = "warning"
                submit_btn.description = "No Results"
                submit_btn.icon = "info-circle"

        except Exception:
            import traceback
            print("RUN REPORT FAILED\n")
            traceback.print_exc()
            submit_btn.button_style = "danger"
            submit_btn.description = "Error"
            submit_btn.icon = "warning"

submit_btn.on_click(on_submit_clicked)

# Arrange the metric rows and season selector
metric_row_1 = widgets.HBox(
    [row_1_label, include_1_cb, bet_type_1_dd, conf_score_1_dd, month_1_dd, conf_group_1_dd, pace_1_dd],
    layout=widgets.Layout(gap="12px", margin="0 0 8px 0", flex_flow="row wrap", align_items="center")
)

metric_row_2 = widgets.HBox(
    [row_2_label, include_2_cb, bet_type_2_dd, conf_score_2_dd, month_2_dd, conf_group_2_dd, pace_2_dd],
    layout=widgets.Layout(gap="12px", margin="0 0 8px 0", flex_flow="row wrap", align_items="center")
)

metric_row_3 = widgets.HBox(
    [row_3_label, include_3_cb, bet_type_3_dd, conf_score_3_dd, month_3_dd, conf_group_3_dd, pace_3_dd],
    layout=widgets.Layout(gap="12px", margin="0 0 8px 0", flex_flow="row wrap", align_items="center")
)

season_row = widgets.VBox(
    [
        widgets.HTML("<b>SEASON</b>"),
        season_ui
    ],
    layout=widgets.Layout(margin="0 0 12px 0")
)

# Put the full UI on the page
ui = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 10px 0;'>Totals Results</h3>"),
    metric_row_1,
    metric_row_2,
    metric_row_3,
    widgets.HBox([submit_btn], layout=widgets.Layout(margin="6px 0 12px 0")),
    season_row,
    output
])

display(ui)

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML

ROOT_DIR = Path(r"C:\Users\djbar\popalgo")
years = [2022, 2023, 2024, 2025, 2026]

# Load each season file and stack them into one dataframe
dfs = []
for y in years:
    p = ROOT_DIR / f"season_results_{y}.csv"
    if not p.exists():
        print(f"Missing file: {p}")
        continue
    temp = pd.read_csv(p)
    if "SEASON" not in temp.columns:
        temp["SEASON"] = y
    dfs.append(temp)

if not dfs:
    raise FileNotFoundError("No season_results_YYYY.csv files were found.")

full_df = pd.concat(dfs, ignore_index=True)

if "SEASON" not in full_df.columns:
    raise KeyError("Missing SEASON column.")

# Check that the main fields needed for the spreads report are present
required_cols = [
    "SEASON", "spread_conf", "spread_result", "spread_play",
    "home_conf_grp", "away_conf_grp",
    "home_pace_group", "away_pace_group"
]
missing = [c for c in required_cols if c not in full_df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

full_df["SEASON"] = pd.to_numeric(full_df["SEASON"], errors="coerce")
full_df["spread_conf"] = pd.to_numeric(full_df["spread_conf"], errors="coerce")
full_df["spread_result"] = pd.to_numeric(full_df["spread_result"], errors="coerce")
full_df["spread_play"] = full_df["spread_play"].astype(str).str.upper().str.strip()

# Clean the month field if it exists, otherwise build it from a date column
month_labels = ["DEC", "JAN", "FEB"]

if "MONTH" in full_df.columns:
    full_df["MONTH"] = full_df["MONTH"].astype(str).str.upper().str.strip()
    full_df["MONTH"] = full_df["MONTH"].replace({
        "FEB/MAR": "FEB",
        "MARCH": "FEB",
        "MAR": "FEB"
    })
else:
    date_col = next((c for c in ["slate_date", "game_date", "date"] if c in full_df.columns), None)
    if date_col is None:
        raise KeyError("Could not find MONTH column or a usable date column.")
    dt = pd.to_datetime(full_df[date_col], errors="coerce")
    m = dt.dt.month
    full_df["MONTH"] = np.select(
        [m == 12, m == 1, m.isin([2, 3])],
        ["DEC", "JAN", "FEB"],
        default="OTHER"
    )

# Standardize conference group fields
full_df["home_conf_grp"] = full_df["home_conf_grp"].astype(str).str.upper().str.strip()
full_df["away_conf_grp"] = full_df["away_conf_grp"].astype(str).str.upper().str.strip()

# Map conference codes to readable labels
conf_label_map = {
    "P6": "Power 5",
    "HM2": "High Mid-Majors",
    "MM": "Mid-Majors",
    "LM": "Low Majors",
}

valid_conf_options = ["Power 5", "High Mid-Majors", "Mid-Majors", "Low Majors"]

full_df["home_conf_label"] = full_df["home_conf_grp"].map(conf_label_map)
full_df["away_conf_label"] = full_df["away_conf_grp"].map(conf_label_map)

full_df = full_df[
    full_df["home_conf_label"].isin(valid_conf_options) &
    full_df["away_conf_label"].isin(valid_conf_options)
].copy()

# Prebuilt conference matchup buckets for the dropdown
custom_conf_group_options = [
    "",
    "P5 vs P5",
    "HM2 & MM vs HM2 & MM",
    "Top 2 vs Bottom 2",
    "HM2 vs HM2",
    "MM vs MM",
    "LM vs LM",
    "Bottom 2 vs Bottom 2",
    "No P5",
    "No LM",
]

top2_labels = {"Power 5", "High Mid-Majors"}
bottom2_labels = {"Mid-Majors", "Low Majors"}
hm2_mm_labels = {"High Mid-Majors", "Mid-Majors"}
no_p5_labels = {"High Mid-Majors", "Mid-Majors", "Low Majors"}
no_lm_labels = {"Power 5", "High Mid-Majors", "Mid-Majors"}

# Return the mask for the selected conference matchup bucket
def conf_group_matchup_mask(df, selected_option):
    if selected_option == "":
        return pd.Series(True, index=df.index)

    home = df["home_conf_label"]
    away = df["away_conf_label"]

    if selected_option == "P5 vs P5":
        return (home == "Power 5") & (away == "Power 5")

    if selected_option == "HM2 & MM vs HM2 & MM":
        return home.isin(hm2_mm_labels) & away.isin(hm2_mm_labels)

    if selected_option == "Top 2 vs Bottom 2":
        return (
            (home.isin(top2_labels) & away.isin(bottom2_labels)) |
            (home.isin(bottom2_labels) & away.isin(top2_labels))
        )

    if selected_option == "HM2 vs HM2":
        return (home == "High Mid-Majors") & (away == "High Mid-Majors")

    if selected_option == "MM vs MM":
        return (home == "Mid-Majors") & (away == "Mid-Majors")

    if selected_option == "LM vs LM":
        return (home == "Low Majors") & (away == "Low Majors")

    if selected_option == "Bottom 2 vs Bottom 2":
        return home.isin(bottom2_labels) & away.isin(bottom2_labels)

    if selected_option == "No P5":
        return home.isin(no_p5_labels) & away.isin(no_p5_labels)

    if selected_option == "No LM":
        return home.isin(no_lm_labels) & away.isin(no_lm_labels)

    return pd.Series(True, index=df.index)

# Standardize the pace fields
full_df["home_pace_group"] = full_df["home_pace_group"].astype(str).str.upper().str.strip()
full_df["away_pace_group"] = full_df["away_pace_group"].astype(str).str.upper().str.strip()

# Convert pace values into cleaner display labels
def normalize_pace_value(x):
    if pd.isna(x):
        return np.nan

    s = str(x).upper().strip()

    mapping = {
        "FAST": "Fast",
        "MID": "Mid",
        "SLOW": "Slow",
    }
    return mapping.get(s, np.nan)

full_df["home_pace_label"] = full_df["home_pace_group"].map(normalize_pace_value)
full_df["away_pace_label"] = full_df["away_pace_group"].map(normalize_pace_value)

# Pace matchup choices for the dropdown
pace_matchup_options = [
    "",
    "Fast vs Fast",
    "Fast vs Mid",
    "Fast vs Slow",
    "Mid vs Mid",
    "Mid vs Slow",
    "Slow vs Slow",
    "No Fast",
    "No Slow",
]

# Return the mask for the selected pace matchup bucket
def pace_matchup_mask(df, selected_option):
    if selected_option == "":
        return pd.Series(True, index=df.index)

    home = df["home_pace_label"]
    away = df["away_pace_label"]

    if selected_option == "Fast vs Fast":
        return (home == "Fast") & (away == "Fast")

    if selected_option == "Fast vs Mid":
        return (
            ((home == "Fast") & (away == "Mid")) |
            ((home == "Mid") & (away == "Fast"))
        )

    if selected_option == "Fast vs Slow":
        return (
            ((home == "Fast") & (away == "Slow")) |
            ((home == "Slow") & (away == "Fast"))
        )

    if selected_option == "Mid vs Mid":
        return (home == "Mid") & (away == "Mid")

    if selected_option == "Mid vs Slow":
        return (
            ((home == "Mid") & (away == "Slow")) |
            ((home == "Slow") & (away == "Mid"))
        )

    if selected_option == "Slow vs Slow":
        return (home == "Slow") & (away == "Slow")

    if selected_option == "No Fast":
        return home.isin(["Mid", "Slow"]) & away.isin(["Mid", "Slow"])

    if selected_option == "No Slow":
        return home.isin(["Fast", "Mid"]) & away.isin(["Fast", "Mid"])

    return pd.Series(True, index=df.index)

# Final cleanup before running the report
full_df = full_df[full_df["MONTH"].isin(month_labels)].copy()
full_df = full_df[full_df["SEASON"].isin(years)].copy()
full_df = full_df[full_df["spread_result"].isin([1, -1])].copy()
full_df = full_df[
    full_df["home_pace_label"].notna() &
    full_df["away_pace_label"].notna()
].copy()
full_df = full_df.reset_index(drop=True)
full_df["_row_id"] = np.arange(len(full_df))

# Build a checkbox group for season selection
def make_checkbox_group(options, selected=None, width="110px"):
    if selected is None:
        selected = options

    boxes = []
    for opt in options:
        cb = widgets.Checkbox(
            value=(opt in selected),
            description=str(opt),
            indent=False,
            layout=widgets.Layout(width=width)
        )
        boxes.append(cb)

    return boxes, widgets.HBox(
        boxes,
        layout=widgets.Layout(
            gap="12px",
            flex_flow="row wrap",
            align_items="center"
        )
    )

# Return the checked season values as integers
def get_checked_values(boxes):
    vals = []
    for b in boxes:
        if b.value:
            try:
                vals.append(int(b.description))
            except:
                pass
    return vals

# Reset the run button whenever something changes
def reset_run_button(change=None):
    submit_btn.button_style = "primary"
    submit_btn.description = "Run Report"
    submit_btn.icon = "table"

# Summarize a filtered subset into wins, losses, win rate, and net units
def summarize_subset(subset):
    subset = subset[subset["spread_result"].isin([1, -1])].copy()

    wins = int((subset["spread_result"] == 1).sum())
    losses = int((subset["spread_result"] == -1).sum())
    n = wins + losses

    if n == 0:
        win_pct = np.nan
        net_units = np.nan
    else:
        win_pct = wins / n
        net_units = wins - (losses * 1.1)

    return {
        "Wins": wins,
        "Losses": losses,
        "Win %": win_pct,
        "Net Units": net_units,
        "N": n
    }

# Translate the month dropdown into the actual month list to filter on
def month_value_to_list(month_value):
    month_map = {
        "DEC": ["DEC"],
        "JAN": ["JAN"],
        "FEB": ["FEB"],
        "DEC/JAN": ["DEC", "JAN"],
        "JAN/FEB": ["JAN", "FEB"],
    }
    return month_map.get(month_value, [])

# Apply one metric's filters and return the filtered dataframe
def apply_metric(df, bet_type, min_conf, month_value, conf_group_matchup_value, pace_matchup_value):
    d = df.copy()

    if bet_type not in ["", "ALL"]:
        d = d[d["spread_play"] == bet_type].copy()

    if min_conf != "":
        d = d[d["spread_conf"] >= float(min_conf)].copy()

    if month_value != "":
        months = month_value_to_list(month_value)
        d = d[d["MONTH"].isin(months)].copy()

    if conf_group_matchup_value != "":
        d = d[conf_group_matchup_mask(d, conf_group_matchup_value)].copy()

    if pace_matchup_value != "":
        d = d[pace_matchup_mask(d, pace_matchup_value)].copy()

    return d

# Style the table output
def style_single_table(show_df):
    def color_units(val):
        try:
            v = float(val)
        except:
            return ""
        if v > 0:
            return "color:#0B7D3E; font-weight:700;"
        if v < 0:
            return "color:#B42318; font-weight:700;"
        return ""

    def highlight_rows(row):
        if str(row["Spreads"]) == "Summary of All Unique":
            return ["background-color:#E8F5E9; font-weight:800; border-top:4px solid #333;"] * len(row)
        return ["background-color:#FFF8E8; font-weight:700;"] * len(row)

    styled_table = (
        show_df.style
        .hide(axis="index")
        .apply(highlight_rows, axis=1)
        .map(color_units, subset=["Net Units"])
        .set_properties(**{
            "text-align": "center",
            "font-size": "18px",
            "padding": "14px 18px",
            "border": "none"
        })
        .set_table_styles([
            {
                "selector": "table",
                "props": [
                    ("margin-left", "auto"),
                    ("margin-right", "auto"),
                    ("margin-top", "8px"),
                    ("margin-bottom", "28px"),
                    ("min-width", "1320px"),
                    ("border-collapse", "separate"),
                    ("border-spacing", "0"),
                    ("font-family", "Arial, Helvetica, sans-serif"),
                    ("background-color", "white"),
                    ("border", "1px solid #DADADA"),
                    ("border-radius", "10px"),
                    ("overflow", "hidden"),
                    ("box-shadow", "0 4px 14px rgba(0,0,0,0.08)")
                ]
            },
            {
                "selector": "thead th",
                "props": [
                    ("background-color", "#1F1F1F"),
                    ("color", "white"),
                    ("font-size", "18px"),
                    ("font-weight", "700"),
                    ("text-align", "center"),
                    ("padding", "14px 18px"),
                    ("border-bottom", "2px solid #111")
                ]
            },
            {
                "selector": "tbody td",
                "props": [
                    ("text-align", "center"),
                    ("padding", "14px 18px"),
                    ("font-size", "18px")
                ]
            },
            {
                "selector": "tbody td.col0",
                "props": [
                    ("font-weight", "700"),
                    ("text-align", "left"),
                    ("padding-left", "20px")
                ]
            }
        ])
    )
    return styled_table

# Main report function
def run_report(selected_seasons, metric_inputs):
    if not selected_seasons:
        print("Pick at least one SEASON checkbox.")
        return False

    active_metrics = []
    for include_flag, label, bet_type, min_conf, month_value, conf_group_matchup_value, pace_matchup_value in metric_inputs:
        if include_flag:
            active_metrics.append((label, bet_type, min_conf, month_value, conf_group_matchup_value, pace_matchup_value))

    if not active_metrics:
        print("Check at least one Include box.")
        return False

    base_df = full_df[full_df["SEASON"].isin(selected_seasons)].copy()

    if base_df.empty:
        print("No rows after season filters.")
        return False

    display(HTML(
        f"""
        <div style="
            margin: 8px 0 20px 0;
            padding: 12px 16px;
            border: 1px solid #DADADA;
            border-radius: 10px;
            background: #F8F8F8;
            font-family: Arial, Helvetica, sans-serif;
        ">
            <div style="font-size:20px; font-weight:700; margin-bottom:6px;">
                Spreads Results
            </div>
            <div style="font-size:15px;">
                <b>Selected Seasons:</b> {", ".join(map(str, [y for y in years if y in selected_seasons]))}
            </div>
            <div style="font-size:14px; margin-top:6px; color:#555;">
                Each metric row is calculated individually. The last row, Summary of All Unique, removes duplicate rows across all included metrics.
            </div>
        </div>
        """
    ))

    result_rows = []
    chart_rows = []
    all_unique_parts = []

    for label, bet_type, min_conf, month_value, conf_group_matchup_value, pace_matchup_value in active_metrics:
        metric_df = apply_metric(
            base_df,
            bet_type=bet_type,
            min_conf=min_conf,
            month_value=month_value,
            conf_group_matchup_value=conf_group_matchup_value,
            pace_matchup_value=pace_matchup_value
        ).copy()

        if not metric_df.empty:
            all_unique_parts.append(metric_df.copy())

        summary = summarize_subset(metric_df)

        bt_label = "ALL" if bet_type == "" else bet_type
        conf_label = "ALL" if min_conf == "" else f"{min_conf}+"
        month_label = "ALL" if month_value == "" else month_value
        conf_group_label = "ALL" if conf_group_matchup_value == "" else conf_group_matchup_value
        pace_label = "ALL" if pace_matchup_value == "" else pace_matchup_value

        spreads_label = (
            f"{label} ({bt_label} | Conf {conf_label} | {month_label} | "
            f"Conf Group {conf_group_label} | Pace {pace_label})"
        )

        result_rows.append({
            "Spreads": spreads_label,
            "Wins": summary["Wins"],
            "Losses": summary["Losses"],
            "Win %": "" if pd.isna(summary["Win %"]) else f'{summary["Win %"]:.1%}',
            "Net Units": "" if pd.isna(summary["Net Units"]) else f'{summary["Net Units"]:.1f}',
            "N": summary["N"]
        })

        chart_rows.append({
            "Metric": label,
            "Net Units": 0 if pd.isna(summary["Net Units"]) else float(summary["Net Units"]),
            "N": summary["N"]
        })

    if all_unique_parts:
        all_unique_df = pd.concat(all_unique_parts, ignore_index=True)
        all_unique_df = all_unique_df.drop_duplicates(subset=["_row_id"]).copy()
    else:
        all_unique_df = base_df.iloc[0:0].copy()

    summary_all_unique = summarize_subset(all_unique_df)
    result_rows.append({
        "Spreads": "Summary of All Unique",
        "Wins": summary_all_unique["Wins"],
        "Losses": summary_all_unique["Losses"],
        "Win %": "" if pd.isna(summary_all_unique["Win %"]) else f'{summary_all_unique["Win %"]:.1%}',
        "Net Units": "" if pd.isna(summary_all_unique["Net Units"]) else f'{summary_all_unique["Net Units"]:.1f}',
        "N": summary_all_unique["N"]
    })

    show_df = pd.DataFrame(result_rows, columns=["Spreads", "Wins", "Losses", "Win %", "Net Units", "N"])
    display(style_single_table(show_df))

    # Build the net-units bar chart by metric
    if chart_rows:
        chart_df = pd.DataFrame(chart_rows)
        bar_colors = ["#0B7D3E" if x >= 0 else "#B42318" for x in chart_df["Net Units"]]

        fig, ax = plt.subplots(figsize=(10.5, 6))
        bars = ax.bar(chart_df["Metric"], chart_df["Net Units"], width=0.6, color=bar_colors)

        ax.axhline(0, linestyle="--", linewidth=1, color="black")
        ax.set_title("Net Units by Metric", fontsize=16, fontweight="bold", pad=12)
        ax.set_xlabel("Metric", fontsize=12)
        ax.set_ylabel("Net Units", fontsize=12)
        ax.tick_params(axis="x", labelsize=11)
        ax.tick_params(axis="y", labelsize=11)

        max_abs = chart_df["Net Units"].abs().max()
        pad = max(1.0, max_abs * 0.10 if pd.notna(max_abs) else 1.0)

        ymin = min(chart_df["Net Units"].min() - pad * 2.1, -pad * 2.1)
        ymax = max(chart_df["Net Units"].max() + pad, pad)
        ax.set_ylim(ymin, ymax)

        for bar, val in zip(bars, chart_df["Net Units"]):
            x = bar.get_x() + bar.get_width() / 2
            if val >= 0:
                ax.text(x, val + pad * 0.20, f"{val:.1f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
            else:
                ax.text(x, val - pad * 0.20, f"{val:.1f}", ha="center", va="top", fontsize=11, fontweight="bold")

        sample_y = ymin + (ymax - ymin) * 0.05
        for bar, n in zip(bars, chart_df["N"]):
            x = bar.get_x() + bar.get_width() / 2
            ax.text(x, sample_y, f"N={int(n)}", ha="center", va="bottom", fontsize=10, fontweight="bold", color="#444444")

        plt.tight_layout()
        plt.show()

    return True

# Dropdown options used in the widget controls
bet_type_options = ["", "ALL", "HOME", "AWAY"]
conf_score_options = ["", 0, 1, 2, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8]
month_options = ["", "DEC", "JAN", "FEB", "DEC/JAN", "JAN/FEB"]
conf_group_matchup_options = custom_conf_group_options

style = {"description_width": "95px"}
dd_layout = widgets.Layout(width="180px")
wide_dd_layout = widgets.Layout(width="230px")
label_layout = widgets.Layout(width="85px")

season_boxes, season_ui = make_checkbox_group(
    years,
    selected=years,
    width="95px"
)

include_1_cb = widgets.Checkbox(
    value=False,
    description="Include",
    indent=False,
    layout=widgets.Layout(width="90px")
)
row_1_label = widgets.HTML("<b>Metric 1</b>", layout=label_layout)

bet_type_1_dd = widgets.Dropdown(
    options=bet_type_options,
    value="",
    description="Bet Type",
    style=style,
    layout=dd_layout
)

conf_score_1_dd = widgets.Dropdown(
    options=conf_score_options,
    value="",
    description="Conf +",
    style=style,
    layout=dd_layout
)

month_1_dd = widgets.Dropdown(
    options=month_options,
    value="",
    description="Month",
    style=style,
    layout=dd_layout
)

conf_group_1_dd = widgets.Dropdown(
    options=conf_group_matchup_options,
    value="",
    description="Conf Grp",
    style=style,
    layout=wide_dd_layout
)

pace_1_dd = widgets.Dropdown(
    options=pace_matchup_options,
    value="",
    description="Pace",
    style=style,
    layout=wide_dd_layout
)

include_2_cb = widgets.Checkbox(
    value=False,
    description="Include",
    indent=False,
    layout=widgets.Layout(width="90px")
)
row_2_label = widgets.HTML("<b>Metric 2</b>", layout=label_layout)

bet_type_2_dd = widgets.Dropdown(
    options=bet_type_options,
    value="",
    description="Bet Type",
    style=style,
    layout=dd_layout
)

conf_score_2_dd = widgets.Dropdown(
    options=conf_score_options,
    value="",
    description="Conf +",
    style=style,
    layout=dd_layout
)

month_2_dd = widgets.Dropdown(
    options=month_options,
    value="",
    description="Month",
    style=style,
    layout=dd_layout
)

conf_group_2_dd = widgets.Dropdown(
    options=conf_group_matchup_options,
    value="",
    description="Conf Grp",
    style=style,
    layout=wide_dd_layout
)

pace_2_dd = widgets.Dropdown(
    options=pace_matchup_options,
    value="",
    description="Pace",
    style=style,
    layout=wide_dd_layout
)

include_3_cb = widgets.Checkbox(
    value=False,
    description="Include",
    indent=False,
    layout=widgets.Layout(width="90px")
)
row_3_label = widgets.HTML("<b>Metric 3</b>", layout=label_layout)

bet_type_3_dd = widgets.Dropdown(
    options=bet_type_options,
    value="",
    description="Bet Type",
    style=style,
    layout=dd_layout
)

conf_score_3_dd = widgets.Dropdown(
    options=conf_score_options,
    value="",
    description="Conf +",
    style=style,
    layout=dd_layout
)

month_3_dd = widgets.Dropdown(
    options=month_options,
    value="",
    description="Month",
    style=style,
    layout=dd_layout
)

conf_group_3_dd = widgets.Dropdown(
    options=conf_group_matchup_options,
    value="",
    description="Conf Grp",
    style=style,
    layout=wide_dd_layout
)

pace_3_dd = widgets.Dropdown(
    options=pace_matchup_options,
    value="",
    description="Pace",
    style=style,
    layout=wide_dd_layout
)

submit_btn = widgets.Button(
    description="Run Report",
    button_style="primary",
    icon="table",
    layout=widgets.Layout(width="170px", height="42px")
)

output = widgets.Output()

# Reset the button whenever a control changes
for w in [
    include_1_cb, bet_type_1_dd, conf_score_1_dd, month_1_dd, conf_group_1_dd, pace_1_dd,
    include_2_cb, bet_type_2_dd, conf_score_2_dd, month_2_dd, conf_group_2_dd, pace_2_dd,
    include_3_cb, bet_type_3_dd, conf_score_3_dd, month_3_dd, conf_group_3_dd, pace_3_dd
]:
    w.observe(reset_run_button, names="value")

for box in season_boxes:
    box.observe(reset_run_button, names="value")

# Run the report when the button is clicked
def on_submit_clicked(_):
    submit_btn.button_style = ""
    submit_btn.description = "Running..."
    submit_btn.icon = "spinner"

    with output:
        output.clear_output(wait=True)
        try:
            ok = run_report(
                selected_seasons=get_checked_values(season_boxes),
                metric_inputs=[
                    (include_1_cb.value, "Metric 1", bet_type_1_dd.value, conf_score_1_dd.value, month_1_dd.value, conf_group_1_dd.value, pace_1_dd.value),
                    (include_2_cb.value, "Metric 2", bet_type_2_dd.value, conf_score_2_dd.value, month_2_dd.value, conf_group_2_dd.value, pace_2_dd.value),
                    (include_3_cb.value, "Metric 3", bet_type_3_dd.value, conf_score_3_dd.value, month_3_dd.value, conf_group_3_dd.value, pace_3_dd.value),
                ]
            )

            if ok:
                submit_btn.button_style = "success"
                submit_btn.description = "Report Ready"
                submit_btn.icon = "check"
            else:
                submit_btn.button_style = "warning"
                submit_btn.description = "No Results"
                submit_btn.icon = "info-circle"

        except Exception:
            import traceback
            print("RUN REPORT FAILED\n")
            traceback.print_exc()
            submit_btn.button_style = "danger"
            submit_btn.description = "Error"
            submit_btn.icon = "warning"

submit_btn.on_click(on_submit_clicked)

# Arrange the metric rows and season selector
metric_row_1 = widgets.HBox(
    [row_1_label, include_1_cb, bet_type_1_dd, conf_score_1_dd, month_1_dd, conf_group_1_dd, pace_1_dd],
    layout=widgets.Layout(gap="12px", margin="0 0 8px 0", flex_flow="row wrap", align_items="center")
)

metric_row_2 = widgets.HBox(
    [row_2_label, include_2_cb, bet_type_2_dd, conf_score_2_dd, month_2_dd, conf_group_2_dd, pace_2_dd],
    layout=widgets.Layout(gap="12px", margin="0 0 8px 0", flex_flow="row wrap", align_items="center")
)

metric_row_3 = widgets.HBox(
    [row_3_label, include_3_cb, bet_type_3_dd, conf_score_3_dd, month_3_dd, conf_group_3_dd, pace_3_dd],
    layout=widgets.Layout(gap="12px", margin="0 0 8px 0", flex_flow="row wrap", align_items="center")
)

season_row = widgets.VBox(
    [
        widgets.HTML("<b>SEASON</b>"),
        season_ui
    ],
    layout=widgets.Layout(margin="0 0 12px 0")
)

# Put the full UI on the page
ui = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 10px 0;'>Spreads Results</h3>"),
    metric_row_1,
    metric_row_2,
    metric_row_3,
    widgets.HBox([submit_btn], layout=widgets.Layout(margin="6px 0 12px 0")),
    season_row,
    output
])

display(ui)